# Ensnarlment in real spatial networks

This is to explore the generalized ensnarlment between real spatial networks.
Data source: https://github.com/jocpae/VesselGraph

Prerequisite:
- Package "Plotly" for 3D plots
- Package "kaleido" for saving 3D plots

## Imports

All analysis & visualization routines now live in the **`gemini_ensnarl`** package (see `gemini_ensnarl/`). Install it from the repository.

In [ ]:
import numpy as np
import networkx as nx
import pandas as pd
from itertools import combinations
from scipy import stats
from typing import Tuple, Optional, Literal

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from matplotlib import colormaps

import plotly
import plotly.graph_objects as go
import plotly.io as pio

from gemini_ensnarl import (
    Gauss_linking_integral,
    build_sibigraph,
    critical_flip_angles,
    candidate_mid_angles,
    sign_patterns_from_angles,
)
from gemini_ensnarl.nulls import generate_spatial_null_model, generate_radii_null_model
from gemini_ensnarl.regions import (
    extract_region_subgraph, build_region_graphs, select_vertices_range,
    graph_diagnostics, get_cc, summarize_cc_topk,
)
from gemini_ensnarl.viz_data import *


## Exploratory

### import data sets

In [ ]:
# Data live in ../data (see data/README.md)
path = '../data/'

# information about the nodes
df_nodes = pd.read_csv(path + "CD1-E-no1_iso3um_stitched_segmentation_bulge_size_3.0_nodes_processed.csv", sep=';')
df_nodes.head()

In [ ]:
df_nodes.info()

In [ ]:
# information about Allen atlas regions
df_regions = pd.read_csv(path + "CD1-E-no1_iso3um_stitched_segmentation_bulge_size_3.0_atlas_processed.csv", sep=';')
df_regions.head()

In [ ]:
## merge the information with the nodes
# create dictionary for the columns of the regions
region_ids = df_regions.to_numpy().argmax(axis=1)
df_nodes['region_id'] = region_ids
df_nodes['region_label'] = df_regions.columns[region_ids].values

df_nodes.head()

In [ ]:
# information about the edges
df_edges = pd.read_csv(path + "CD1-E-no1_iso3um_stitched_segmentation_bulge_size_3.0_edges_processed.csv", sep=';')

In [ ]:
df_edges.head()

In [ ]:
df_edges.info()

### Square slicing

#### Zone I: cerebellum

In [ ]:
# --- Step 1: select a region ---
xlim = [1055., 1120.]
ylim = [3160., 3260.]
zlim = [855., 1085.]

In [ ]:
nodes_sel, regs_sum = select_vertices_range(df_nodes, xlim, ylim, zlim)
print("regions:", regs_sum[0])
print("counts:", regs_sum[1])

In [ ]:
# --- Step 2: select edges touching at least one selected node ---
df_edges_sel = df_edges[df_edges["node1id"].isin(nodes_sel) | df_edges["node2id"].isin(nodes_sel)]

# --- Step 3: update node set with all incident nodes ---
nodes_sel2 = set(df_edges_sel["node1id"]).union(set(df_edges_sel["node2id"]))
df_nodes_sel2 = df_nodes[df_nodes["id"].isin(nodes_sel2)]

print(f"Initial selected nodes: {len(nodes_sel)}")
print(f"Selected edges: {len(df_edges_sel)}")
print(f"Updated node set: {len(nodes_sel2)}")

In [ ]:
# create a node label dictionary
n2i_dict = {id:i for i,id in enumerate(df_nodes_sel2['id'].values)}
print("#nodes in dictionary", len(n2i_dict))

In [ ]:
# check the regions
np.unique(df_nodes_sel2['region_label'].values, return_counts=True)

In [ ]:
## Plot
# Region labels and colors
region_categories = sorted(regs_sum[0])
print(region_categories)

region_color_map = {
    region_categories[0]: '#BCBD22',
    region_categories[1]: '#9467BD',
    region_categories[2]: '#F58518', #'#8C564B'
    region_categories[3]: '#FF7F0E', # orange
    region_categories[4]: '#109618', # green
    region_categories[5]: '#620042', # dark brown
}

In [ ]:
# Build a fast lookup for node positions - edges color ~ radius
ns = 2.5
lw = 3.5
xval = 0.8
cmap = 'coolwarm'

pos_dict = df_nodes_sel2.set_index("id")[
    ["pos_x", "pos_y", "pos_z"]
].to_dict("index")

# prepare the color map
r_vals = df_edges_sel["minRadiusAvg"].values
val_min, val_max = r_vals.min(), r_vals.max()
print("max, min abs value in edge vals:", val_max, val_min)

# Normalize + convert colormap
colorscale = matplotlib_to_plotly_colorscale(cmap)
# Colorbar
colorbar_trace = dummy_colorbar_trace(val_min, val_max, colorscale, xval=xval)

# assign colors
norm = mcolors.Normalize(vmin=val_min, vmax=val_max)
cmap_vals = plt.colormaps.get_cmap(cmap)

edge_traces = []
for _, edge in df_edges_sel.iterrows():
    n1, n2 = edge["node1id"], edge["node2id"]

    if (n1 not in pos_dict) or (n2 not in pos_dict):
        continue

    x0, y0, z0 = pos_dict[n1]["pos_x"], pos_dict[n1]["pos_y"], pos_dict[n1]["pos_z"]
    x1, y1, z1 = pos_dict[n2]["pos_x"], pos_dict[n2]["pos_y"], pos_dict[n2]["pos_z"]

    r = edge["minRadiusAvg"]
    # normalized_r = (r - r_min) / (r_max - r_min + 1e-12)
    color = mcolors.to_hex(cmap_vals(norm(r)))
    edge_traces.append(go.Scatter3d(
            x=[x0, x1, None],
            y=[y0, y1, None],
            z=[z0, z1, None],
            mode="lines",
            line=dict(color=color, width=lw),
            hoverinfo="text",
            text=f"Radius: {r:.3f}",
            showlegend=False,
        ))

node_traces = []
for region, color in region_color_map.items():
    df_r = df_nodes_sel2[df_nodes_sel2["region_label"] == region]

    node_traces.append(
        go.Scatter3d(
            x=df_r["pos_x"],
            y=df_r["pos_y"],
            z=df_r["pos_z"],
            mode="markers",
            name=f"Region {region}",
            marker=dict(
                size=ns,
                color=color,
                opacity=0.9,
            ),
            hoverinfo="text",
            text=[
                f"Node {nid}<br>Region: {region}"
                for nid in df_r["id"]
            ],
        )
    )

fig = go.Figure(data=[*node_traces, *edge_traces, colorbar_trace])

axisFormat = dict(
                showbackground=False,
                showticklabels=False,
                autorange=True,
                showgrid=True,
        )


fig.update_layout(
    # title="3D Brain Vascular Subnetwork",
    scene=dict(
        xaxis=dict(title="x"),
        yaxis=dict(title="y"),
        zaxis=dict(title="z"),
        aspectmode="data",
    ),
    margin=dict(l=0, r=0, b=0, t=0),
    # legend=dict(itemsizing="constant"),
)

fig.show()

In [ ]:
#set an appropriate camera (for later plots as well)
camera = dict(
    eye=dict(x=1.8, y=1.1, z=.5),
    center=dict(x=0, y=0, z=0),
    up=dict(x=0, y=0, z=.1)
)
fig.update_layout(scene_camera=camera)
fig.show()

In [ ]:
pio.write_image(fig, "vessel-z1_view.png", width=700, height=600, scale=2) #, scale=2

##### Edge counts

In [ ]:
### see how #edges change along the radius cuts
col_rad = "minRadiusAvg"
# cuts = np.arange(1., 16., 0.5)
cuts = sorted(df_edges_sel[col_rad].unique()[1:-1])

summary = []
for cut in cuts:
    # summarize the edges below and above
    num_below = len(df_edges_sel[df_edges_sel[col_rad] < cut])
    num_above = len(df_edges_sel[df_edges_sel[col_rad] >= cut])

    summary.append([cut, num_below, num_above])

In [ ]:
# visualize the summary
tk_fz = 14
lb_fz = 16
plt.figure(figsize=(7,3.5))
summary = np.array(summary)
plt.plot(summary[:, 0], summary[:, 1], '.--', c='b', alpha=0.6)
plt.plot(summary[:, 0], summary[:, 2], '.--', c='r', alpha=0.6)
# plt.yscale('log')
plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('# edges', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
# plt.legend()
# plt.grid()
plt.savefig("vessel-z1-edgrad.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# visualize the summary + cuts
cut_sel = [1.5, 2, 5]

plt.figure(figsize=(7,3.5))
# plot vertical lines
for cut in cut_sel:
    plt.axvline(x=cut, linestyle='--', color='k', alpha=0.5)

summary = np.array(summary)
plt.plot(summary[:, 0], summary[:, 1], '.--', c='b', alpha=0.6)
plt.plot(summary[:, 0], summary[:, 2], '.--', c='r', alpha=0.6)
# plt.yscale('log')
plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('# edges', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
# plt.legend()
# plt.grid()
plt.savefig("vessels-z1-edgradc.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
## Q: how does the edge distance correlate with radius?
res_dist = []
res_radi= []
G2 = build_region_graphs(df_nodes_sel2, df_edges_sel)

for _, row in df_edges_sel.iterrows():
    n1, n2 = row["node1id"], row["node2id"]
    res_radi.append(float(row["minRadiusAvg"]))

    # compute the distance between the two nodes
    pos1 = np.array(G2.nodes[n1]['pos'])
    pos2 = np.array(G2.nodes[n2]['pos'])
    dist = np.linalg.norm(pos1-pos2)
    res_dist.append(dist)

In [ ]:
# pearson correlation
from scipy import stats

corr = stats.pearsonr(res_dist, res_radi)
print("correlation:", corr)

In [ ]:
plt.figure(figsize=(5,3.8))
plt.scatter(res_dist, res_radi, marker='.', alpha=.5)
plt.xlabel('Distance')
plt.ylabel('Radius')
plt.savefig("vessels2-dist-radius.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(5,3.8))
plt.scatter(res_dist, res_radi, marker='.', alpha=.5)
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Distance')
plt.ylabel('Radius')
plt.savefig("vessels2-dist-radius-log.png", dpi=300, bbox_inches='tight')
plt.show()

##### Unbalance scores + Centralities

In [ ]:
### See how balance changes with cut
# sort edges
col_rad = "minRadiusAvg"
df_edges_sel = df_edges_sel.sort_values(by=col_rad)

G_sel = build_region_graphs(df_nodes_sel2, df_edges_sel)

## compute the full GLI between each pair of edges in order
Lgli_full = np.zeros((len(df_edges_sel), len(df_edges_sel)))
for i in range(Lgli_full.shape[0]):
    e1 = [df_edges_sel.iloc[i]['node1id'], df_edges_sel.iloc[i]['node2id']]
    e1_pos = [np.array(G_sel.nodes[e1[0]]['pos']), np.array(G_sel.nodes[e1[1]]['pos'])]
    for j in range(i+1, Lgli_full.shape[1]):
        e2 = [df_edges_sel.iloc[j]['node1id'], df_edges_sel.iloc[j]['node2id']]
        # check whether they share a vertex
        if e1[0] in e2 or e1[1] in e2:
            Lgli_full[i, j] = 0
        else:
            # record the position
            e2_pos = [np.array(G_sel.nodes[e2[0]]['pos']), np.array(G_sel.nodes[e2[1]]['pos'])]
            # direct computation of Gauss linking integral
            gli_ij = Gauss_linking_integral(e1_pos, e2_pos)
            Lgli_full[i, j] = gli_ij

In [ ]:
### See how *centrality distribution* changes with cut
res_cut = []

res_bal = []
res_cme = []
res_csd = []
res_cax = []
idx = 0
for _, row in df_edges_sel.iterrows():

    # obtain the linking
    Lam_gli = Lgli_full[:idx+1, :][:, idx+1:]
    print("Lam_gli shape:", Lam_gli.shape)

    if Lam_gli.shape[1] == 0:
        continue

    # record the cut
    res_cut.append(float(row[col_rad]))
    print("cut:", res_cut[-1])

    idx += 1

    # weigh matrix of the bipartite graph
    W_gli = np.block([[np.zeros((Lam_gli.shape[0], Lam_gli.shape[0])), Lam_gli],
                      [Lam_gli.T, np.zeros((Lam_gli.shape[1], Lam_gli.shape[1]))]])

    ## Balance score
    # prepare connected components if any
    Gb = nx.from_numpy_array(W_gli)
    Gb_ccs = [Gb.subgraph(c) for c in nx.connected_components(Gb)]
    print("#connected components:", len(Gb_ccs))
    print("sizes:", [(g.number_of_nodes(), g.number_of_edges()) for g in Gb_ccs])

    # for each cc
    res = []
    for idx_g in range(len(Gb_ccs)):
        Gb_cc = Gb_ccs[idx_g]
        if Gb_cc.number_of_nodes() < 3:
            continue
        # get the adjacency
        W_cc = nx.to_numpy_array(Gb_cc)
        deg = np.sum(abs(W_cc), axis=1)

        # symmetric version
        D2_inv = np.diag(1./np.sqrt(deg))
        P_sym = D2_inv.dot(W_cc).dot(D2_inv)
        eigvals = np.linalg.eigvals(P_sym)
        # print("Max eigenval of P:", np.max(eigvals))
        res.append(np.max(eigvals))
    print("balance score:", res)
    # set one balance score for each cut value
    if len(res) == 0:
        res.append(np.nan)
    else:
        res_bal.append(np.mean(res).real)

    ## Centralities
    res = np.sum(abs(W_gli), axis=0)
    res_cme.append(np.mean(res))
    res_csd.append(np.std(res))
    res_cax.append(np.max(res))

1. Unbalance score

In [ ]:
res_cut = np.array(res_cut)
res_bal = np.array(res_bal)

In [ ]:
# save data
df_res = pd.DataFrame({'cut': res_cut, 'bal': res_bal})
df_res.to_csv("vessels-z1-balrad.csv", index=False)
# df_res.to_csv("/content/drive/MyDrive/ensnarl_res/vessels-z1-balrad.csv", index=False)

In [ ]:
plt.figure(figsize=(7,3.5))
# plt.scatter(cut_arr, bal_arr, marker='.', alpha=0.6)
plt.plot(res_cut, 1-res_bal, marker='.', linestyle='dashed', alpha=0.6, c='blueviolet')
plt.xlabel('Radius cuts')
plt.ylabel('Unalance score')
plt.savefig("vessels-z1-unbalrad-l.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(7,3.5))
for cut in cut_sel:
    plt.axvline(x=cut, linestyle='--', color='k', alpha=0.5)

# plt.scatter(cut_arr, bal_arr, marker='.', alpha=0.6)
plt.plot(res_cut, 1-res_bal, marker='.', linestyle='dashed', alpha=0.6, c='blueviolet')
plt.xlabel('Radius cuts')
plt.ylabel('Unbalance score')
plt.savefig("vessels-z1-unbalrad-lc.png", dpi=300, bbox_inches='tight')
plt.show()

2. Centrality

In [ ]:
# set one balance score for each cut value
res_cut = np.array(res_cut)
res_cme = np.array(res_cme)
res_csd = np.array(res_csd)
res_cax = np.array(res_cax)

In [ ]:
# save data
df_res = pd.DataFrame({'cut': res_cut, 'cent-mean': res_cme, 'cent-std': res_csd, 'cent-max': res_cax})
df_res.to_csv("vessels-z1-cenrad.csv", index=False)
# df_res.to_csv("/content/drive/MyDrive/ensnarl_res/vessels-z1-cenrad.csv", index=False)

In [ ]:
plt.figure(figsize=(7,3.5))
plt.plot(res_cut, res_cme, marker='.', linestyle='dashed', alpha=0.6, c='orangered')
plt.fill_between(res_cut, res_cme-res_csd, res_cme+res_csd, alpha=0.2, color='orangered')
plt.xlabel('Radius cuts')
plt.ylabel('Centrality')
plt.savefig("vessels-z1-cenrad-l.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(7,3.5))
for cut in cut_sel:
    plt.axvline(x=cut, linestyle='--', color='k', alpha=0.5)

plt.plot(res_cut, res_cme, marker='.', linestyle='dashed', alpha=0.6, c='orangered')
plt.fill_between(res_cut, res_cme-res_csd, res_cme+res_csd, alpha=0.2, color='orangered')
plt.xlabel('Radius cuts')
plt.ylabel('Centrality')
plt.savefig("vessels-z1-cenrad-lc.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(7,3.5))
plt.plot(res_cut, res_cax, marker='.', linestyle='dashed', alpha=0.6, c='orangered')
plt.xlabel('Radius cuts')
plt.ylabel('Centrality')
plt.savefig("vessels-z1-cmxrad-l.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(7,3.5))
for cut in cut_sel:
    plt.axvline(x=cut, linestyle='--', color='k', alpha=0.5)
plt.plot(res_cut, res_cax, marker='.', linestyle='dashed', alpha=0.6, c='orangered')
plt.xlabel('Radius cuts')
plt.ylabel('Centrality')
plt.savefig("vessels-z1-cmxrad-lc.png", dpi=300, bbox_inches='tight')
plt.show()

###### Null model 1: SER and SRG

In [ ]:
# random sample and compute the unbalance score for comparison
num_sam = 10

# choose the model
lab_mod = ['er', 'dist'][1]
p_dist = True if lab_mod == 'dist' else False
print("probability by distance?", p_dist)

num_nodes = len(nodes_sel2)
num_edges = len(df_edges_sel)
# Use empirical bounding box for the null positions
xyz_ranges = ((xlim[0], xlim[1]), (ylim[0], ylim[1]), (zlim[0], zlim[1]))

col_rad = "minRadiusAvg"

resall_cut = []
# balance
resall_bal = []
# centrality
resall_cme = []
resall_csd = []
resall_cax = []

for s in range(num_sam):
    print("sample:", s)

    ### generate null models
    nodes_null_df, edges_null_df, G_null = generate_spatial_null_model(
        n_nodes=num_nodes, m_edges=num_edges, df_edges_sel=df_edges_sel,
        xyz_ranges=xyz_ranges, directed=False, p_dist=p_dist)

    ### sweep the unbalance score
    # sort edges
    edges_null_df = edges_null_df.sort_values(by=col_rad)

    # compute the full GLI between each pair of edges in order
    Lgli_full = np.zeros((len(edges_null_df), len(edges_null_df)))
    for i in range(Lgli_full.shape[0]):
        e1 = [edges_null_df.iloc[i]['node1id'], edges_null_df.iloc[i]['node2id']]
        e1_pos = [np.array(G_null.nodes[e1[0]]['pos']), np.array(G_null.nodes[e1[1]]['pos'])]
        for j in range(i+1, Lgli_full.shape[1]):
            e2 = [edges_null_df.iloc[j]['node1id'], edges_null_df.iloc[j]['node2id']]
            # check whether they share a vertex
            if e1[0] in e2 or e1[1] in e2:
                Lgli_full[i, j] = 0
            else:
                # record the position
                e2_pos = [np.array(G_null.nodes[e2[0]]['pos']), np.array(G_null.nodes[e2[1]]['pos'])]
                # direct computation of Gauss linking integral
                gli_ij = Gauss_linking_integral(e1_pos, e2_pos)
                Lgli_full[i, j] = gli_ij
    # compute the balance score
    res_cut = []
    res_bal = []
    res_cme = []
    res_csd = []
    res_cax = []
    idx = 0
    for _, row in edges_null_df.iterrows():

        # obtain the linking
        Lam_gli = Lgli_full[:idx+1, :][:, idx+1:]
        print("Lam_gli shape:", Lam_gli.shape)

        if Lam_gli.shape[1] == 0:
            continue

        # record the cut
        res_cut.append(float(row[col_rad]))
        print("cut:", res_cut[-1])

        idx += 1
        # characterize the balance
        W_gli = np.block([[np.zeros((Lam_gli.shape[0], Lam_gli.shape[0])), Lam_gli],
                          [Lam_gli.T, np.zeros((Lam_gli.shape[1], Lam_gli.shape[1]))]])

        # prepare connected components if any
        Gb = nx.from_numpy_array(W_gli)
        Gb_ccs = [Gb.subgraph(c) for c in nx.connected_components(Gb)]
        print("#connected components:", len(Gb_ccs))
        print("sizes:", [(g.number_of_nodes(), g.number_of_edges()) for g in Gb_ccs])

        # for each cc
        res = []
        for idx_g in range(len(Gb_ccs)):
            Gb_cc = Gb_ccs[idx_g]
            if Gb_cc.number_of_nodes() < 3:
                continue
            # get the adjacency
            W_cc = nx.to_numpy_array(Gb_cc)
            deg = np.sum(abs(W_cc), axis=1)

            # symmetric version
            D2_inv = np.diag(1./np.sqrt(deg))
            P_sym = D2_inv.dot(W_cc).dot(D2_inv)
            eigvals = np.linalg.eigvals(P_sym)
            # print("Max eigenval of P:", np.max(eigvals))
            res.append(np.max(eigvals))
        print("balance score:", res)
        # set one balance score for each cut value
        if len(res) == 0:
            res.append(np.nan)
        else:
            res_bal.append(np.mean(res).real)

        ## Centrality
        # record the distribution of centralities
        res = np.sum(abs(W_gli), axis=0)
        res_cme.append(np.mean(res))
        res_csd.append(np.std(res))
        res_cax.append(np.max(res))

    # store the results
    resall_cut.append(res_cut)
    resall_bal.append(res_bal)
    resall_cme.append(res_cme)
    resall_csd.append(res_csd)
    resall_cax.append(res_cax)

1. Unbalance score

In [ ]:
cut_arr = np.array(resall_cut).mean(axis=0)
cut_std = np.array(resall_cut).std(axis=0)
# check the std of cuts
print("Cut std:", min(cut_std), max(cut_std))

bal_arr = np.array(resall_bal).mean(axis=0)
bal_std = np.array(resall_bal).std(axis=0)

In [ ]:
# plot average +- standard deviation
plt.figure(figsize=(7,3.5))
plt.plot(cut_arr, 1-bal_arr, linestyle='dashed', marker='.', alpha=0.6, c='k')
plt.fill_between(cut_arr, 1-bal_arr-bal_std, 1-bal_arr+bal_std, alpha=0.2, color='grey')
plt.xlabel('Radius cuts')
plt.ylabel('Unbalance score')
plt.savefig("vessels-z1-unbalrad-sw-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# save the data
df_res = pd.DataFrame({'cut': cut_arr, 'bal': bal_arr, 'bal_std': bal_std})
df_res.to_csv("vessels-z1-unbalrad-sw-{}-s{}.csv".format(lab_mod, num_sam), index=False)

2. Centrality

In [ ]:
cut_arr = np.array(resall_cut).mean(axis=0)
cut_std = np.array(resall_cut).std(axis=0)
# check the std of cuts
print("Cut std:", min(cut_std), max(cut_std))

cme_arr = np.array(resall_cme).mean(axis=0)
cme_std = np.array(resall_cme).std(axis=0)
csd_arr = np.array(resall_csd).mean(axis=0)
csd_std = np.array(resall_csd).std(axis=0)
cax_arr = np.array(resall_cax).mean(axis=0)
cax_std = np.array(resall_cax).std(axis=0)

In [ ]:
  # plot average +- standard deviation: centrality mean & centrality deviation
plt.figure(figsize=(7,3.5))
plt.plot(cut_arr, cme_arr, linestyle='dashed', marker='.', alpha=0.6, c='k')
plt.fill_between(cut_arr, cme_arr-csd_arr, cme_arr+csd_arr, alpha=0.2, color='grey')
plt.xlabel('Radius cuts')
plt.ylabel('Centrality')
plt.savefig("vessels-z1-cenrad-sw-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot average +- standard deviation: centrality mean
plt.figure(figsize=(7,3.5))
plt.plot(cut_arr, cme_arr, linestyle='dashed', marker='.', alpha=0.6, c='k')
plt.fill_between(cut_arr, cme_arr-cme_std, cme_arr+cme_std, alpha=0.2, color='grey')
plt.xlabel('Radius cuts')
plt.ylabel('Centrality Mean')
plt.savefig("vessels-z1-cmerad-sw-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot average +- standard deviation: centrality max
plt.figure(figsize=(7,3.5))
plt.plot(cut_arr, cax_arr, linestyle='dashed', marker='.', alpha=0.6, c='k')
plt.fill_between(cut_arr, cax_arr-cax_std, cax_arr+cax_std, alpha=0.2, color='grey')
plt.xlabel('Radius cuts')
plt.ylabel('Centrality Max')
plt.savefig("vessels-z1-caxrad-sw-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# save the data
df_res = pd.DataFrame({'cut': cut_arr, 'cent-mean-mean': cme_arr, 'cent-mean-std': cme_std,
                       'cent-std-mean': csd_arr, 'cent-std-std': csd_std,
                       'cent-max-mean': cax_arr, 'cent-max-std': cax_std})
df_res.to_csv("vessels-z1-cenrad-sw-{}-s{}.csv".format(lab_mod, num_sam), index=False)

###### Null model 2: shuffle radius

In [ ]:
# random sample and compute the unbalance score for comparison
num_sam = 10

# choose the model
lab_mod = ['radi'][0]
print("null model", lab_mod)

num_nodes = len(nodes_sel2)
num_edges = len(df_edges_sel)
# Use empirical bounding box for the null positions
xyz_ranges = ((xlim[0], xlim[1]), (ylim[0], ylim[1]), (zlim[0], zlim[1]))

col_rad = "minRadiusAvg"

resall_cut = []
# balance
resall_bal = []
# centrality
resall_cut = []
resall_cme = []
resall_csd = []
resall_cax = []

for s in range(num_sam):
    print("sample:", s)

    ### generate null models
    edges_null_df, G_null = generate_radii_null_model(
        m_edges=num_edges, df_edges_sel=df_edges_sel, df_nodes_sel=df_nodes_sel2, directed=False)

    ### sweep the unbalance score
    # sort edges
    edges_null_df = edges_null_df.sort_values(by=col_rad)

    # compute the full GLI between each pair of edges in order
    Lgli_full = np.zeros((len(edges_null_df), len(edges_null_df)))
    for i in range(Lgli_full.shape[0]):
        e1 = [edges_null_df.iloc[i]['node1id'], edges_null_df.iloc[i]['node2id']]
        e1_pos = [np.array(G_null.nodes[e1[0]]['pos']), np.array(G_null.nodes[e1[1]]['pos'])]
        for j in range(i+1, Lgli_full.shape[1]):
            e2 = [edges_null_df.iloc[j]['node1id'], edges_null_df.iloc[j]['node2id']]
            # check whether they share a vertex
            if e1[0] in e2 or e1[1] in e2:
                Lgli_full[i, j] = 0
            else:
                # record the position
                e2_pos = [np.array(G_null.nodes[e2[0]]['pos']), np.array(G_null.nodes[e2[1]]['pos'])]
                # direct computation of Gauss linking integral
                gli_ij = Gauss_linking_integral(e1_pos, e2_pos)
                Lgli_full[i, j] = gli_ij

    # compute the balance score
    res_cut = []
    res_bal = []
    res_bal = []
    res_cme = []
    res_csd = []
    res_cax = []
    idx = 0
    for _, row in edges_null_df.iterrows():

        # obtain the linking
        Lam_gli = Lgli_full[:idx+1, :][:, idx+1:]
        print("Lam_gli shape:", Lam_gli.shape)

        if Lam_gli.shape[1] == 0:
            continue

        # record the cut
        res_cut.append(float(row[col_rad]))
        print("cut:", res_cut[-1])

        idx += 1
        # weight matrix of the bipartite graph
        W_gli = np.block([[np.zeros((Lam_gli.shape[0], Lam_gli.shape[0])), Lam_gli],
                          [Lam_gli.T, np.zeros((Lam_gli.shape[1], Lam_gli.shape[1]))]])

        ## Balance
        # prepare connected components if any
        Gb = nx.from_numpy_array(W_gli)
        Gb_ccs = [Gb.subgraph(c) for c in nx.connected_components(Gb)]
        print("#connected components:", len(Gb_ccs))
        print("sizes:", [(g.number_of_nodes(), g.number_of_edges()) for g in Gb_ccs])

        # for each cc
        res = []
        for idx_g in range(len(Gb_ccs)):
            Gb_cc = Gb_ccs[idx_g]
            if Gb_cc.number_of_nodes() < 3:
                continue
            # get the adjacency
            W_cc = nx.to_numpy_array(Gb_cc)
            deg = np.sum(abs(W_cc), axis=1)

            # symmetric version
            D2_inv = np.diag(1./np.sqrt(deg))
            P_sym = D2_inv.dot(W_cc).dot(D2_inv)
            eigvals = np.linalg.eigvals(P_sym)
            # print("Max eigenval of P:", np.max(eigvals))
            res.append(np.max(eigvals))
        print("balance score:", res)
        # set one balance score for each cut value
        if len(res) == 0:
            res.append(np.nan)
        else:
            res_bal.append(np.mean(res).real)

        ## Centrality
        # record the distribution of centralities
        res = np.sum(abs(W_gli), axis=0)
        res_cme.append(np.mean(res))
        res_csd.append(np.std(res))
        res_cax.append(np.max(res))

    # store the results
    resall_cut.append(res_cut)
    resall_bal.append(res_bal)
    resall_cme.append(res_cme)
    resall_csd.append(res_csd)
    resall_cax.append(res_cax)

1. Unbalance score

In [ ]:
cut_arr = np.array(resall_cut).mean(axis=0)
cut_std = np.array(resall_cut).std(axis=0)
# check the std of cuts
print("Cut std:", min(cut_std), max(cut_std))

bal_arr = np.array(resall_bal).mean(axis=0)
bal_std = np.array(resall_bal).std(axis=0)

In [ ]:
# plot average +- standard deviation
plt.figure(figsize=(7,3.5))
plt.plot(cut_arr, 1-bal_arr, linestyle='dashed', marker='.', alpha=0.6, c='k')
plt.fill_between(cut_arr, 1-bal_arr-bal_std, 1-bal_arr+bal_std, alpha=0.2, color='grey')
plt.xlabel('Radius cuts')
plt.ylabel('Balance score')
plt.savefig("vessels-z1-unbalrad-null-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# save the data
df_res = pd.DataFrame({'cut': cut_arr, 'bal': bal_arr, 'bal_std': bal_std})
df_res.to_csv("vessels-z1-unbalrad-null-{}-s{}.csv".format(lab_mod, num_sam), index=False)

In [ ]:
### comparison
# read in data
df_bal = pd.read_csv("vessels-z1-balrad.csv")
df_radi = pd.read_csv("vessels-z1-unbalrad-null-radi-s10.csv")

In [ ]:
colors = ['r', 'b', 'y', 'c']
# plot together
plt.figure(figsize=(7,3.5))

# null model - radius
plt.plot(df_radi['cut'], 1-df_radi['bal'], linestyle='dashed', marker='.', alpha=0.3, c=colors[3], label='Rad')
plt.fill_between(df_radi['cut'], 1-df_radi['bal']-df_radi['bal_std'], 1-df_radi['bal']+df_radi['bal_std'], alpha=0.2, color=colors[3])

# actual value
plt.plot(df_bal['cut'], 1-df_bal['bal'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.xlabel('Radius cuts')
plt.ylabel('Unbalance score')
plt.legend()
plt.savefig("vessels-z1-unbalrad-null2.png", dpi=300, bbox_inches='tight')
plt.show()

2. Centrality

In [ ]:
cut_arr = np.array(resall_cut).mean(axis=0)
cut_std = np.array(resall_cut).std(axis=0)
# check the std of cuts
print("Cut std:", min(cut_std), max(cut_std))

cme_arr = np.array(resall_cme).mean(axis=0)
cme_std = np.array(resall_cme).std(axis=0)
csd_arr = np.array(resall_csd).mean(axis=0)
csd_std = np.array(resall_csd).std(axis=0)
cax_arr = np.array(resall_cax).mean(axis=0)
cax_std = np.array(resall_cax).std(axis=0)

In [ ]:
# plot average +- standard deviation: centrality mean & centrality deviation
plt.figure(figsize=(7,3.5))
plt.plot(cut_arr, cme_arr, linestyle='dashed', marker='.', alpha=0.6, c='k')
plt.fill_between(cut_arr, cme_arr-csd_arr, cme_arr+csd_arr, alpha=0.2, color='grey')
plt.xlabel('Radius cuts')
plt.ylabel('Centrality')
plt.savefig("vessels-z1-cenrad-null-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot average +- standard deviation: centrality mean
plt.figure(figsize=(7,3.5))
plt.plot(cut_arr, cme_arr, linestyle='dashed', marker='.', alpha=0.6, c='k')
plt.fill_between(cut_arr, cme_arr-cme_std, cme_arr+cme_std, alpha=0.2, color='grey')
plt.xlabel('Radius cuts')
plt.ylabel('Centrality')
plt.savefig("vessels-z1-cmerad-null-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot average +- standard deviation: centrality max
plt.figure(figsize=(7,3.5))
plt.plot(cut_arr, cax_arr, linestyle='dashed', marker='.', alpha=0.6, c='k')
plt.fill_between(cut_arr, cax_arr-cax_std, cax_arr+cax_std, alpha=0.2, color='grey')
plt.xlabel('Radius cuts')
plt.ylabel('Centrality')
plt.savefig("vessels-z1-caxrad-null-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# save the data
df_res = pd.DataFrame({'cut': cut_arr, 'cent-mean-mean': cme_arr, 'cent-mean-std': cme_std,
                       'cent-std-mean': csd_arr, 'cent-std-std': csd_std,
                       'cent-max-mean': cax_arr, 'cent-max-std': cax_std})
df_res.to_csv("vessels-z1-cenrad-null-{}-s{}.csv".format(lab_mod, num_sam), index=False)

In [ ]:
### comparison
# read in data
df_cen = pd.read_csv("vessels-z1-cenrad.csv")
df_radi = pd.read_csv("vessels-z1-cenrad-null-radi-s10.csv")

In [ ]:
colors = ['r', 'b', 'y', 'c']
# plot together
plt.figure(figsize=(7,3.5))

# null model - radius
plt.plot(df_radi['cut'], df_radi['cent-mean-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[3], label='Rad')
plt.fill_between(df_radi['cut'], df_radi['cent-mean-mean']-df_radi['cent-mean-std'],
                 df_radi['cent-mean-mean']+df_radi['cent-mean-std'], alpha=0.2, color=colors[3])

# actual value
plt.plot(df_cen['cut'], df_cen['cent-mean'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.xlabel('Radius cuts')
plt.ylabel('Centrality Mean')
plt.legend()
plt.savefig("vessels-z1-cmerad-null2.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot together
plt.figure(figsize=(7,3.5))

# null model - radius
plt.plot(df_radi['cut'], df_radi['cent-max-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[3], label='Rad')
plt.fill_between(df_radi['cut'], df_radi['cent-max-mean']-df_radi['cent-max-std'],
                 df_radi['cent-max-mean']+df_radi['cent-max-std'], alpha=0.2, color=colors[3])

# actual value
plt.plot(df_cen['cut'], df_cen['cent-max'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.xlabel('Radius cuts')
plt.ylabel('Centrality Max')
plt.legend()
plt.savefig("vessels-z1-caxrad-null2.png", dpi=300, bbox_inches='tight')
plt.show()

###### Summary

In [ ]:
# uniform scale
ylim_unb = [-0.05, 0.63]
ylim_cme = [0.0005, 45]
ylim_cax = [0.1, 180]

1. Unbalance scores

In [ ]:
## ALL
df_bal = pd.read_csv("vessels-z1-balrad.csv")
df_radi = pd.read_csv("vessels-z1-unbalrad-null-radi-s10.csv")
df_er = pd.read_csv("vessels-z1-unbalrad-sw-er-s10.csv")
df_dist = pd.read_csv("vessels-z1-unbalrad-sw-dist-s10.csv")

In [ ]:
colors = ['r', 'b', 'y', 'c']
# plot together
tk_fz = 14
lb_fz = 16
lg_fz = 12
plt.figure(figsize=(7,3.5))

# null model - radius
plt.plot(df_radi['cut'], 1-df_radi['bal'], linestyle='dashed', marker='.', alpha=0.3, c=colors[3], label='Rad')
plt.fill_between(df_radi['cut'], 1-df_radi['bal']-df_radi['bal_std'], 1-df_radi['bal']+df_radi['bal_std'], alpha=0.2, color=colors[3])

# null model - ER
plt.plot(df_er['cut'], 1-df_er['bal'], linestyle='dashed', marker='.', alpha=0.3, c=colors[2], label='SER')
plt.fill_between(df_er['cut'], 1-df_er['bal']-df_er['bal_std'], 1-df_er['bal']+df_er['bal_std'], alpha=0.2, color=colors[2])

# null model - dist
plt.plot(df_dist['cut'], 1-df_dist['bal'], linestyle='dashed', marker='.', alpha=0.3, c=colors[1], label='SRG')
plt.fill_between(df_dist['cut'], 1-df_dist['bal']-df_dist['bal_std'], 1-df_dist['bal']+df_dist['bal_std'], alpha=0.2, color=colors[1])

# actual value
plt.plot(df_bal['cut'], 1-df_bal['bal'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('unbalance score', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.legend(fontsize=lg_fz, frameon=False)
plt.savefig("vessels-z1-unbalrad-all.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
colors = ['r', 'b', 'y', 'c']
# plot together
plt.figure(figsize=(7,3.5))

# null model - ER
plt.plot(df_er['cut'], 1-df_er['bal'], linestyle='dashed', marker='.', alpha=0.3, c=colors[2], label='SER')
plt.fill_between(df_er['cut'], 1-df_er['bal']-df_er['bal_std'], 1-df_er['bal']+df_er['bal_std'], alpha=0.2, color=colors[2])

# null model - dist
plt.plot(df_dist['cut'], 1-df_dist['bal'], linestyle='dashed', marker='.', alpha=0.3, c=colors[1], label='SRG')
plt.fill_between(df_dist['cut'], 1-df_dist['bal']-df_dist['bal_std'], 1-df_dist['bal']+df_dist['bal_std'], alpha=0.2, color=colors[1])

# actual value
plt.plot(df_bal['cut'], 1-df_bal['bal'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.ylim(ylim_unb)
plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('unbalance score', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.legend(fontsize=lg_fz, frameon=False)
plt.savefig("vessels-z1-unbalrad-all3.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
colors = ['r', 'b', 'y', 'c']
# plot together
plt.figure(figsize=(7,3.5))

# null model - radius
plt.plot(df_radi['cut'], 1-df_radi['bal'], linestyle='dashed', marker='.', alpha=0.3, c=colors[3], label='Rad')
plt.fill_between(df_radi['cut'], 1-df_radi['bal']-df_radi['bal_std'], 1-df_radi['bal']+df_radi['bal_std'], alpha=0.2, color=colors[3])

# null model - dist
plt.plot(df_dist['cut'], 1-df_dist['bal'], linestyle='dashed', marker='.', alpha=0.3, c=colors[1], label='SRG')
plt.fill_between(df_dist['cut'], 1-df_dist['bal']-df_dist['bal_std'], 1-df_dist['bal']+df_dist['bal_std'], alpha=0.2, color=colors[1])

# actual value
plt.plot(df_bal['cut'], 1-df_bal['bal'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('unbalance score', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.legend(fontsize=lg_fz, frameon=False)
plt.savefig("vessels-z1-unbalrad-null3.png", dpi=300, bbox_inches='tight')
plt.show()

2. Centrality

In [ ]:
# read in data
df_cen = pd.read_csv("vessels-z1-cenrad.csv")
df_radi = pd.read_csv("vessels-z1-cenrad-null-radi-s10.csv")
df_er = pd.read_csv("vessels-z1-cenrad-sw-er-s10.csv")
df_dist = pd.read_csv("vessels-z1-cenrad-sw-dist-s10.csv")

In [ ]:
colors = ['orangered', 'b', 'y', 'c']

In [ ]:
# plot together
plt.figure(figsize=(7,3.5))

# null model - radius
plt.plot(df_radi['cut'], df_radi['cent-mean-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[3], label='Rad')
plt.fill_between(df_radi['cut'], df_radi['cent-mean-mean']-df_radi['cent-mean-std'],
                 df_radi['cent-mean-mean']+df_radi['cent-mean-std'], alpha=0.2, color=colors[3])

# null model - ER
plt.plot(df_er['cut'], df_er['cent-mean-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[2], label='SER')
plt.fill_between(df_er['cut'], df_er['cent-mean-mean']-df_er['cent-mean-std'],
                 df_er['cent-mean-mean']+df_er['cent-mean-std'], alpha=0.2, color=colors[2])

# null model - dist
plt.plot(df_dist['cut'], df_dist['cent-mean-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[1], label='SRG')
plt.fill_between(df_dist['cut'], df_dist['cent-mean-mean']-df_dist['cent-mean-std'],
                 df_dist['cent-mean-mean']+df_dist['cent-mean-std'], alpha=0.2, color=colors[1])

# actual value
plt.plot(df_cen['cut'], df_cen['cent-mean'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

# plt.yscale('log')
plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('centrality mean', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.legend(fontsize=lg_fz, frameon=False)
plt.savefig("vessels-z1-cmerad-all.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot together
plt.figure(figsize=(7,3.5))

# null model - radius
plt.plot(df_radi['cut'], df_radi['cent-mean-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[3], label='Rad')
plt.fill_between(df_radi['cut'], df_radi['cent-mean-mean']-df_radi['cent-mean-std'],
                 df_radi['cent-mean-mean']+df_radi['cent-mean-std'], alpha=0.2, color=colors[3])

# null model - ER
plt.plot(df_er['cut'], df_er['cent-mean-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[2], label='SER')
plt.fill_between(df_er['cut'], df_er['cent-mean-mean']-df_er['cent-mean-std'],
                 df_er['cent-mean-mean']+df_er['cent-mean-std'], alpha=0.2, color=colors[2])

# null model - dist
plt.plot(df_dist['cut'], df_dist['cent-mean-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[1], label='SRG')
plt.fill_between(df_dist['cut'], df_dist['cent-mean-mean']-df_dist['cent-mean-std'],
                 df_dist['cent-mean-mean']+df_dist['cent-mean-std'], alpha=0.2, color=colors[1])

# actual value
plt.plot(df_cen['cut'], df_cen['cent-mean'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.yscale('log')
plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('centrality mean', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.legend(fontsize=lg_fz, frameon=False)
plt.savefig("vessels-z1-cmerad-all-log.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(7,3.5))

# null model - ER
plt.plot(df_er['cut'], df_er['cent-mean-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[2], label='SER')
plt.fill_between(df_er['cut'], df_er['cent-mean-mean']-df_er['cent-mean-std'],
                 df_er['cent-mean-mean']+df_er['cent-mean-std'], alpha=0.2, color=colors[2])

# null model - dist
plt.plot(df_dist['cut'], df_dist['cent-mean-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[1], label='SRG')
plt.fill_between(df_dist['cut'], df_dist['cent-mean-mean']-df_dist['cent-mean-std'],
                 df_dist['cent-mean-mean']+df_dist['cent-mean-std'], alpha=0.2, color=colors[1])

# actual value
plt.plot(df_cen['cut'], df_cen['cent-mean'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.ylim(ylim_cme)
plt.yscale('log')
plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('centrality mean', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.legend(fontsize=lg_fz, frameon=False)
plt.savefig("vessels-z1-cmerad-all3-log.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(7,3.5))

# null model - radius
plt.plot(df_radi['cut'], df_radi['cent-mean-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[3], label='Rad')
plt.fill_between(df_radi['cut'], df_radi['cent-mean-mean']-df_radi['cent-mean-std'],
                 df_radi['cent-mean-mean']+df_radi['cent-mean-std'], alpha=0.2, color=colors[3])

# null model - dist
plt.plot(df_dist['cut'], df_dist['cent-mean-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[1], label='SRG')
plt.fill_between(df_dist['cut'], df_dist['cent-mean-mean']-df_dist['cent-mean-std'],
                 df_dist['cent-mean-mean']+df_dist['cent-mean-std'], alpha=0.2, color=colors[1])

# actual value
plt.plot(df_cen['cut'], df_cen['cent-mean'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.yscale('log')
plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('centrality mean', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.legend(fontsize=lg_fz, frameon=False)
plt.savefig("vessels-z1-cmerad-null3-log.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot together
plt.figure(figsize=(7,3.5))

# null model - radius
plt.plot(df_radi['cut'], 1-df_radi['cent-max-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[3], label='Rad')
plt.fill_between(df_radi['cut'], 1-df_radi['cent-max-mean']-df_radi['cent-max-std'],
                 1-df_radi['cent-max-mean']+df_radi['cent-max-std'], alpha=0.2, color=colors[3])

# null model - ER
plt.plot(df_er['cut'], df_er['cent-max-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[2], label='SER')
plt.fill_between(df_er['cut'], df_er['cent-max-mean']-df_er['cent-max-std'],
                 df_er['cent-max-mean']+df_er['cent-max-std'], alpha=0.2, color=colors[2])

# null model - dist
plt.plot(df_dist['cut'], df_dist['cent-max-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[1], label='SRG')
plt.fill_between(df_dist['cut'], df_dist['cent-max-mean']-df_dist['cent-max-std'],
                 df_dist['cent-max-mean']+df_dist['cent-max-std'], alpha=0.2, color=colors[1])

# actual value
plt.plot(df_cen['cut'], df_cen['cent-max'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

# plt.yscale('log')
plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('centrality max', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.legend(fontsize=lg_fz, frameon=False)
plt.savefig("vessels-z1-caxrad-all.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot together
plt.figure(figsize=(7,3.5))

# null model - radius
plt.plot(df_radi['cut'], df_radi['cent-max-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[3], label='Rad')
plt.fill_between(df_radi['cut'], df_radi['cent-max-mean']-df_radi['cent-max-std'],
                 df_radi['cent-max-mean']+df_radi['cent-max-std'], alpha=0.2, color=colors[3])

# null model - ER
plt.plot(df_er['cut'], df_er['cent-max-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[2], label='SER')
plt.fill_between(df_er['cut'], df_er['cent-max-mean']-df_er['cent-max-std'],
                 df_er['cent-max-mean']+df_er['cent-max-std'], alpha=0.2, color=colors[2])

# null model - dist
plt.plot(df_dist['cut'], df_dist['cent-max-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[1], label='SRG')
plt.fill_between(df_dist['cut'], df_dist['cent-max-mean']-df_dist['cent-max-std'],
                 df_dist['cent-max-mean']+df_dist['cent-max-std'], alpha=0.2, color=colors[1])

# actual value
plt.plot(df_cen['cut'], df_cen['cent-max'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.yscale('log')
plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('centrality max', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.legend(fontsize=lg_fz, frameon=False)
plt.savefig("vessels-z1-caxrad-all-log.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot together
plt.figure(figsize=(7,3.5))

# null model - ER
plt.plot(df_er['cut'], df_er['cent-max-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[2], label='SER')
plt.fill_between(df_er['cut'], df_er['cent-max-mean']-df_er['cent-max-std'],
                 df_er['cent-max-mean']+df_er['cent-max-std'], alpha=0.2, color=colors[2])

# null model - dist
plt.plot(df_dist['cut'], df_dist['cent-max-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[1], label='SRG')
plt.fill_between(df_dist['cut'], df_dist['cent-max-mean']-df_dist['cent-max-std'],
                 df_dist['cent-max-mean']+df_dist['cent-max-std'], alpha=0.2, color=colors[1])

# actual value
plt.plot(df_cen['cut'], df_cen['cent-max'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.ylim(ylim_cax)
plt.yscale('log')
plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('centrality max', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.legend(fontsize=lg_fz, frameon=False)
plt.savefig("vessels-z1-caxrad-all3-log.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot together
plt.figure(figsize=(7,3.5))

# null model - radius
plt.plot(df_radi['cut'], df_radi['cent-max-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[3], label='Rad')
plt.fill_between(df_radi['cut'], df_radi['cent-max-mean']-df_radi['cent-max-std'],
                 df_radi['cent-max-mean']+df_radi['cent-max-std'], alpha=0.2, color=colors[3])

# null model - dist
plt.plot(df_dist['cut'], df_dist['cent-max-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[1], label='SRG')
plt.fill_between(df_dist['cut'], df_dist['cent-max-mean']-df_dist['cent-max-std'],
                 df_dist['cent-max-mean']+df_dist['cent-max-std'], alpha=0.2, color=colors[1])

# actual value
plt.plot(df_cen['cut'], df_cen['cent-max'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.yscale('log')
plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('centrality max', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.legend(fontsize=lg_fz, frameon=False)
plt.savefig("vessels-z1-caxrad-null3-log.png", dpi=300, bbox_inches='tight')
plt.show()

##### Single cut

In [ ]:
# Is the graph connected?
G_sel2 = build_region_graphs(df_nodes_sel2, df_edges_sel)
component_summary = summarize_cc_topk(G_sel2, k=10)

###### cut = 2

In [ ]:
# Case 1: when # above and below are about the same
ns = 2.
lw = 3.5
cut = 2

## Visualization
# --- 1. Select edges for G1 (minRadiusAvg < cut) ---
edges_G1 = [(u, v, data) for u, v, data in G_sel2.edges(data=True) if data["minRadiusAvg"] < cut]

# --- 2. Select remaining edges for G2 (minRadiusAvg >= cut OR missing) ---
edges_G2 = [(u, v, data) for u, v, data in G_sel2.edges(data=True) if data["minRadiusAvg"] >= cut]

# print("#edges:", len(edges_G1), len(edges_G2))

# --- 3. Build subgraphs while preserving attributes ---
G1 = nx.Graph()
G1.add_nodes_from(G_sel2.nodes(data=True))
G1.add_edges_from(edges_G1)

G2 = nx.Graph()
G2.add_nodes_from(G_sel2.nodes(data=True))
G2.add_edges_from(edges_G2)

# --- 4. Remove isolated nodes ---
# G1.remove_nodes_from(list(nx.isolates(G1)))
# G2.remove_nodes_from(list(nx.isolates(G2)))

pos1 = nx.get_node_attributes(G1, 'pos')
pos2 = nx.get_node_attributes(G2, 'pos')
lab_G = 'vessel-z1-c{}'.format(cut)

In [ ]:
# edge colors
alpha = 0.9
row_c = "rgba(33, 89, 209,{})".format(alpha)
col_c = "rgba(196, 29, 29,{})".format(alpha)
# tick title sizes
ax_fz = 30
# figsize
figsize_g = [550, 700]

plot_networkx_12nolabel(G1, G2, pos1, pos2, size=ns, width=lw, row_c=row_c, col_c=col_c,
                        camera=camera, savefig=True, figname=lab_G+'-unic-G', figsize=figsize_g, ax_fz=ax_fz) #row_c='#2159d1', col_c='#c41d1d'

In [ ]:
## GLI analysis
# compute the GLI operator
# GLI between edges directly
edges1 = list(G1.edges())
edges2 = list(G2.edges())

Lam_gli = np.zeros((len(edges1), len(edges2)))
for i in range(Lam_gli.shape[0]):
    e1 = edges1[i]
    e1_pos = [np.array(G1.nodes[e1[0]]['pos']), np.array(G1.nodes[e1[1]]['pos'])]
    for j in range(Lam_gli.shape[1]):
        e2 = edges2[j]
        # check whether they share a vertex
        if e1[0] in e2 or e1[1] in e2:
            Lam_gli[i, j] = 0
        else:
            # record the position
            e2_pos = [np.array(G2.nodes[e2[0]]['pos']), np.array(G2.nodes[e2[1]]['pos'])]
            # direct computation of Gauss linking integral
            gli_ij = Gauss_linking_integral(e1_pos, e2_pos)
            Lam_gli[i, j] = gli_ij

In [ ]:
# Visualization
cmap_max = 0.1
val_max = np.abs(Lam_gli).max()
val_min = np.abs(Lam_gli[Lam_gli != 0]).min()
num_0s = np.sum(Lam_gli == 0)
print("edges in 1 and 2:", Lam_gli.shape)
print("#0 values in Lam_gli:", num_0s)
print("max abs value in Lam_gli:", val_max, "min abs value:", val_min)

In [ ]:
cmap_max = min([cmap_max, val_max])
print("max in cmap:", cmap_max)

plt.figure(figsize=(6, 3.5))
# plt.imshow(Lam_gli, cmap='seismic', vmax=val_max, vmin=-val_max) #, aspect='auto'
plt.imshow(Lam_gli, cmap='seismic', vmax=cmap_max, vmin=-cmap_max) #, aspect='auto'

# Optional: show grid or colorbar
plt.grid(False)
plt.colorbar(shrink=0.6)
plt.savefig("{}-Lam_gli-max{}.png".format(lab_G, cmap_max), dpi=300, bbox_inches='tight')
plt.show()

We see that there are very small values in the GLI operator, should we treat them as zero (i.e., no contribution) or as a contribution?
- Let's maintain the values and explore the patterns there.

In [ ]:
### linking centrality ###
# set the same range (from exploratory analysis)
cmin = 0.008588
cmax = 2.0399
crange = [cmin, cmax]

In [ ]:
### linking centrality ###
figsize_c = [800, 700]

# rows (edges1) and columns (edges2) that are not all zeros
row_sum = np.sum(abs(Lam_gli), axis=1)
col_sum = np.sum(abs(Lam_gli), axis=0)

print("node size:", ns, "line width:", lw)

cmap = 'plasma'
pos1 = nx.get_node_attributes(G1, 'pos')
pos2 = nx.get_node_attributes(G2, 'pos')

plot_networkx_colore(G1, G2, pos1, pos2, row_sum, col_sum, cmap=cmap, xval=0.80,
                     ns=ns, lw=lw, camera=camera, savefig=True, figname=lab_G+'-unic-G-cent',
                     figsize=figsize_c, val_range=crange)

In [ ]:
# bipartite graph clustering
### Construct the bipartite signed graph
fac = 1000 # scale up the values, for numerical reasons
Lam_gfac = Lam_gli * fac
# labels_p1 = ['({},{})'.format(e[0], e[1]) for e in edges1]
# labels_p2 = ['({},{})'.format(e[0], e[1]) for e in edges2]

# Gb = build_sibigraph(Lam_gtol, labels_p1, labels_p2)

# use index instead
labels_p1 = [i for i in range(len(edges1))]
labels_p2 = [i+len(edges1) for i in range(len(edges2))]

Gb = build_sibigraph(Lam_gfac, labels_p1, labels_p2)
print("#nodes:", Gb.number_of_nodes(), "#edges:", Gb.number_of_edges())

# prepare connected components if any
Gb_ccs = [Gb.subgraph(c) for c in nx.connected_components(Gb)]
print("#connected components:", len(Gb_ccs))
print("sizes:", [(g.number_of_nodes(), g.number_of_edges()) for g in Gb_ccs])

In [ ]:
### Following tasks inside each cc ###
idx_g = 0

Gb_cc = Gb_ccs[idx_g]

# get the adjacency
nodes_cc = sorted(Gb_cc.nodes())
W_cc = nx.to_numpy_array(Gb_cc, nodelist=nodes_cc)

# get edges lists
edges1_cc = [edges1[i] for i in nodes_cc if i < len(edges1)]
edges2_cc = [edges2[i-len(edges1)] for i in nodes_cc if i >= len(edges1)]
print("two parts:", (len(edges1_cc), len(edges2_cc)))


# check the balance
deg = np.sum(abs(W_cc), axis=1)

# symmetric version
D2_inv = np.diag(1./np.sqrt(deg))
P_sym = D2_inv.dot(W_cc).dot(D2_inv)
eigvals = np.linalg.eigvals(P_sym)
print("Max eigenval of P:", np.max(eigvals))
# print("eigenvalues of P:", eigvals)

In [ ]:
# Laplacian
D = np.diag(deg)
L = D - W_cc
eigvals, eigvecs = np.linalg.eigh(L)
print("Min eigenvalues of L:", min(eigvals)/fac)

# normalized Laplacian
L_sym = np.eye(W_cc.shape[0]) - P_sym
eigvals, eigvecs = np.linalg.eigh(L_sym)
print("Min eigenvalues of L sym:", min(eigvals))

# indices of the smallest value
tol = 1e-2
eval_min = min(eigvals)
idx = [i for i, val in enumerate(eigvals) if abs(val-eval_min) < tol]
print("Min eigenvalue(s) of L sym:", eigvals[idx])
print("first few eigenvalues:", eigvals[:5])

plt.figure(figsize=(18, 5))
plt.subplot(121)
# choose the eigenvector(s)
for i in idx:
    plt.plot(eigvecs[:, i], marker='.', linestyle='dashed')
plt.grid(True)

plt.subplot(122)
plt.plot(eigvals, marker='.', linestyle='dashed')
plt.grid(True)
plt.show()

In [ ]:
### Bi-Clustering ###
tol = 1e-4
nodec = []
idx_0s = []

for i,val in enumerate(eigvecs[:, idx[0]]):
    if val > tol:
        nodec.append(2)
    elif val < -tol:
        nodec.append(0)
    else:
        idx_0s.append(i)
        nodec.append(1)

if len(idx_0s) == 0:
    print("clear cut!")
    nodec = [1 if nodec[i] > 0 else 0 for i in range(len(nodec))]
else:
    print("# zeros in the eigenvector:", len(idx_0s))

In [ ]:
# allocation zeros
from itertools import combinations

s = np.array([1. if nodec[i] > 0 else -1. for i in range(len(nodec))]) # default setting where all in the 1-group
best_cost = s.T.dot(L).dot(s)
best_partition = [s]
size = len(idx_0s)
idx_0s = np.array(idx_0s)

res = []
for num in range(1, int((size+1.)/2.)+1):
    for idx_c in combinations(list(range(size)), num):
        s_now = s.copy()
        s_now[idx_0s[list(idx_c)]] = -1 # allocate these edges to the -1-group
        cost = s_now.T.dot(L).dot(s_now)
        if cost < best_cost-tol:
            print("change to -1:", idx_c)
            best_cost = cost
            best_partition = [s_now]
        elif abs(cost-best_cost) < tol:
            best_partition.append(s_now)

print("Best cost:", best_cost/fac)
# print("Best partition:", best_partition)

In [ ]:
### Post processing ###
# apply the optimal orientation
s = best_partition[0]
S = np.diag(s)
W_s = S.dot(W_cc.dot(S))
B_s = W_s[:len(edges1_cc), :][:, -len(edges2_cc):]

# Distribution of the linking values
plt.hist((B_s/fac).flatten(), bins=100)
plt.yscale('log')
plt.show()

In [ ]:
# select strong negative signals
negr, negc = np.where(B_s < -0.1*fac)
print("#very negative edges:", len(negr))

# record
res = [(edges1_cc[negr[i]], edges2_cc[negc[i]]) for i in range(len(negr))]
res_idx = [((n2i_dict[e1[0]], n2i_dict[e1[1]]), (n2i_dict[e2[0]], n2i_dict[e2[1]])) for e1,e2 in res]

# medium
negr, negc = np.where(B_s < -0.05*fac)
print("#medium negative edges:", len(negr))

# record
res2 = [(edges1_cc[negr[i]], edges2_cc[negc[i]]) for i in range(len(negr))]
res2_idx = [((n2i_dict[e1[0]], n2i_dict[e1[1]]), (n2i_dict[e2[0]], n2i_dict[e2[1]])) for e1,e2 in res]

# visualization
edges1_str = [epair[0] for epair in res]
edges2_str = [epair[1] for epair in res]

edges1_med = [epair[0] for epair in res2]
edges2_med = [epair[1] for epair in res2]

In [ ]:
## select the colors
alphas = [0.3, 0.5, 1.]
color1 = "rgba(0, 0, 139, {})"
color2 = "rgba(139, 0, 0, {})"

colorset = [[color1.format(a), color2.format(a)] for a in alphas]
# change the middle color
alpha = 0.9
colorset[1] = ["rgba(33, 89, 209,{})".format(alpha), "rgba(196, 29, 29,{})".format(alpha)]

In [ ]:
lab_fac = '-f{}'.format(fac)
# ns = 3.5
# lw = 5
print("node size:", ns, "line width:", lw)

plot_networkx_2colore_sel(G_sel2, G1, G2, edges1_str, edges2_str, edges1_med, edges2_med,
                          ns=ns, lw=lw, camera=camera, savefig=True, colorset=colorset,
                          figsize=figsize_g, figname=lab_G+lab_fac+'-unic-G-signed-l2m')

###### cut = 5

In [ ]:
# Case 1: when # above and below are about the same
ns = 2.
lw = 3.5
cut = 5

## Visualization
# --- 1. Select edges for G1 (minRadiusAvg < cut) ---
edges_G1 = [(u, v, data) for u, v, data in G_sel2.edges(data=True) if data["minRadiusAvg"] < cut]

# --- 2. Select remaining edges for G2 (minRadiusAvg >= cut OR missing) ---
edges_G2 = [(u, v, data) for u, v, data in G_sel2.edges(data=True) if data["minRadiusAvg"] >= cut]

# print("#edges:", len(edges_G1), len(edges_G2))

# --- 3. Build subgraphs while preserving attributes ---
G1 = nx.Graph()
G1.add_nodes_from(G_sel2.nodes(data=True))
G1.add_edges_from(edges_G1)

G2 = nx.Graph()
G2.add_nodes_from(G_sel2.nodes(data=True))
G2.add_edges_from(edges_G2)

# --- 4. Remove isolated nodes ---
# G1.remove_nodes_from(list(nx.isolates(G1)))
# G2.remove_nodes_from(list(nx.isolates(G2)))

pos1 = nx.get_node_attributes(G1, 'pos')
pos2 = nx.get_node_attributes(G2, 'pos')
lab_G = 'vessel-z1-c{}'.format(cut)

In [ ]:
# edge colors
alpha = 0.9
row_c = "rgba(33, 89, 209,{})".format(alpha)
col_c = "rgba(196, 29, 29,{})".format(alpha)
# tick title sizes
ax_fz = 30
# figsize
figsize_g = [550, 700]

plot_networkx_12nolabel(G1, G2, pos1, pos2, size=ns, width=lw, row_c=row_c, col_c=col_c,
                        camera=camera, savefig=True, figname=lab_G+'-unic-G', figsize=figsize_g, ax_fz=ax_fz) #row_c='#2159d1', col_c='#c41d1d'

In [ ]:
## GLI analysis
# compute the GLI operator
# GLI between edges directly
edges1 = list(G1.edges())
edges2 = list(G2.edges())

Lam_gli = np.zeros((len(edges1), len(edges2)))
for i in range(Lam_gli.shape[0]):
    e1 = edges1[i]
    e1_pos = [np.array(G1.nodes[e1[0]]['pos']), np.array(G1.nodes[e1[1]]['pos'])]
    for j in range(Lam_gli.shape[1]):
        e2 = edges2[j]
        # check whether they share a vertex
        if e1[0] in e2 or e1[1] in e2:
            Lam_gli[i, j] = 0
        else:
            # record the position
            e2_pos = [np.array(G2.nodes[e2[0]]['pos']), np.array(G2.nodes[e2[1]]['pos'])]
            # direct computation of Gauss linking integral
            gli_ij = Gauss_linking_integral(e1_pos, e2_pos)
            Lam_gli[i, j] = gli_ij

In [ ]:
# Visualization
cmap_max = 0.1
val_max = np.abs(Lam_gli).max()
val_min = np.abs(Lam_gli[Lam_gli != 0]).min()
num_0s = np.sum(Lam_gli == 0)
print("edges in 1 and 2:", Lam_gli.shape)
print("#0 values in Lam_gli:", num_0s)
print("max abs value in Lam_gli:", val_max, "min abs value:", val_min)

In [ ]:
cmap_max = min([cmap_max, val_max])
print("max in cmap:", cmap_max)

plt.figure(figsize=(6, 3.5))
# plt.imshow(Lam_gli, cmap='seismic', vmax=val_max, vmin=-val_max) #, aspect='auto'
plt.imshow(Lam_gli, cmap='seismic', vmax=cmap_max, vmin=-cmap_max) #, aspect='auto'

# Optional: show grid or colorbar
plt.grid(False)
plt.colorbar(shrink=0.6)
plt.savefig("{}-Lam_gli-max{}.png".format(lab_G, cmap_max), dpi=300, bbox_inches='tight')
plt.show()

We see that there are very small values in the GLI operator, should we treat them as zero (i.e., no contribution) or as a contribution?
- Let's maintain the values and explore the patterns there.

In [ ]:
### linking centrality ###
# set the same range (from exploratory analysis)
cmin = 0.008588
cmax = 2.0399
crange = [cmin, cmax]

In [ ]:
### linking centrality ###
figsize_c = [800, 700]

# rows (edges1) and columns (edges2) that are not all zeros
row_sum = np.sum(abs(Lam_gli), axis=1)
col_sum = np.sum(abs(Lam_gli), axis=0)

print("node size:", ns, "line width:", lw)

cmap = 'plasma'
pos1 = nx.get_node_attributes(G1, 'pos')
pos2 = nx.get_node_attributes(G2, 'pos')

plot_networkx_colore(G1, G2, pos1, pos2, row_sum, col_sum, cmap=cmap, xval=0.80,
                     ns=ns, lw=lw, camera=camera, savefig=True, figname=lab_G+'-unic-G-cent',
                     figsize=figsize_c, val_range=crange)

In [ ]:
# bipartite graph clustering
### Construct the bipartite signed graph
fac = 100 # scale up the values, for numerical reasons
Lam_gfac = Lam_gli * fac
# labels_p1 = ['({},{})'.format(e[0], e[1]) for e in edges1]
# labels_p2 = ['({},{})'.format(e[0], e[1]) for e in edges2]

# Gb = build_sibigraph(Lam_gtol, labels_p1, labels_p2)

# use index instead
labels_p1 = [i for i in range(len(edges1))]
labels_p2 = [i+len(edges1) for i in range(len(edges2))]

Gb = build_sibigraph(Lam_gfac, labels_p1, labels_p2)
print("#nodes:", Gb.number_of_nodes(), "#edges:", Gb.number_of_edges())

# prepare connected components if any
Gb_ccs = [Gb.subgraph(c) for c in nx.connected_components(Gb)]
print("#connected components:", len(Gb_ccs))
print("sizes:", [(g.number_of_nodes(), g.number_of_edges()) for g in Gb_ccs])

In [ ]:
### Following tasks inside each cc ###
idx_g = 0

Gb_cc = Gb_ccs[idx_g]

# get the adjacency
nodes_cc = sorted(Gb_cc.nodes())
W_cc = nx.to_numpy_array(Gb_cc, nodelist=nodes_cc)

# get edges lists
edges1_cc = [edges1[i] for i in nodes_cc if i < len(edges1)]
edges2_cc = [edges2[i-len(edges1)] for i in nodes_cc if i >= len(edges1)]
print("two parts:", (len(edges1_cc), len(edges2_cc)))


# check the balance
deg = np.sum(abs(W_cc), axis=1)

# symmetric version
D2_inv = np.diag(1./np.sqrt(deg))
P_sym = D2_inv.dot(W_cc).dot(D2_inv)
eigvals = np.linalg.eigvals(P_sym)
print("Max eigenval of P:", np.max(eigvals))
# print("eigenvalues of P:", eigvals)

In [ ]:
# Laplacian
D = np.diag(deg)
L = D - W_cc
eigvals, eigvecs = np.linalg.eigh(L)
print("Min eigenvalues of L:", min(eigvals)/fac)

# normalized Laplacian
L_sym = np.eye(W_cc.shape[0]) - P_sym
eigvals, eigvecs = np.linalg.eigh(L_sym)
print("Min eigenvalues of L sym:", min(eigvals))

# indices of the smallest value
tol = 0.01
eval_min = min(eigvals)
idx = [i for i, val in enumerate(eigvals) if abs(val-eval_min) < tol]
print("Min eigenvalue(s) of L sym:", eigvals[idx])
print("first few eigenvalues:", eigvals[:5])

plt.figure(figsize=(18, 5))
plt.subplot(121)
# choose the eigenvector(s)
for i in idx:
    plt.plot(eigvecs[:, i], marker='.', linestyle='dashed')
plt.grid(True)

plt.subplot(122)
plt.plot(eigvals, marker='.', linestyle='dashed')
plt.grid(True)
plt.show()

Bi-clustering 1: choose the smallest eigenvalue:

In [ ]:
### Bi-Clustering ###
tol = 1e-4
nodec = []
idx_0s = []

for i,val in enumerate(eigvecs[:, idx[0]]):
    if val > tol:
        nodec.append(2)
    elif val < -tol:
        nodec.append(0)
    else:
        idx_0s.append(i)
        nodec.append(1)

if len(idx_0s) == 0:
    print("clear cut!")
    nodec = [1 if nodec[i] > 0 else 0 for i in range(len(nodec))]
else:
    print("# zeros in the eigenvector:", len(idx_0s))

In [ ]:
# allocation zeros
from itertools import combinations

s = np.array([1. if nodec[i] > 0 else -1. for i in range(len(nodec))]) # default setting where all in the 1-group
best_cost = s.T.dot(L).dot(s)
best_partition = [s]
size = len(idx_0s)
idx_0s = np.array(idx_0s)

res = []
for num in range(1, int((size+1.)/2.)+1):
    for idx_c in combinations(list(range(size)), num):
        s_now = s.copy()
        s_now[idx_0s[list(idx_c)]] = -1 # allocate these edges to the -1-group
        cost = s_now.T.dot(L).dot(s_now)
        if cost < best_cost-tol:
            print("change to -1:", idx_c)
            best_cost = cost
            best_partition = [s_now]
        elif abs(cost-best_cost) < tol:
            best_partition.append(s_now)

print("Best cost:", best_cost/fac)
# print("Best partition:", best_partition)

In [ ]:
### Post processing ###
# apply the optimal orientation
s = best_partition[0]
S = np.diag(s)
W_s = S.dot(W_cc.dot(S))
B_s = W_s[:len(edges1_cc), :][:, -len(edges2_cc):]

# Distribution of the linking values
plt.hist((B_s/fac).flatten(), bins=100)
plt.yscale('log')
plt.show()

In [ ]:
# select strong negative signals
negr, negc = np.where(B_s < -0.1*fac)
print("#very negative edges:", len(negr))

# record
res = [(edges1_cc[negr[i]], edges2_cc[negc[i]]) for i in range(len(negr))]
res_idx = [((n2i_dict[e1[0]], n2i_dict[e1[1]]), (n2i_dict[e2[0]], n2i_dict[e2[1]])) for e1,e2 in res]

# medium
negr, negc = np.where(B_s < -0.05*fac)
print("#medium negative edges:", len(negr))

# record
res2 = [(edges1_cc[negr[i]], edges2_cc[negc[i]]) for i in range(len(negr))]
res2_idx = [((n2i_dict[e1[0]], n2i_dict[e1[1]]), (n2i_dict[e2[0]], n2i_dict[e2[1]])) for e1,e2 in res]

# visualization
edges1_str = [epair[0] for epair in res]
edges2_str = [epair[1] for epair in res]

edges1_med = [epair[0] for epair in res2]
edges2_med = [epair[1] for epair in res2]

In [ ]:
lab_fac = '-f{}'.format(fac)
# ns = 3.5
# lw = 5
print("node size:", ns, "line width:", lw)

plot_networkx_2colore_sel(G_sel2, G1, G2, edges1_str, edges2_str, edges1_med, edges2_med,
                          ns=ns, lw=lw, camera=camera, savefig=True, # colorset=colorset,
                          figsize=figsize_g, figname=lab_G+lab_fac+'-unic-G-signed-l2m-1')

Bi-clustering 2: combine smallest eigenvectors

In [ ]:
# Combine the first two eigenvectors to see whether we have better results?
tol = 1e-4

from itertools import combinations

eigvals, eigvecs = np.linalg.eigh(L_sym)
print("Min eigenvalues of L_sym:", min(eigvals)) # !!! L_sym

# indices of the smallest value
tol_ev = 0.01
eval_min = min(eigvals)
idx = [i for i, val in enumerate(eigvals) if abs(val-eval_min) < tol_ev]
print("Min eigenvalue(s) of L_sym:", eigvals[idx])

u1 = eigvecs[:, idx[0]]
u2 = eigvecs[:, idx[1]]

flip_angles, nodes_deg = critical_flip_angles(u1, u2)
print("#Flip angles:", len(flip_angles))
theta_candidates = candidate_mid_angles(flip_angles)

# Get sign patterns for all candidates
S = sign_patterns_from_angles(u1, u2, theta_candidates)  # shape (n, m)

# For each candidate, evaluate your signed cut cost C(theta_j)
best_cost = np.inf
best_theta = []
best_partition = []

for j, theta in enumerate(theta_candidates):
    s = S[:, j]         # +-1 labels
    # 0 entries?
    idx_0 = np.where(s == 0)[0]
    if len(idx_0) > 0:
        print("0 occurs in eigenvalues")
        ## try all combination of assignments
        # all in the negative one
        s[idx_0] = -1
        cost = s.T.dot(L).dot(s)
        if cost < best_cost-tol:
            best_cost = cost
            best_theta = [theta]
            best_partition = [s]
        elif abs(cost-best_cost) < tol:
            best_theta.append(theta)
            best_partition.append(s)

        # all other combinations
        for num in range(1, len(idx_0)):
            for idx_01 in combinations(idx_0, num):
                s_now = s.copy()
                s_now[idx_0] = -1
                s_now[list(idx_01)] = 1
                cost = s_now.T.dot(L).dot(s_now)
                if cost < best_cost-tol:
                    best_cost = cost
                    best_theta = [theta]
                    best_partition = [s_now]
                elif abs(cost-best_cost) < tol:
                    best_theta.append(theta)
                    best_partition.append(s_now)
        # the last one
        s[idx_0] = 1
        cost = s.T.dot(L).dot(s)
        if cost < best_cost-tol:
            best_cost = cost
            best_theta = [theta]
            best_partition = [s]
        elif abs(cost-best_cost) < tol:
            best_theta.append(theta)
            best_partition.append(s)
    else:
        # compute the objective
        cost = s.T.dot(L).dot(s)
        if cost < best_cost-tol:
            best_cost = cost
            best_theta = [theta]
            best_partition = [s]
        elif abs(cost-best_cost) < tol:
            best_theta.append(theta)
            best_partition.append(s)

print("Best theta:", best_theta)
print("Best cost:", best_cost/fac)
print("# Best partitions:", len(best_partition))

In [ ]:
### Post processing ###
# apply the optimal orientation
s = best_partition[0]
S = np.diag(s)
W_s = S.dot(W_cc.dot(S))
B_s = W_s[:len(edges1_cc), :][:, -len(edges2_cc):]

# Distribution of the linking values
plt.hist((B_s/fac).flatten(), bins=100)
plt.yscale('log')
plt.show()

In [ ]:
# select strong negative signals
negr, negc = np.where(B_s < -0.1*fac)
print("#very negative edges:", len(negr))

# record
res = [(edges1_cc[negr[i]], edges2_cc[negc[i]]) for i in range(len(negr))]
res_idx = [((n2i_dict[e1[0]], n2i_dict[e1[1]]), (n2i_dict[e2[0]], n2i_dict[e2[1]])) for e1,e2 in res]

# medium
negr, negc = np.where(B_s < -0.05*fac)
print("#medium negative edges:", len(negr))

# record
res2 = [(edges1_cc[negr[i]], edges2_cc[negc[i]]) for i in range(len(negr))]
res2_idx = [((n2i_dict[e1[0]], n2i_dict[e1[1]]), (n2i_dict[e2[0]], n2i_dict[e2[1]])) for e1,e2 in res]

# visualization
edges1_str = [epair[0] for epair in res]
edges2_str = [epair[1] for epair in res]

edges1_med = [epair[0] for epair in res2]
edges2_med = [epair[1] for epair in res2]

In [ ]:
alphas = [0.3, 0.5, 1.]
color1 = "rgba(0, 0, 139, {})"
color2 = "rgba(139, 0, 0, {})"

colorset = [[color1.format(a), color2.format(a)] for a in alphas]
# change the middle color
alpha = 0.9
colorset[1] = ["rgba(33, 89, 209,{})".format(alpha), "rgba(196, 29, 29,{})".format(alpha)]

In [ ]:
lab_fac = '-f{}'.format(fac)
# ns = 3.5
# lw = 5
print("node size:", ns, "line width:", lw)

plot_networkx_2colore_sel(G_sel2, G1, G2, edges1_str, edges2_str, edges1_med, edges2_med,
                          ns=ns, lw=lw, camera=camera, savefig=True, colorset=colorset,
                          figsize=figsize_g, figname=lab_G+lab_fac+'-unic-G-signed-l2m-2')

###### cut = 1.5

In [ ]:
# Case 1: when # above and below are about the same
ns = 2.
lw = 3.5
cut = 1.5

## Visualization
# --- 1. Select edges for G1 (minRadiusAvg < cut) ---
edges_G1 = [(u, v, data) for u, v, data in G_sel2.edges(data=True) if data["minRadiusAvg"] < cut]

# --- 2. Select remaining edges for G2 (minRadiusAvg >= cut OR missing) ---
edges_G2 = [(u, v, data) for u, v, data in G_sel2.edges(data=True) if data["minRadiusAvg"] >= cut]

# print("#edges:", len(edges_G1), len(edges_G2))

# --- 3. Build subgraphs while preserving attributes ---
G1 = nx.Graph()
G1.add_nodes_from(G_sel2.nodes(data=True))
G1.add_edges_from(edges_G1)

G2 = nx.Graph()
G2.add_nodes_from(G_sel2.nodes(data=True))
G2.add_edges_from(edges_G2)

# --- 4. Remove isolated nodes ---
# G1.remove_nodes_from(list(nx.isolates(G1)))
# G2.remove_nodes_from(list(nx.isolates(G2)))

pos1 = nx.get_node_attributes(G1, 'pos')
pos2 = nx.get_node_attributes(G2, 'pos')
lab_G = 'vessel-z1-c{}'.format(cut)

In [ ]:
# edge colors
alpha = 0.9
row_c = "rgba(33, 89, 209,{})".format(alpha)
col_c = "rgba(196, 29, 29,{})".format(alpha)
# tick title sizes
ax_fz = 30
# figsize
figsize_g = [550, 700]

plot_networkx_12nolabel(G1, G2, pos1, pos2, size=ns, width=lw, row_c=row_c, col_c=col_c,
                        camera=camera, savefig=True, figname=lab_G+'-unic-G', figsize=figsize_g, ax_fz=ax_fz) #row_c='#2159d1', col_c='#c41d1d'

In [ ]:
## GLI analysis
# compute the GLI operator
# GLI between edges directly
edges1 = list(G1.edges())
edges2 = list(G2.edges())

Lam_gli = np.zeros((len(edges1), len(edges2)))
for i in range(Lam_gli.shape[0]):
    e1 = edges1[i]
    e1_pos = [np.array(G1.nodes[e1[0]]['pos']), np.array(G1.nodes[e1[1]]['pos'])]
    for j in range(Lam_gli.shape[1]):
        e2 = edges2[j]
        # check whether they share a vertex
        if e1[0] in e2 or e1[1] in e2:
            Lam_gli[i, j] = 0
        else:
            # record the position
            e2_pos = [np.array(G2.nodes[e2[0]]['pos']), np.array(G2.nodes[e2[1]]['pos'])]
            # direct computation of Gauss linking integral
            gli_ij = Gauss_linking_integral(e1_pos, e2_pos)
            Lam_gli[i, j] = gli_ij

In [ ]:
# Visualization
cmap_max = 0.1
val_max = np.abs(Lam_gli).max()
val_min = np.abs(Lam_gli[Lam_gli != 0]).min()
num_0s = np.sum(Lam_gli == 0)
print("edges in 1 and 2:", Lam_gli.shape)
print("#0 values in Lam_gli:", num_0s)
print("max abs value in Lam_gli:", val_max, "min abs value:", val_min)

In [ ]:
cmap_max = min([cmap_max, val_max])
print("max in cmap:", cmap_max)

plt.figure(figsize=(6, 3.5))
# plt.imshow(Lam_gli, cmap='seismic', vmax=val_max, vmin=-val_max) #, aspect='auto'
plt.imshow(Lam_gli, cmap='seismic', vmax=cmap_max, vmin=-cmap_max) #, aspect='auto'

# Optional: show grid or colorbar
plt.grid(False)
plt.colorbar(shrink=0.6)
plt.savefig("{}-Lam_gli-max{}.png".format(lab_G, cmap_max), dpi=300, bbox_inches='tight')
plt.show()

We see that there are very small values in the GLI operator, should we treat them as zero (i.e., no contribution) or as a contribution?
- Let's maintain the values and explore the patterns there.

In [ ]:
### linking centrality ###
# set the same range (from exploratory analysis)
cmin = 0.008588
cmax = 2.0399
crange = [cmin, cmax]

In [ ]:
### linking centrality ###
figsize_c = [800, 700]

# rows (edges1) and columns (edges2) that are not all zeros
row_sum = np.sum(abs(Lam_gli), axis=1)
col_sum = np.sum(abs(Lam_gli), axis=0)

print("node size:", ns, "line width:", lw)

cmap = 'plasma'
pos1 = nx.get_node_attributes(G1, 'pos')
pos2 = nx.get_node_attributes(G2, 'pos')

plot_networkx_colore(G1, G2, pos1, pos2, row_sum, col_sum, cmap=cmap, xval=0.80,
                     ns=ns, lw=lw, camera=camera, savefig=True, figname=lab_G+'-unic-G-cent',
                     figsize=figsize_c, val_range=crange)

In [ ]:
# bipartite graph clustering
### Construct the bipartite signed graph
fac = 100 # scale up the values, for numerical reasons
Lam_gfac = Lam_gli * fac
# labels_p1 = ['({},{})'.format(e[0], e[1]) for e in edges1]
# labels_p2 = ['({},{})'.format(e[0], e[1]) for e in edges2]

# Gb = build_sibigraph(Lam_gtol, labels_p1, labels_p2)

# use index instead
labels_p1 = [i for i in range(len(edges1))]
labels_p2 = [i+len(edges1) for i in range(len(edges2))]

Gb = build_sibigraph(Lam_gfac, labels_p1, labels_p2)
print("#nodes:", Gb.number_of_nodes(), "#edges:", Gb.number_of_edges())

# prepare connected components if any
Gb_ccs = [Gb.subgraph(c) for c in nx.connected_components(Gb)]
print("#connected components:", len(Gb_ccs))
print("sizes:", [(g.number_of_nodes(), g.number_of_edges()) for g in Gb_ccs])

In [ ]:
### Following tasks inside each cc ###
idx_g = 0

Gb_cc = Gb_ccs[idx_g]

# get the adjacency
nodes_cc = sorted(Gb_cc.nodes())
W_cc = nx.to_numpy_array(Gb_cc, nodelist=nodes_cc)

# get edges lists
edges1_cc = [edges1[i] for i in nodes_cc if i < len(edges1)]
edges2_cc = [edges2[i-len(edges1)] for i in nodes_cc if i >= len(edges1)]
print("two parts:", (len(edges1_cc), len(edges2_cc)))


# check the balance
deg = np.sum(abs(W_cc), axis=1)

# symmetric version
D2_inv = np.diag(1./np.sqrt(deg))
P_sym = D2_inv.dot(W_cc).dot(D2_inv)
eigvals = np.linalg.eigvals(P_sym)
print("Max eigenval of P:", np.max(eigvals))
# print("eigenvalues of P:", eigvals)

In [ ]:
# Laplacian
D = np.diag(deg)
L = D - W_cc
eigvals, eigvecs = np.linalg.eigh(L)
print("Min eigenvalues of L:", min(eigvals)/fac)

# normalized Laplacian
L_sym = np.eye(W_cc.shape[0]) - P_sym
eigvals, eigvecs = np.linalg.eigh(L_sym)
print("Min eigenvalues of L sym:", min(eigvals))

# indices of the smallest value
tol = 1e-2
eval_min = min(eigvals)
idx = [i for i, val in enumerate(eigvals) if abs(val-eval_min) < tol]
print("Min eigenvalue(s) of L sym:", eigvals[idx])
print("first few eigenvalues:", eigvals[:5])

plt.figure(figsize=(18, 5))
plt.subplot(121)
# choose the eigenvector(s)
for i in idx:
    plt.plot(eigvecs[:, i], marker='.', linestyle='dashed')
plt.grid(True)

plt.subplot(122)
plt.plot(eigvals, marker='.', linestyle='dashed')
plt.grid(True)
plt.show()

In [ ]:
### Bi-Clustering ###
tol = 1e-4
nodec = []
idx_0s = []

for i,val in enumerate(eigvecs[:, idx[0]]):
    if val > tol:
        nodec.append(2)
    elif val < -tol:
        nodec.append(0)
    else:
        idx_0s.append(i)
        nodec.append(1)

if len(idx_0s) == 0:
    print("clear cut!")
    nodec = [1 if nodec[i] > 0 else 0 for i in range(len(nodec))]
else:
    print("# zeros in the eigenvector:", len(idx_0s))

In [ ]:
# allocation zeros
from itertools import combinations

s = np.array([1. if nodec[i] > 0 else -1. for i in range(len(nodec))]) # default setting where all in the 1-group
best_cost = s.T.dot(L).dot(s)
best_partition = [s]
size = len(idx_0s)
idx_0s = np.array(idx_0s)

res = []
for num in range(1, int((size+1.)/2.)+1):
    for idx_c in combinations(list(range(size)), num):
        s_now = s.copy()
        s_now[idx_0s[list(idx_c)]] = -1 # allocate these edges to the -1-group
        cost = s_now.T.dot(L).dot(s_now)
        if cost < best_cost-tol:
            print("change to -1:", idx_c)
            best_cost = cost
            best_partition = [s_now]
        elif abs(cost-best_cost) < tol:
            best_partition.append(s_now)

print("Best cost:", best_cost/fac)
# print("Best partition:", best_partition)

In [ ]:
### Post processing ###
# apply the optimal orientation
s = best_partition[0]
S = np.diag(s)
W_s = S.dot(W_cc.dot(S))
B_s = W_s[:len(edges1_cc), :][:, -len(edges2_cc):]

# Distribution of the linking values
plt.hist((B_s/fac).flatten(), bins=100)
plt.yscale('log')
plt.show()

In [ ]:
# select strong negative signals
negr, negc = np.where(B_s < -0.1*fac)
print("#very negative edges:", len(negr))

# record
res = [(edges1_cc[negr[i]], edges2_cc[negc[i]]) for i in range(len(negr))]
res_idx = [((n2i_dict[e1[0]], n2i_dict[e1[1]]), (n2i_dict[e2[0]], n2i_dict[e2[1]])) for e1,e2 in res]

# medium
negr, negc = np.where(B_s < -0.05*fac)
print("#medium negative edges:", len(negr))

# record
res2 = [(edges1_cc[negr[i]], edges2_cc[negc[i]]) for i in range(len(negr))]
res2_idx = [((n2i_dict[e1[0]], n2i_dict[e1[1]]), (n2i_dict[e2[0]], n2i_dict[e2[1]])) for e1,e2 in res]

# visualization
edges1_str = [epair[0] for epair in res]
edges2_str = [epair[1] for epair in res]

edges1_med = [epair[0] for epair in res2]
edges2_med = [epair[1] for epair in res2]

In [ ]:
lab_fac = '-f{}'.format(fac)
# ns = 3.5
# lw = 5
print("node size:", ns, "line width:", lw)

plot_networkx_2colore_sel(G_sel2, G1, G2, edges1_str, edges2_str, edges1_med, edges2_med,
                          ns=ns, lw=lw, camera=camera, savefig=True, colorset=colorset,
                          figsize=figsize_g, figname=lab_G+lab_fac+'-unic-G-signed-l2m')

#### Zone II: mixed (Midbrain)

In [ ]:
# --- Step 1: select a region ---
xlim = [1026., 1091.]
ylim = [2785., 2885.]
zlim = [945., 1175.]

In [ ]:
nodes_sel, regs_sum = select_vertices_range(df_nodes, xlim, ylim, zlim)
print("regions:", regs_sum[0])
print("counts:", regs_sum[1])

In [ ]:
# --- Step 2: select edges touching at least one selected node ---
df_edges_sel = df_edges[df_edges["node1id"].isin(nodes_sel) | df_edges["node2id"].isin(nodes_sel)]

# --- Step 3: update node set with all incident nodes ---
nodes_sel2 = set(df_edges_sel["node1id"]).union(set(df_edges_sel["node2id"]))
df_nodes_sel2 = df_nodes[df_nodes["id"].isin(nodes_sel2)]

print(f"Initial selected nodes: {len(nodes_sel)}")
print(f"Selected edges: {len(df_edges_sel)}")
print(f"Updated node set: {len(nodes_sel2)}")

In [ ]:
# create a node label dictionary
n2i_dict = {id:i for i,id in enumerate(df_nodes_sel2['id'].values)}
print("#nodes in dictionary", len(n2i_dict))

In [ ]:
# check the regions
np.unique(df_nodes_sel2['region_label'].values, return_counts=True)

In [ ]:
# Region labels and colors
region_categories = np.unique(df_nodes_sel2['region_label'].values)
print(region_categories)

# region_color_map = {
#     region_categories[0]: "red",
#     region_categories[1]: "blue",
#     region_categories[2]: "black",
# }
region_color_map = {
    region_categories[0]: '#7F7F7F',
    region_categories[1]: '#DD4477',
    region_categories[2]: '#7F7F7F', # grey /'#8C564B'
    region_categories[3]: '#FF7F0E', # orange
    region_categories[4]: '#109618', # green
    region_categories[5]: '#620042', # dark brown
    region_categories[6]: '#9467BD',
    region_categories[7]: '#F58518',
    region_categories[8]: '#BCBD22', # dark brown
}

In [ ]:
# Build a fast lookup for node positions - edges color ~ radius
ns = 2.5
lw = 3.5
xval = 0.8
cmap = 'coolwarm'

pos_dict = df_nodes_sel2.set_index("id")[
    ["pos_x", "pos_y", "pos_z"]
].to_dict("index")

# prepare the color map
r_vals = df_edges_sel["minRadiusAvg"].values
val_min, val_max = r_vals.min(), r_vals.max()
print("max, min abs value in edge vals:", val_max, val_min)

# Normalize + convert colormap
colorscale = matplotlib_to_plotly_colorscale(cmap)
# Colorbar
colorbar_trace = dummy_colorbar_trace(val_min, val_max, colorscale, xval=xval)

# assign colors
norm = mcolors.Normalize(vmin=val_min, vmax=val_max)
cmap_vals = plt.colormaps.get_cmap(cmap)

edge_traces = []
for _, edge in df_edges_sel.iterrows():
    n1, n2 = edge["node1id"], edge["node2id"]

    if (n1 not in pos_dict) or (n2 not in pos_dict):
        continue

    x0, y0, z0 = pos_dict[n1]["pos_x"], pos_dict[n1]["pos_y"], pos_dict[n1]["pos_z"]
    x1, y1, z1 = pos_dict[n2]["pos_x"], pos_dict[n2]["pos_y"], pos_dict[n2]["pos_z"]

    r = edge["minRadiusAvg"]
    # normalized_r = (r - r_min) / (r_max - r_min + 1e-12)
    color = mcolors.to_hex(cmap_vals(norm(r)))
    edge_traces.append(go.Scatter3d(
            x=[x0, x1, None],
            y=[y0, y1, None],
            z=[z0, z1, None],
            mode="lines",
            line=dict(color=color, width=lw),
            hoverinfo="text",
            text=f"Radius: {r:.3f}",
            showlegend=False,
        ))

node_traces = []
for region, color in region_color_map.items():
    df_r = df_nodes_sel2[df_nodes_sel2["region_label"] == region]

    node_traces.append(
        go.Scatter3d(
            x=df_r["pos_x"],
            y=df_r["pos_y"],
            z=df_r["pos_z"],
            mode="markers",
            name=f"Region {region}",
            marker=dict(
                size=ns,
                color=color,
                opacity=0.9,
            ),
            hoverinfo="text",
            text=[
                f"Node {nid}<br>Region: {region}"
                for nid in df_r["id"]
            ],
        )
    )

fig = go.Figure(data=[*node_traces, *edge_traces, colorbar_trace])

axisFormat = dict(
                showbackground=False,
                showticklabels=False,
                autorange=True,
                showgrid=True,
        )


fig.update_layout(
    # title="3D Brain Vascular Subnetwork",
    scene=dict(
        xaxis=dict(title="x"),
        yaxis=dict(title="y"),
        zaxis=dict(title="z"),
        aspectmode="data",
    ),
    margin=dict(l=0, r=0, b=0, t=0),
    # legend=dict(itemsizing="constant"),
)

fig.show()

In [ ]:
#set an appropriate camera (for later plots as well)
camera = dict(
    eye=dict(x=1.63, y=1., z=.6), # eye=dict(x=1.8, y=1.1, z=.5),
    center=dict(x=0, y=0, z=0),
    up=dict(x=0, y=0, z=.1)
)
fig.update_layout(scene_camera=camera)
fig.show()

In [ ]:
pio.write_image(fig, "vessel-z2_view.png", width=700, height=600, scale=2) #, scale=2

In [ ]:
# output the range
full_fig = fig.full_figure_for_development()
print(full_fig.layout.scene.xaxis.range)
print(full_fig.layout.scene.yaxis.range)
print(full_fig.layout.scene.zaxis.range)

##### Edge counts

In [ ]:
### see how #edges change along the radius cuts
col_rad = "minRadiusAvg"
# cuts = np.arange(1., 16., 0.5)
cuts = sorted(df_edges_sel[col_rad].unique()[1:-1])

summary = []
for cut in cuts:
    # summarize the edges below and above
    num_below = len(df_edges_sel[df_edges_sel[col_rad] < cut])
    num_above = len(df_edges_sel[df_edges_sel[col_rad] >= cut])

    summary.append([cut, num_below, num_above])

In [ ]:
# visualize the summary
tk_fz = 14
lb_fz = 16
plt.figure(figsize=(7,3.5))
summary = np.array(summary)
plt.plot(summary[:, 0], summary[:, 1], '.--', c='b', alpha=0.6)
plt.plot(summary[:, 0], summary[:, 2], '.--', c='r', alpha=0.6)
# plt.yscale('log')
plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('# edges', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.savefig("vessel-z2-edgrad.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# visualize the summary + cuts
cut_sel = [1.5, 2.5, 5]

plt.figure(figsize=(7,3.5))
# plot vertical lines
for cut in cut_sel:
    plt.axvline(x=cut, linestyle='--', color='k', alpha=0.5)

summary = np.array(summary)
plt.plot(summary[:, 0], summary[:, 1], '.--', c='b', alpha=0.6)
plt.plot(summary[:, 0], summary[:, 2], '.--', c='r', alpha=0.6)
# plt.yscale('log')
plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('# edges', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.savefig("vessels-z2-edgradc.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
## Q: how does the edge distance correlate with radius?
res_dist = []
res_radi= []
G2 = build_region_graphs(df_nodes_sel2, df_edges_sel)

for _, row in df_edges_sel.iterrows():
    n1, n2 = row["node1id"], row["node2id"]
    res_radi.append(float(row["minRadiusAvg"]))

    # compute the distance between the two nodes
    pos1 = np.array(G2.nodes[n1]['pos'])
    pos2 = np.array(G2.nodes[n2]['pos'])
    dist = np.linalg.norm(pos1-pos2)
    res_dist.append(dist)

In [ ]:
# pearson correlation
from scipy import stats

corr = stats.pearsonr(res_dist, res_radi)
print("correlation:", corr)

In [ ]:
plt.figure(figsize=(5,3.8))
plt.scatter(res_dist, res_radi, marker='.', alpha=.5)
plt.xlabel('Distance')
plt.ylabel('Radius')
plt.savefig("vessels2-dist-radius.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(5,3.8))
plt.scatter(res_dist, res_radi, marker='.', alpha=.5)
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Distance')
plt.ylabel('Radius')
plt.savefig("vessels2-dist-radius-log.png", dpi=300, bbox_inches='tight')
plt.show()

##### Unbalance scores + Centralities

In [ ]:
### See how balance changes with cut
# sort edges
col_rad = "minRadiusAvg"
df_edges_sel = df_edges_sel.sort_values(by=col_rad)

G_sel = build_region_graphs(df_nodes_sel2, df_edges_sel)

## compute the full GLI between each pair of edges in order
Lgli_full = np.zeros((len(df_edges_sel), len(df_edges_sel)))
for i in range(Lgli_full.shape[0]):
    e1 = [df_edges_sel.iloc[i]['node1id'], df_edges_sel.iloc[i]['node2id']]
    e1_pos = [np.array(G_sel.nodes[e1[0]]['pos']), np.array(G_sel.nodes[e1[1]]['pos'])]
    for j in range(i+1, Lgli_full.shape[1]):
        e2 = [df_edges_sel.iloc[j]['node1id'], df_edges_sel.iloc[j]['node2id']]
        # check whether they share a vertex
        if e1[0] in e2 or e1[1] in e2:
            Lgli_full[i, j] = 0
        else:
            # record the position
            e2_pos = [np.array(G_sel.nodes[e2[0]]['pos']), np.array(G_sel.nodes[e2[1]]['pos'])]
            # direct computation of Gauss linking integral
            gli_ij = Gauss_linking_integral(e1_pos, e2_pos)
            Lgli_full[i, j] = gli_ij

In [ ]:
### See how *centrality distribution* changes with cut
res_cut = []

res_bal = []
res_cme = []
res_csd = []
res_cax = []
idx = 0
for _, row in df_edges_sel.iterrows():

    # obtain the linking
    Lam_gli = Lgli_full[:idx+1, :][:, idx+1:]
    print("Lam_gli shape:", Lam_gli.shape)

    if Lam_gli.shape[1] == 0:
        continue

    # record the cut
    res_cut.append(float(row[col_rad]))
    print("cut:", res_cut[-1])

    idx += 1

    # weigh matrix of the bipartite graph
    W_gli = np.block([[np.zeros((Lam_gli.shape[0], Lam_gli.shape[0])), Lam_gli],
                      [Lam_gli.T, np.zeros((Lam_gli.shape[1], Lam_gli.shape[1]))]])

    ## Balance score
    # prepare connected components if any
    Gb = nx.from_numpy_array(W_gli)
    Gb_ccs = [Gb.subgraph(c) for c in nx.connected_components(Gb)]
    print("#connected components:", len(Gb_ccs))
    print("sizes:", [(g.number_of_nodes(), g.number_of_edges()) for g in Gb_ccs])

    # for each cc
    res = []
    for idx_g in range(len(Gb_ccs)):
        Gb_cc = Gb_ccs[idx_g]
        if Gb_cc.number_of_nodes() < 3:
            continue
        # get the adjacency
        W_cc = nx.to_numpy_array(Gb_cc)
        deg = np.sum(abs(W_cc), axis=1)

        # symmetric version
        D2_inv = np.diag(1./np.sqrt(deg))
        P_sym = D2_inv.dot(W_cc).dot(D2_inv)
        eigvals = np.linalg.eigvals(P_sym)
        # print("Max eigenval of P:", np.max(eigvals))
        res.append(np.max(eigvals))
    print("balance score:", res)
    # set one balance score for each cut value
    if len(res) == 0:
        res.append(np.nan)
    else:
        res_bal.append(np.mean(res).real)

    ## Centralities
    res = np.sum(abs(W_gli), axis=0)
    res_cme.append(np.mean(res))
    res_csd.append(np.std(res))
    res_cax.append(np.max(res))

1. Unbalance score

In [ ]:
res_cut = np.array(res_cut)
res_bal = np.array(res_bal)

In [ ]:
# save data
df_res = pd.DataFrame({'cut': res_cut, 'bal': res_bal})
df_res.to_csv("vessels-z2-balrad.csv", index=False)

In [ ]:
plt.figure(figsize=(7,3.5))
# plt.scatter(cut_arr, bal_arr, marker='.', alpha=0.6)
plt.plot(res_cut, 1-res_bal, marker='.', linestyle='dashed', alpha=0.6, c='blueviolet')
plt.xlabel('Radius cuts')
plt.ylabel('Unalance score')
plt.savefig("vessels-z2-unbalrad-l.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(7,3.5))
for cut in cut_sel:
    plt.axvline(x=cut, linestyle='--', color='k', alpha=0.5)

# plt.scatter(cut_arr, bal_arr, marker='.', alpha=0.6)
plt.plot(res_cut, 1-res_bal, marker='.', linestyle='dashed', alpha=0.6, c='blueviolet')
plt.xlabel('Radius cuts')
plt.ylabel('Unbalance score')
plt.savefig("vessels-z2-unbalrad-lc.png", dpi=300, bbox_inches='tight')
plt.show()

2. Centrality

In [ ]:
# set one balance score for each cut value
res_cut = np.array(res_cut)
res_cme = np.array(res_cme)
res_csd = np.array(res_csd)
res_cax = np.array(res_cax)

In [ ]:
# save data
df_res = pd.DataFrame({'cut': res_cut, 'cent-mean': res_cme, 'cent-std': res_csd, 'cent-max': res_cax})
df_res.to_csv("vessels-z2-cenrad.csv", index=False)

In [ ]:
plt.figure(figsize=(7,3.5))
plt.plot(res_cut, res_cme, marker='.', linestyle='dashed', alpha=0.6, c='orangered')
plt.fill_between(res_cut, res_cme-res_csd, res_cme+res_csd, alpha=0.2, color='orangered')
plt.xlabel('Radius cuts')
plt.ylabel('Centrality')
plt.savefig("vessels-z2-cenrad-l.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(7,3.5))
for cut in cut_sel:
    plt.axvline(x=cut, linestyle='--', color='k', alpha=0.5)

plt.plot(res_cut, res_cme, marker='.', linestyle='dashed', alpha=0.6, c='orangered')
plt.fill_between(res_cut, res_cme-res_csd, res_cme+res_csd, alpha=0.2, color='orangered')
plt.xlabel('Radius cuts')
plt.ylabel('Centrality')
plt.savefig("vessels-z2-cenrad-lc.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(7,3.5))
plt.plot(res_cut, res_cax, marker='.', linestyle='dashed', alpha=0.6, c='orangered')
plt.xlabel('Radius cuts')
plt.ylabel('Centrality')
plt.savefig("vessels-z2-cmxrad-l.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(7,3.5))
for cut in cut_sel:
    plt.axvline(x=cut, linestyle='--', color='k', alpha=0.5)
plt.plot(res_cut, res_cax, marker='.', linestyle='dashed', alpha=0.6, c='orangered')
plt.xlabel('Radius cuts')
plt.ylabel('Centrality')
plt.savefig("vessels-z2-cmxrad-lc.png", dpi=300, bbox_inches='tight')
plt.show()

###### Null model 1: SER and SRG

In [ ]:
# random sample and compute the unbalance score for comparison
num_sam = 10

# choose the model
lab_mod = ['er', 'dist'][1]
p_dist = True if lab_mod == 'dist' else False
print("probability by distance?", p_dist)

num_nodes = len(nodes_sel2)
num_edges = len(df_edges_sel)
# Use empirical bounding box for the null positions
xyz_ranges = ((xlim[0], xlim[1]), (ylim[0], ylim[1]), (zlim[0], zlim[1]))

col_rad = "minRadiusAvg"

resall_cut = []
# balance
resall_bal = []
# centrality
resall_cme = []
resall_csd = []
resall_cax = []

for s in range(num_sam):
    print("sample:", s)

    ### generate null models
    nodes_null_df, edges_null_df, G_null = generate_spatial_null_model(
        n_nodes=num_nodes, m_edges=num_edges, df_edges_sel=df_edges_sel,
        xyz_ranges=xyz_ranges, directed=False, p_dist=p_dist)

    ### sweep the unbalance score
    # sort edges
    edges_null_df = edges_null_df.sort_values(by=col_rad)

    # compute the full GLI between each pair of edges in order
    Lgli_full = np.zeros((len(edges_null_df), len(edges_null_df)))
    for i in range(Lgli_full.shape[0]):
        e1 = [edges_null_df.iloc[i]['node1id'], edges_null_df.iloc[i]['node2id']]
        e1_pos = [np.array(G_null.nodes[e1[0]]['pos']), np.array(G_null.nodes[e1[1]]['pos'])]
        for j in range(i+1, Lgli_full.shape[1]):
            e2 = [edges_null_df.iloc[j]['node1id'], edges_null_df.iloc[j]['node2id']]
            # check whether they share a vertex
            if e1[0] in e2 or e1[1] in e2:
                Lgli_full[i, j] = 0
            else:
                # record the position
                e2_pos = [np.array(G_null.nodes[e2[0]]['pos']), np.array(G_null.nodes[e2[1]]['pos'])]
                # direct computation of Gauss linking integral
                gli_ij = Gauss_linking_integral(e1_pos, e2_pos)
                Lgli_full[i, j] = gli_ij
    # compute the balance score
    res_cut = []
    res_bal = []
    res_cme = []
    res_csd = []
    res_cax = []
    idx = 0
    for _, row in edges_null_df.iterrows():

        # obtain the linking
        Lam_gli = Lgli_full[:idx+1, :][:, idx+1:]
        print("Lam_gli shape:", Lam_gli.shape)

        if Lam_gli.shape[1] == 0:
            continue

        # record the cut
        res_cut.append(float(row[col_rad]))
        print("cut:", res_cut[-1])

        idx += 1
        # characterize the balance
        W_gli = np.block([[np.zeros((Lam_gli.shape[0], Lam_gli.shape[0])), Lam_gli],
                          [Lam_gli.T, np.zeros((Lam_gli.shape[1], Lam_gli.shape[1]))]])

        # prepare connected components if any
        Gb = nx.from_numpy_array(W_gli)
        Gb_ccs = [Gb.subgraph(c) for c in nx.connected_components(Gb)]
        print("#connected components:", len(Gb_ccs))
        print("sizes:", [(g.number_of_nodes(), g.number_of_edges()) for g in Gb_ccs])

        # for each cc
        res = []
        for idx_g in range(len(Gb_ccs)):
            Gb_cc = Gb_ccs[idx_g]
            if Gb_cc.number_of_nodes() < 3:
                continue
            # get the adjacency
            W_cc = nx.to_numpy_array(Gb_cc)
            deg = np.sum(abs(W_cc), axis=1)

            # symmetric version
            D2_inv = np.diag(1./np.sqrt(deg))
            P_sym = D2_inv.dot(W_cc).dot(D2_inv)
            eigvals = np.linalg.eigvals(P_sym)
            # print("Max eigenval of P:", np.max(eigvals))
            res.append(np.max(eigvals))
        print("balance score:", res)
        # set one balance score for each cut value
        if len(res) == 0:
            res.append(np.nan)
        else:
            res_bal.append(np.mean(res).real)

        ## Centrality
        # record the distribution of centralities
        res = np.sum(abs(W_gli), axis=0)
        res_cme.append(np.mean(res))
        res_csd.append(np.std(res))
        res_cax.append(np.max(res))

    # store the results
    resall_cut.append(res_cut)
    resall_bal.append(res_bal)
    resall_cme.append(res_cme)
    resall_csd.append(res_csd)
    resall_cax.append(res_cax)

1. Unbalance score

In [ ]:
cut_arr = np.array(resall_cut).mean(axis=0)
cut_std = np.array(resall_cut).std(axis=0)
# check the std of cuts
print("Cut std:", min(cut_std), max(cut_std))

bal_arr = np.array(resall_bal).mean(axis=0)
bal_std = np.array(resall_bal).std(axis=0)

In [ ]:
# plot average +- standard deviation
plt.figure(figsize=(7,3.5))
plt.plot(cut_arr, 1-bal_arr, linestyle='dashed', marker='.', alpha=0.6, c='k')
plt.fill_between(cut_arr, 1-bal_arr-bal_std, 1-bal_arr+bal_std, alpha=0.2, color='grey')
plt.xlabel('Radius cuts')
plt.ylabel('Balance score')
plt.savefig("vessels-z2-unbalrad-sw-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# save the data
df_res = pd.DataFrame({'cut': cut_arr, 'bal': bal_arr, 'bal_std': bal_std})
df_res.to_csv("vessels-z2-unbalrad-sw-{}-s{}.csv".format(lab_mod, num_sam), index=False)

2. Centrality

In [ ]:
cut_arr = np.array(resall_cut).mean(axis=0)
cut_std = np.array(resall_cut).std(axis=0)
# check the std of cuts
print("Cut std:", min(cut_std), max(cut_std))

cme_arr = np.array(resall_cme).mean(axis=0)
cme_std = np.array(resall_cme).std(axis=0)
csd_arr = np.array(resall_csd).mean(axis=0)
csd_std = np.array(resall_csd).std(axis=0)
cax_arr = np.array(resall_cax).mean(axis=0)
cax_std = np.array(resall_cax).std(axis=0)

In [ ]:
  # plot average +- standard deviation: centrality mean & centrality deviation
plt.figure(figsize=(7,3.5))
plt.plot(cut_arr, cme_arr, linestyle='dashed', marker='.', alpha=0.6, c='k')
plt.fill_between(cut_arr, cme_arr-csd_arr, cme_arr+csd_arr, alpha=0.2, color='grey')
plt.xlabel('Radius cuts')
plt.ylabel('Centrality')
plt.savefig("vessels-z2-cenrad-sw-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot average +- standard deviation: centrality mean
plt.figure(figsize=(7,3.5))
plt.plot(cut_arr, cme_arr, linestyle='dashed', marker='.', alpha=0.6, c='k')
plt.fill_between(cut_arr, cme_arr-cme_std, cme_arr+cme_std, alpha=0.2, color='grey')
plt.xlabel('Radius cuts')
plt.ylabel('Centrality')
plt.savefig("vessels-z2-cmerad-sw-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot average +- standard deviation: centrality max
plt.figure(figsize=(7,3.5))
plt.plot(cut_arr, cax_arr, linestyle='dashed', marker='.', alpha=0.6, c='k')
plt.fill_between(cut_arr, cax_arr-cax_std, cax_arr+cax_std, alpha=0.2, color='grey')
plt.xlabel('Radius cuts')
plt.ylabel('Centrality')
plt.savefig("vessels-z2-caxrad-sw-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# save the data
df_res = pd.DataFrame({'cut': cut_arr, 'cent-mean-mean': cme_arr, 'cent-mean-std': cme_std,
                       'cent-std-mean': csd_arr, 'cent-std-std': csd_std,
                       'cent-max-mean': cax_arr, 'cent-max-std': cax_std})
df_res.to_csv("vessels-z2-cenrad-sw-{}-s{}.csv".format(lab_mod, num_sam), index=False)

###### Null model 2: shuffle radius

In [ ]:
# random sample and compute the unbalance score for comparison
num_sam = 10

# choose the model
lab_mod = ['radi'][0]
print("null model", lab_mod)

num_nodes = len(nodes_sel2)
num_edges = len(df_edges_sel)
# Use empirical bounding box for the null positions
xyz_ranges = ((xlim[0], xlim[1]), (ylim[0], ylim[1]), (zlim[0], zlim[1]))

col_rad = "minRadiusAvg"

resall_cut = []
# balance
resall_bal = []
# centrality
resall_cut = []
resall_cme = []
resall_csd = []
resall_cax = []

for s in range(num_sam):
    print("sample:", s)

    ### generate null models
    edges_null_df, G_null = generate_radii_null_model(
        m_edges=num_edges, df_edges_sel=df_edges_sel, df_nodes_sel=df_nodes_sel2, directed=False)

    ### sweep the unbalance score
    # sort edges
    edges_null_df = edges_null_df.sort_values(by=col_rad)

    # compute the full GLI between each pair of edges in order
    Lgli_full = np.zeros((len(edges_null_df), len(edges_null_df)))
    for i in range(Lgli_full.shape[0]):
        e1 = [edges_null_df.iloc[i]['node1id'], edges_null_df.iloc[i]['node2id']]
        e1_pos = [np.array(G_null.nodes[e1[0]]['pos']), np.array(G_null.nodes[e1[1]]['pos'])]
        for j in range(i+1, Lgli_full.shape[1]):
            e2 = [edges_null_df.iloc[j]['node1id'], edges_null_df.iloc[j]['node2id']]
            # check whether they share a vertex
            if e1[0] in e2 or e1[1] in e2:
                Lgli_full[i, j] = 0
            else:
                # record the position
                e2_pos = [np.array(G_null.nodes[e2[0]]['pos']), np.array(G_null.nodes[e2[1]]['pos'])]
                # direct computation of Gauss linking integral
                gli_ij = Gauss_linking_integral(e1_pos, e2_pos)
                Lgli_full[i, j] = gli_ij

    # compute the balance score
    res_cut = []
    res_bal = []
    res_bal = []
    res_cme = []
    res_csd = []
    res_cax = []
    idx = 0
    for _, row in edges_null_df.iterrows():

        # obtain the linking
        Lam_gli = Lgli_full[:idx+1, :][:, idx+1:]
        print("Lam_gli shape:", Lam_gli.shape)

        if Lam_gli.shape[1] == 0:
            continue

        # record the cut
        res_cut.append(float(row[col_rad]))
        print("cut:", res_cut[-1])

        idx += 1
        # weight matrix of the bipartite graph
        W_gli = np.block([[np.zeros((Lam_gli.shape[0], Lam_gli.shape[0])), Lam_gli],
                          [Lam_gli.T, np.zeros((Lam_gli.shape[1], Lam_gli.shape[1]))]])

        ## Balance
        # prepare connected components if any
        Gb = nx.from_numpy_array(W_gli)
        Gb_ccs = [Gb.subgraph(c) for c in nx.connected_components(Gb)]
        print("#connected components:", len(Gb_ccs))
        print("sizes:", [(g.number_of_nodes(), g.number_of_edges()) for g in Gb_ccs])

        # for each cc
        res = []
        for idx_g in range(len(Gb_ccs)):
            Gb_cc = Gb_ccs[idx_g]
            if Gb_cc.number_of_nodes() < 3:
                continue
            # get the adjacency
            W_cc = nx.to_numpy_array(Gb_cc)
            deg = np.sum(abs(W_cc), axis=1)

            # symmetric version
            D2_inv = np.diag(1./np.sqrt(deg))
            P_sym = D2_inv.dot(W_cc).dot(D2_inv)
            eigvals = np.linalg.eigvals(P_sym)
            # print("Max eigenval of P:", np.max(eigvals))
            res.append(np.max(eigvals))
        print("balance score:", res)
        # set one balance score for each cut value
        if len(res) == 0:
            res.append(np.nan)
        else:
            res_bal.append(np.mean(res).real)

        ## Centrality
        # record the distribution of centralities
        res = np.sum(abs(W_gli), axis=0)
        res_cme.append(np.mean(res))
        res_csd.append(np.std(res))
        res_cax.append(np.max(res))

    # store the results
    resall_cut.append(res_cut)
    resall_bal.append(res_bal)
    resall_cme.append(res_cme)
    resall_csd.append(res_csd)
    resall_cax.append(res_cax)

1. Unbalance score

In [ ]:
cut_arr = np.array(resall_cut).mean(axis=0)
cut_std = np.array(resall_cut).std(axis=0)
# check the std of cuts
print("Cut std:", min(cut_std), max(cut_std))

bal_arr = np.array(resall_bal).mean(axis=0)
bal_std = np.array(resall_bal).std(axis=0)

In [ ]:
# plot average +- standard deviation
plt.figure(figsize=(7,3.5))
plt.plot(cut_arr, 1-bal_arr, linestyle='dashed', marker='.', alpha=0.6, c='k')
plt.fill_between(cut_arr, 1-bal_arr-bal_std, 1-bal_arr+bal_std, alpha=0.2, color='grey')
plt.xlabel('Radius cuts')
plt.ylabel('Balance score')
plt.savefig("vessels-z2-unbalrad-null-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# save the data
df_res = pd.DataFrame({'cut': cut_arr, 'bal': bal_arr, 'bal_std': bal_std})
df_res.to_csv("vessels-z2-unbalrad-null-{}-s{}.csv".format(lab_mod, num_sam), index=False)

In [ ]:
### comparison
# read in data
df_bal = pd.read_csv("vessels-z2-balrad.csv")
df_radi = pd.read_csv("vessels-z2-unbalrad-null-radi-s10.csv")

In [ ]:
colors = ['r', 'b', 'y', 'c']
# plot together
plt.figure(figsize=(7,3.5))

# null model - radius
plt.plot(df_radi['cut'], 1-df_radi['bal'], linestyle='dashed', marker='.', alpha=0.3, c=colors[3], label='Rad')
plt.fill_between(df_radi['cut'], 1-df_radi['bal']-df_radi['bal_std'], 1-df_radi['bal']+df_radi['bal_std'], alpha=0.2, color=colors[3])

# actual value
plt.plot(df_bal['cut'], 1-df_bal['bal'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.xlabel('Radius cuts')
plt.ylabel('Unbalance score')
plt.legend()
plt.savefig("vessels-z2-unbalrad-null2.png", dpi=300, bbox_inches='tight')
plt.show()

2. Centrality

In [ ]:
cut_arr = np.array(resall_cut).mean(axis=0)
cut_std = np.array(resall_cut).std(axis=0)
# check the std of cuts
print("Cut std:", min(cut_std), max(cut_std))

cme_arr = np.array(resall_cme).mean(axis=0)
cme_std = np.array(resall_cme).std(axis=0)
csd_arr = np.array(resall_csd).mean(axis=0)
csd_std = np.array(resall_csd).std(axis=0)
cax_arr = np.array(resall_cax).mean(axis=0)
cax_std = np.array(resall_cax).std(axis=0)

In [ ]:
# plot average +- standard deviation: centrality mean & centrality deviation
plt.figure(figsize=(7,3.5))
plt.plot(cut_arr, cme_arr, linestyle='dashed', marker='.', alpha=0.6, c='k')
plt.fill_between(cut_arr, cme_arr-csd_arr, cme_arr+csd_arr, alpha=0.2, color='grey')
plt.xlabel('Radius cuts')
plt.ylabel('Centrality')
plt.savefig("vessels-z2-cenrad-null-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot average +- standard deviation: centrality mean
plt.figure(figsize=(7,3.5))
plt.plot(cut_arr, cme_arr, linestyle='dashed', marker='.', alpha=0.6, c='k')
plt.fill_between(cut_arr, cme_arr-cme_std, cme_arr+cme_std, alpha=0.2, color='grey')
plt.xlabel('Radius cuts')
plt.ylabel('Centrality Mean')
plt.savefig("vessels-z2-cmerad-null-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot average +- standard deviation: centrality max
plt.figure(figsize=(7,3.5))
plt.plot(cut_arr, cax_arr, linestyle='dashed', marker='.', alpha=0.6, c='k')
plt.fill_between(cut_arr, cax_arr-cax_std, cax_arr+cax_std, alpha=0.2, color='grey')
plt.xlabel('Radius cuts')
plt.ylabel('Centrality Max')
plt.savefig("vessels-z2-caxrad-null-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# save the data
df_res = pd.DataFrame({'cut': cut_arr, 'cent-mean-mean': cme_arr, 'cent-mean-std': cme_std,
                       'cent-std-mean': csd_arr, 'cent-std-std': csd_std,
                       'cent-max-mean': cax_arr, 'cent-max-std': cax_std})
df_res.to_csv("vessels-z2-cenrad-null-{}-s{}.csv".format(lab_mod, num_sam), index=False)

In [ ]:
### comparison
# read in data
df_cen = pd.read_csv("vessels-z2-cenrad.csv")
df_radi = pd.read_csv("vessels-z2-cenrad-null-radi-s10.csv")

In [ ]:
colors = ['r', 'b', 'y', 'c']
# plot together
plt.figure(figsize=(7,3.5))

# null model - radius
plt.plot(df_radi['cut'], df_radi['cent-mean-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[3], label='Rad')
plt.fill_between(df_radi['cut'], df_radi['cent-mean-mean']-df_radi['cent-mean-std'],
                 df_radi['cent-mean-mean']+df_radi['cent-mean-std'], alpha=0.2, color=colors[3])

# actual value
plt.plot(df_cen['cut'], df_cen['cent-mean'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.xlabel('Radius cuts')
plt.ylabel('Centrality Mean')
plt.legend()
plt.savefig("vessels-z2-cmerad-null2.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot together
plt.figure(figsize=(7,3.5))

# null model - radius
plt.plot(df_radi['cut'], df_radi['cent-max-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[3], label='Rad')
plt.fill_between(df_radi['cut'], df_radi['cent-max-mean']-df_radi['cent-max-std'],
                 df_radi['cent-max-mean']+df_radi['cent-max-std'], alpha=0.2, color=colors[3])

# actual value
plt.plot(df_cen['cut'], df_cen['cent-max'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.xlabel('Radius cuts')
plt.ylabel('Centrality Max')
plt.legend()
plt.savefig("vessels-z2-caxrad-null2.png", dpi=300, bbox_inches='tight')
plt.show()

###### Summary

1. Unbalance scores

In [ ]:
## ALL
df_bal = pd.read_csv("vessels-z2-balrad.csv")
df_radi = pd.read_csv("vessels-z2-unbalrad-null-radi-s10.csv")
df_er = pd.read_csv("vessels-z2-unbalrad-sw-er-s10.csv")
df_dist = pd.read_csv("vessels-z2-unbalrad-sw-dist-s10.csv")

In [ ]:
colors = ['r', 'b', 'y', 'c']
# plot together
tk_fz = 14
lb_fz = 16
lg_fz = 12

plt.figure(figsize=(7,3.5))

# null model - radius
plt.plot(df_radi['cut'], 1-df_radi['bal'], linestyle='dashed', marker='.', alpha=0.3, c=colors[3], label='Rad')
plt.fill_between(df_radi['cut'], 1-df_radi['bal']-df_radi['bal_std'], 1-df_radi['bal']+df_radi['bal_std'], alpha=0.2, color=colors[3])

# null model - ER
plt.plot(df_er['cut'], 1-df_er['bal'], linestyle='dashed', marker='.', alpha=0.3, c=colors[2], label='SER')
plt.fill_between(df_er['cut'], 1-df_er['bal']-df_er['bal_std'], 1-df_er['bal']+df_er['bal_std'], alpha=0.2, color=colors[2])

# null model - dist
plt.plot(df_dist['cut'], 1-df_dist['bal'], linestyle='dashed', marker='.', alpha=0.3, c=colors[1], label='SRG')
plt.fill_between(df_dist['cut'], 1-df_dist['bal']-df_dist['bal_std'], 1-df_dist['bal']+df_dist['bal_std'], alpha=0.2, color=colors[1])

# actual value
plt.plot(df_bal['cut'], 1-df_bal['bal'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('unbalance score', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.legend(fontsize=lg_fz, frameon=False)
plt.savefig("vessels-z2-unbalrad-all.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot together
plt.figure(figsize=(7,3.5))

# null model - ER
plt.plot(df_er['cut'], 1-df_er['bal'], linestyle='dashed', marker='.', alpha=0.3, c=colors[2], label='SER')
plt.fill_between(df_er['cut'], 1-df_er['bal']-df_er['bal_std'], 1-df_er['bal']+df_er['bal_std'], alpha=0.2, color=colors[2])

# null model - dist
plt.plot(df_dist['cut'], 1-df_dist['bal'], linestyle='dashed', marker='.', alpha=0.3, c=colors[1], label='SRG')
plt.fill_between(df_dist['cut'], 1-df_dist['bal']-df_dist['bal_std'], 1-df_dist['bal']+df_dist['bal_std'], alpha=0.2, color=colors[1])

# actual value
plt.plot(df_bal['cut'], 1-df_bal['bal'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.ylim(ylim_unb)
plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('unbalance score', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.legend(fontsize=lg_fz, frameon=False)
plt.savefig("vessels-z2-unbalrad-all3.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot together
plt.figure(figsize=(7,3.5))

# null model - radius
plt.plot(df_radi['cut'], 1-df_radi['bal'], linestyle='dashed', marker='.', alpha=0.3, c=colors[3], label='Rad')
plt.fill_between(df_radi['cut'], 1-df_radi['bal']-df_radi['bal_std'], 1-df_radi['bal']+df_radi['bal_std'], alpha=0.2, color=colors[3])


# null model - dist
plt.plot(df_dist['cut'], 1-df_dist['bal'], linestyle='dashed', marker='.', alpha=0.3, c=colors[1], label='SRG')
plt.fill_between(df_dist['cut'], 1-df_dist['bal']-df_dist['bal_std'], 1-df_dist['bal']+df_dist['bal_std'], alpha=0.2, color=colors[1])

# actual value
plt.plot(df_bal['cut'], 1-df_bal['bal'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('unbalance score', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.legend(fontsize=lg_fz, frameon=False)
plt.savefig("vessels-z2-unbalrad-null3.png", dpi=300, bbox_inches='tight')
plt.show()

2. Centrality

In [ ]:
# read in data
df_cen = pd.read_csv("vessels-z2-cenrad.csv")
df_radi = pd.read_csv("vessels-z2-cenrad-null-radi-s10.csv")
df_er = pd.read_csv("vessels-z2-cenrad-sw-er-s10.csv")
df_dist = pd.read_csv("vessels-z2-cenrad-sw-dist-s10.csv")

In [ ]:
colors = ['orangered', 'b', 'y', 'c']

In [ ]:
# plot together
plt.figure(figsize=(7,3.5))

# null model - radius
plt.plot(df_radi['cut'], df_radi['cent-mean-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[3], label='Rad')
plt.fill_between(df_radi['cut'], df_radi['cent-mean-mean']-df_radi['cent-mean-std'],
                 df_radi['cent-mean-mean']+df_radi['cent-mean-std'], alpha=0.2, color=colors[3])

# null model - ER
plt.plot(df_er['cut'], df_er['cent-mean-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[2], label='SER')
plt.fill_between(df_er['cut'], df_er['cent-mean-mean']-df_er['cent-mean-std'],
                 df_er['cent-mean-mean']+df_er['cent-mean-std'], alpha=0.2, color=colors[2])

# null model - dist
plt.plot(df_dist['cut'], df_dist['cent-mean-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[1], label='SRG')
plt.fill_between(df_dist['cut'], df_dist['cent-mean-mean']-df_dist['cent-mean-std'],
                 df_dist['cent-mean-mean']+df_dist['cent-mean-std'], alpha=0.2, color=colors[1])

# actual value
plt.plot(df_cen['cut'], df_cen['cent-mean'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

# plt.yscale('log')
plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('centrality mean', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.legend(fontsize=lg_fz, frameon=False)
plt.savefig("vessels-z2-cmerad-all.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot together
plt.figure(figsize=(7,3.5))

# null model - radius
plt.plot(df_radi['cut'], df_radi['cent-mean-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[3], label='Rad')
plt.fill_between(df_radi['cut'], df_radi['cent-mean-mean']-df_radi['cent-mean-std'],
                 df_radi['cent-mean-mean']+df_radi['cent-mean-std'], alpha=0.2, color=colors[3])

# null model - ER
plt.plot(df_er['cut'], df_er['cent-mean-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[2], label='SER')
plt.fill_between(df_er['cut'], df_er['cent-mean-mean']-df_er['cent-mean-std'],
                 df_er['cent-mean-mean']+df_er['cent-mean-std'], alpha=0.2, color=colors[2])

# null model - dist
plt.plot(df_dist['cut'], df_dist['cent-mean-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[1], label='SRG')
plt.fill_between(df_dist['cut'], df_dist['cent-mean-mean']-df_dist['cent-mean-std'],
                 df_dist['cent-mean-mean']+df_dist['cent-mean-std'], alpha=0.2, color=colors[1])

# actual value
plt.plot(df_cen['cut'], df_cen['cent-mean'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.yscale('log')
plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('centrality mean', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.legend(fontsize=lg_fz, frameon=False)
plt.savefig("vessels-z2-cmerad-all-log.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(7,3.5))

# null model - ER
plt.plot(df_er['cut'], df_er['cent-mean-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[2], label='SER')
plt.fill_between(df_er['cut'], df_er['cent-mean-mean']-df_er['cent-mean-std'],
                 df_er['cent-mean-mean']+df_er['cent-mean-std'], alpha=0.2, color=colors[2])

# null model - dist
plt.plot(df_dist['cut'], df_dist['cent-mean-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[1], label='SRG')
plt.fill_between(df_dist['cut'], df_dist['cent-mean-mean']-df_dist['cent-mean-std'],
                 df_dist['cent-mean-mean']+df_dist['cent-mean-std'], alpha=0.2, color=colors[1])

# actual value
plt.plot(df_cen['cut'], df_cen['cent-mean'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.ylim(ylim_cme)
plt.yscale('log')
plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('centrality mean', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.legend(fontsize=lg_fz, frameon=False)
plt.savefig("vessels-z2-cmerad-all3-log.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(7,3.5))

# null model - radius
plt.plot(df_radi['cut'], df_radi['cent-mean-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[3], label='Rad')
plt.fill_between(df_radi['cut'], df_radi['cent-mean-mean']-df_radi['cent-mean-std'],
                 df_radi['cent-mean-mean']+df_radi['cent-mean-std'], alpha=0.2, color=colors[3])

# null model - dist
plt.plot(df_dist['cut'], df_dist['cent-mean-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[1], label='SRG')
plt.fill_between(df_dist['cut'], df_dist['cent-mean-mean']-df_dist['cent-mean-std'],
                 df_dist['cent-mean-mean']+df_dist['cent-mean-std'], alpha=0.2, color=colors[1])

# actual value
plt.plot(df_cen['cut'], df_cen['cent-mean'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.yscale('log')
plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('centrality mean', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.legend(fontsize=lg_fz, frameon=False)
plt.savefig("vessels-z2-cmerad-null3-log.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot together
plt.figure(figsize=(7,3.5))

# null model - radius
plt.plot(df_radi['cut'], df_radi['cent-max-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[3], label='Rad')
plt.fill_between(df_radi['cut'], df_radi['cent-max-mean']-df_radi['cent-max-std'],
                 df_radi['cent-max-mean']+df_radi['cent-max-std'], alpha=0.2, color=colors[3])

# null model - ER
plt.plot(df_er['cut'], df_er['cent-max-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[2], label='SER')
plt.fill_between(df_er['cut'], df_er['cent-max-mean']-df_er['cent-max-std'],
                 df_er['cent-max-mean']+df_er['cent-max-std'], alpha=0.2, color=colors[2])

# null model - dist
plt.plot(df_dist['cut'], df_dist['cent-max-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[1], label='SRG')
plt.fill_between(df_dist['cut'], df_dist['cent-max-mean']-df_dist['cent-max-std'],
                 df_dist['cent-max-mean']+df_dist['cent-max-std'], alpha=0.2, color=colors[1])

# actual value
plt.plot(df_cen['cut'], df_cen['cent-max'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

# plt.yscale('log')
plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('centrality max', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.legend(fontsize=lg_fz, frameon=False)
plt.savefig("vessels-z2-caxrad-all.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot together
plt.figure(figsize=(7,3.5))

# null model - radius
plt.plot(df_radi['cut'], df_radi['cent-max-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[3], label='Rad')
plt.fill_between(df_radi['cut'], df_radi['cent-max-mean']-df_radi['cent-max-std'],
                 df_radi['cent-max-mean']+df_radi['cent-max-std'], alpha=0.2, color=colors[3])

# null model - ER
plt.plot(df_er['cut'], df_er['cent-max-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[2], label='SER')
plt.fill_between(df_er['cut'], df_er['cent-max-mean']-df_er['cent-max-std'],
                 df_er['cent-max-mean']+df_er['cent-max-std'], alpha=0.2, color=colors[2])

# null model - dist
plt.plot(df_dist['cut'], df_dist['cent-max-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[1], label='SRG')
plt.fill_between(df_dist['cut'], df_dist['cent-max-mean']-df_dist['cent-max-std'],
                 df_dist['cent-max-mean']+df_dist['cent-max-std'], alpha=0.2, color=colors[1])

# actual value
plt.plot(df_cen['cut'], df_cen['cent-max'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.yscale('log')
plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('centrality max', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.legend(fontsize=lg_fz, frameon=False)
plt.savefig("vessels-z2-caxrad-all-log.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot together
plt.figure(figsize=(7,3.5))

# null model - ER
plt.plot(df_er['cut'], df_er['cent-max-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[2], label='SER')
plt.fill_between(df_er['cut'], df_er['cent-max-mean']-df_er['cent-max-std'],
                 df_er['cent-max-mean']+df_er['cent-max-std'], alpha=0.2, color=colors[2])

# null model - dist
plt.plot(df_dist['cut'], df_dist['cent-max-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[1], label='SRG')
plt.fill_between(df_dist['cut'], df_dist['cent-max-mean']-df_dist['cent-max-std'],
                 df_dist['cent-max-mean']+df_dist['cent-max-std'], alpha=0.2, color=colors[1])

# actual value
plt.plot(df_cen['cut'], df_cen['cent-max'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.ylim(ylim_cax)
plt.yscale('log')
plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('centrality max', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.legend(fontsize=lg_fz, frameon=False, loc='lower center')
plt.savefig("vessels-z2-caxrad-all3-log.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot together
plt.figure(figsize=(7,3.5))

# null model - radius
plt.plot(df_radi['cut'], df_radi['cent-max-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[3], label='Rad')
plt.fill_between(df_radi['cut'], df_radi['cent-max-mean']-df_radi['cent-max-std'],
                 df_radi['cent-max-mean']+df_radi['cent-max-std'], alpha=0.2, color=colors[3])

# null model - dist
plt.plot(df_dist['cut'], df_dist['cent-max-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[1], label='SRG')
plt.fill_between(df_dist['cut'], df_dist['cent-max-mean']-df_dist['cent-max-std'],
                 df_dist['cent-max-mean']+df_dist['cent-max-std'], alpha=0.2, color=colors[1])

# actual value
plt.plot(df_cen['cut'], df_cen['cent-max'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.yscale('log')
plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('centrality max', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.legend(fontsize=lg_fz, frameon=False)
plt.savefig("vessels-z2-caxrad-null3-log.png", dpi=300, bbox_inches='tight')
plt.show()

##### Single cut

In [ ]:
# Is the graph connected?
G_sel2 = build_region_graphs(df_nodes_sel2, df_edges_sel)
component_summary = summarize_cc_topk(G_sel2, k=10)

In [ ]:
figsize_g = [500, 550]
figsize_c = [500, 550]

###### cut = 2.5

In [ ]:
# Case 1: when # above and below are about the same
ns = 2.
lw = 3.5
cut = 2.5

## Visualization
# --- 1. Select edges for G1 (minRadiusAvg < cut) ---
edges_G1 = [(u, v, data) for u, v, data in G_sel2.edges(data=True) if data["minRadiusAvg"] < cut]

# --- 2. Select remaining edges for G2 (minRadiusAvg >= cut OR missing) ---
edges_G2 = [(u, v, data) for u, v, data in G_sel2.edges(data=True) if data["minRadiusAvg"] >= cut]

# print("#edges:", len(edges_G1), len(edges_G2))

# --- 3. Build subgraphs while preserving attributes ---
G1 = nx.Graph()
G1.add_nodes_from(G_sel2.nodes(data=True))
G1.add_edges_from(edges_G1)

G2 = nx.Graph()
G2.add_nodes_from(G_sel2.nodes(data=True))
G2.add_edges_from(edges_G2)

# --- 4. Remove isolated nodes ---
# G1.remove_nodes_from(list(nx.isolates(G1)))
# G2.remove_nodes_from(list(nx.isolates(G2)))

pos1 = nx.get_node_attributes(G1, 'pos')
pos2 = nx.get_node_attributes(G2, 'pos')
lab_G = 'vessel-z2-c{}'.format(cut)

In [ ]:
# edge colors
alpha = 0.9
row_c = "rgba(33, 89, 209,{})".format(alpha)
col_c = "rgba(196, 29, 29,{})".format(alpha)
# tick title sizes
ax_fz = 30
# figsize
figsize_g = [500, 550]

plot_networkx_12nolabel(G1, G2, pos1, pos2, size=ns, width=lw, row_c=row_c, col_c=col_c,
                        camera=camera, savefig=True, figname=lab_G+'-unic-G', figsize=figsize_g, ax_fz=ax_fz) #row_c='#2159d1', col_c='#c41d1d'

In [ ]:
## GLI analysis
# compute the GLI operator
# GLI between edges directly
edges1 = list(G1.edges())
edges2 = list(G2.edges())

Lam_gli = np.zeros((len(edges1), len(edges2)))
for i in range(Lam_gli.shape[0]):
    e1 = edges1[i]
    e1_pos = [np.array(G1.nodes[e1[0]]['pos']), np.array(G1.nodes[e1[1]]['pos'])]
    for j in range(Lam_gli.shape[1]):
        e2 = edges2[j]
        # check whether they share a vertex
        if e1[0] in e2 or e1[1] in e2:
            Lam_gli[i, j] = 0
        else:
            # record the position
            e2_pos = [np.array(G2.nodes[e2[0]]['pos']), np.array(G2.nodes[e2[1]]['pos'])]
            # direct computation of Gauss linking integral
            gli_ij = Gauss_linking_integral(e1_pos, e2_pos)
            Lam_gli[i, j] = gli_ij

In [ ]:
# Visualization
cmap_max = 0.1
val_max = np.abs(Lam_gli).max()
val_min = np.abs(Lam_gli[Lam_gli != 0]).min()
num_0s = np.sum(Lam_gli == 0)
print("edges in 1 and 2:", Lam_gli.shape)
print("#0 values in Lam_gli:", num_0s)
print("max abs value in Lam_gli:", val_max, "min abs value:", val_min)

In [ ]:
cmap_max = min([cmap_max, val_max])
print("max in cmap:", cmap_max)

plt.figure(figsize=(6, 3.5))
# plt.imshow(Lam_gli, cmap='seismic', vmax=val_max, vmin=-val_max) #, aspect='auto'
plt.imshow(Lam_gli, cmap='seismic', vmax=cmap_max, vmin=-cmap_max) #, aspect='auto'

# Optional: show grid or colorbar
plt.grid(False)
plt.colorbar(shrink=0.6)
plt.savefig("{}-Lam_gli-max{}.png".format(lab_G, cmap_max), dpi=300, bbox_inches='tight')
plt.show()

We see that there are very small values in the GLI operator, should we treat them as zero (i.e., no contribution) or as a contribution?
- Let's maintain the values and explore the patterns there.

In [ ]:
### linking centrality ###
# set the same range (from exploratory analysis)
cmin = 0.00112
cmax = 2.1692
crange = [cmin, cmax]

In [ ]:
### linking centrality ###
figsize_c = [500, 550]

# rows (edges1) and columns (edges2) that are not all zeros
row_sum = np.sum(abs(Lam_gli), axis=1)
col_sum = np.sum(abs(Lam_gli), axis=0)

print("node size:", ns, "line width:", lw)

cmap = 'plasma'
pos1 = nx.get_node_attributes(G1, 'pos')
pos2 = nx.get_node_attributes(G2, 'pos')

plot_networkx_colore(G1, G2, pos1, pos2, row_sum, col_sum, cmap=cmap, xval=0.80,
                     ns=ns, lw=lw, camera=camera, savefig=True, figname=lab_G+'-unic-G-cent',
                     figsize=figsize_c, val_range=crange)

In [ ]:
# bipartite graph clustering
### Construct the bipartite signed graph
fac = 100 # scale up the values, for numerical reasons
Lam_gfac = Lam_gli * fac
# labels_p1 = ['({},{})'.format(e[0], e[1]) for e in edges1]
# labels_p2 = ['({},{})'.format(e[0], e[1]) for e in edges2]

# Gb = build_sibigraph(Lam_gtol, labels_p1, labels_p2)

# use index instead
labels_p1 = [i for i in range(len(edges1))]
labels_p2 = [i+len(edges1) for i in range(len(edges2))]

Gb = build_sibigraph(Lam_gfac, labels_p1, labels_p2)
print("#nodes:", Gb.number_of_nodes(), "#edges:", Gb.number_of_edges())

# prepare connected components if any
Gb_ccs = [Gb.subgraph(c) for c in nx.connected_components(Gb)]
print("#connected components:", len(Gb_ccs))
print("sizes:", [(g.number_of_nodes(), g.number_of_edges()) for g in Gb_ccs])

In [ ]:
### Following tasks inside each cc ###
idx_g = 0

Gb_cc = Gb_ccs[idx_g]

# get the adjacency
nodes_cc = sorted(Gb_cc.nodes())
W_cc = nx.to_numpy_array(Gb_cc, nodelist=nodes_cc)

# get edges lists
edges1_cc = [edges1[i] for i in nodes_cc if i < len(edges1)]
edges2_cc = [edges2[i-len(edges1)] for i in nodes_cc if i >= len(edges1)]
print("two parts:", (len(edges1_cc), len(edges2_cc)))


# check the balance
deg = np.sum(abs(W_cc), axis=1)

# symmetric version
D2_inv = np.diag(1./np.sqrt(deg))
P_sym = D2_inv.dot(W_cc).dot(D2_inv)
eigvals = np.linalg.eigvals(P_sym)
print("Max eigenval of P:", np.max(eigvals))
# print("eigenvalues of P:", eigvals)

In [ ]:
# Laplacian
D = np.diag(deg)
L = D - W_cc
eigvals, eigvecs = np.linalg.eigh(L)
print("Min eigenvalues of L:", min(eigvals)/fac)

# normalized Laplacian
L_sym = np.eye(W_cc.shape[0]) - P_sym
eigvals, eigvecs = np.linalg.eigh(L_sym)
print("Min eigenvalues of L sym:", min(eigvals))

# indices of the smallest value
tol = 1e-2
eval_min = min(eigvals)
idx = [i for i, val in enumerate(eigvals) if abs(val-eval_min) < tol]
print("Min eigenvalue(s) of L sym:", eigvals[idx])
print("first few eigenvalues:", eigvals[:5])

plt.figure(figsize=(18, 5))
plt.subplot(121)
# choose the eigenvector(s)
for i in idx:
    plt.plot(eigvecs[:, i], marker='.', linestyle='dashed')
plt.grid(True)

plt.subplot(122)
plt.plot(eigvals, marker='.', linestyle='dashed')
plt.grid(True)
plt.show()

Bi-clustering 1: choose the smallest eigenvalue:

In [ ]:
### Bi-Clustering ###
tol = 1e-4
nodec = []
idx_0s = []

for i,val in enumerate(eigvecs[:, idx[0]]):
    if val > tol:
        nodec.append(2)
    elif val < -tol:
        nodec.append(0)
    else:
        idx_0s.append(i)
        nodec.append(1)

if len(idx_0s) == 0:
    print("clear cut!")
    nodec = [1 if nodec[i] > 0 else 0 for i in range(len(nodec))]
else:
    print("# zeros in the eigenvector:", len(idx_0s))

In [ ]:
# allocation zeros
from itertools import combinations

s = np.array([1. if nodec[i] > 0 else -1. for i in range(len(nodec))]) # default setting where all in the 1-group
best_cost = s.T.dot(L).dot(s)
best_partition = [s]
size = len(idx_0s)
idx_0s = np.array(idx_0s)

res = []
for num in range(1, int((size+1.)/2.)+1):
    for idx_c in combinations(list(range(size)), num):
        s_now = s.copy()
        s_now[idx_0s[list(idx_c)]] = -1 # allocate these edges to the -1-group
        cost = s_now.T.dot(L).dot(s_now)
        if cost < best_cost-tol:
            print("change to -1:", idx_c)
            best_cost = cost
            best_partition = [s_now]
        elif abs(cost-best_cost) < tol:
            best_partition.append(s_now)

print("Best cost:", best_cost/fac)
# print("Best partition:", best_partition)

In [ ]:
### Post processing ###
# apply the optimal orientation
s = best_partition[0]
S = np.diag(s)
W_s = S.dot(W_cc.dot(S))
B_s = W_s[:len(edges1_cc), :][:, -len(edges2_cc):]

# Distribution of the linking values
plt.hist((B_s/fac).flatten(), bins=100)
plt.yscale('log')
plt.show()

In [ ]:
# select strong negative signals
negr, negc = np.where(B_s < -0.1*fac)
print("#very negative edges:", len(negr))

# record
res = [(edges1_cc[negr[i]], edges2_cc[negc[i]]) for i in range(len(negr))]
res_idx = [((n2i_dict[e1[0]], n2i_dict[e1[1]]), (n2i_dict[e2[0]], n2i_dict[e2[1]])) for e1,e2 in res]

# medium
negr, negc = np.where(B_s < -0.05*fac)
print("#medium negative edges:", len(negr))

# record
res2 = [(edges1_cc[negr[i]], edges2_cc[negc[i]]) for i in range(len(negr))]
res2_idx = [((n2i_dict[e1[0]], n2i_dict[e1[1]]), (n2i_dict[e2[0]], n2i_dict[e2[1]])) for e1,e2 in res]

# visualization
edges1_str = [epair[0] for epair in res]
edges2_str = [epair[1] for epair in res]

edges1_med = [epair[0] for epair in res2]
edges2_med = [epair[1] for epair in res2]

In [ ]:
## select the colors
alphas = [0.3, 0.5, 1.]
color1 = "rgba(0, 0, 139, {})"
color2 = "rgba(139, 0, 0, {})"

colorset = [[color1.format(a), color2.format(a)] for a in alphas]
# change the middle color
alpha = 0.9
colorset[1] = ["rgba(33, 89, 209,{})".format(alpha), "rgba(196, 29, 29,{})".format(alpha)]

In [ ]:
lab_fac = '-f{}'.format(fac)
# ns = 3.5
# lw = 5
print("node size:", ns, "line width:", lw)

plot_networkx_2colore_sel(G_sel2, G1, G2, edges1_str, edges2_str, edges1_med, edges2_med,
                          ns=ns, lw=lw, camera=camera, savefig=True, colorset=colorset,
                          figsize=figsize_g, figname=lab_G+lab_fac+'-unic-G-signed-l2m-1')

Bi-clustering 2: combine smallest eigenvectors

We observe that the last two eigenvalues are very close to each other.

In [ ]:
# Combine the first two eigenvectors to see whether we have better results?
tol = 1e-4

from itertools import combinations

eigvals, eigvecs = np.linalg.eigh(L_sym)
print("Min eigenvalues of L_sym:", min(eigvals)) # !!! L_sym

# indices of the smallest value
tol_ev = 0.02
eval_min = min(eigvals)
idx = [i for i, val in enumerate(eigvals) if abs(val-eval_min) < tol_ev]
print("Min eigenvalue(s) of L_sym:", eigvals[idx])

u1 = eigvecs[:, idx[0]]
u2 = eigvecs[:, idx[1]]

flip_angles, nodes_deg = critical_flip_angles(u1, u2)
print("#Flip angles:", len(flip_angles))
theta_candidates = candidate_mid_angles(flip_angles)

# Get sign patterns for all candidates
S = sign_patterns_from_angles(u1, u2, theta_candidates)  # shape (n, m)

# For each candidate, evaluate your signed cut cost C(theta_j)
best_cost = np.inf
best_theta = []
best_partition = []

for j, theta in enumerate(theta_candidates):
    s = S[:, j]         # +-1 labels
    # 0 entries?
    idx_0 = np.where(s == 0)[0]
    if len(idx_0) > 0:
        print("0 occurs in eigenvalues")
        ## try all combination of assignments
        # all in the negative one
        s[idx_0] = -1
        cost = s.T.dot(L).dot(s)
        if cost < best_cost-tol:
            best_cost = cost
            best_theta = [theta]
            best_partition = [s]
        elif abs(cost-best_cost) < tol:
            best_theta.append(theta)
            best_partition.append(s)

        # all other combinations
        for num in range(1, len(idx_0)):
            for idx_01 in combinations(idx_0, num):
                s_now = s.copy()
                s_now[idx_0] = -1
                s_now[list(idx_01)] = 1
                cost = s_now.T.dot(L).dot(s_now)
                if cost < best_cost-tol:
                    best_cost = cost
                    best_theta = [theta]
                    best_partition = [s_now]
                elif abs(cost-best_cost) < tol:
                    best_theta.append(theta)
                    best_partition.append(s_now)
        # the last one
        s[idx_0] = 1
        cost = s.T.dot(L).dot(s)
        if cost < best_cost-tol:
            best_cost = cost
            best_theta = [theta]
            best_partition = [s]
        elif abs(cost-best_cost) < tol:
            best_theta.append(theta)
            best_partition.append(s)
    else:
        # compute the objective
        cost = s.T.dot(L).dot(s)
        if cost < best_cost-tol:
            best_cost = cost
            best_theta = [theta]
            best_partition = [s]
        elif abs(cost-best_cost) < tol:
            best_theta.append(theta)
            best_partition.append(s)

print("Best theta:", best_theta)
print("Best cost:", best_cost/fac)
print("# Best partitions:", len(best_partition))

In [ ]:
### Post processing ###
# apply the optimal orientation
s = best_partition[0]
S = np.diag(s)
W_s = S.dot(W_cc.dot(S))
B_s = W_s[:len(edges1_cc), :][:, -len(edges2_cc):]

# Distribution of the linking values
plt.hist((B_s/fac).flatten(), bins=100)
plt.yscale('log')
plt.show()

In [ ]:
# select strong negative signals
negr, negc = np.where(B_s < -0.1*fac)
print("#very negative edges:", len(negr))

# record
res = [(edges1_cc[negr[i]], edges2_cc[negc[i]]) for i in range(len(negr))]
res_idx = [((n2i_dict[e1[0]], n2i_dict[e1[1]]), (n2i_dict[e2[0]], n2i_dict[e2[1]])) for e1,e2 in res]

# medium
negr, negc = np.where(B_s < -0.05*fac)
print("#medium negative edges:", len(negr))

# record
res2 = [(edges1_cc[negr[i]], edges2_cc[negc[i]]) for i in range(len(negr))]
res2_idx = [((n2i_dict[e1[0]], n2i_dict[e1[1]]), (n2i_dict[e2[0]], n2i_dict[e2[1]])) for e1,e2 in res]

# visualization
edges1_str = [epair[0] for epair in res]
edges2_str = [epair[1] for epair in res]

edges1_med = [epair[0] for epair in res2]
edges2_med = [epair[1] for epair in res2]

In [ ]:
alphas = [0.3, 0.5, 1.]
color1 = "rgba(0, 0, 139, {})"
color2 = "rgba(139, 0, 0, {})"

colorset = [[color1.format(a), color2.format(a)] for a in alphas]
# change the middle color
alpha = 0.9
colorset[1] = ["rgba(33, 89, 209,{})".format(alpha), "rgba(196, 29, 29,{})".format(alpha)]

In [ ]:
lab_fac = '-f{}'.format(fac)
# ns = 3.5
# lw = 5
print("node size:", ns, "line width:", lw)

plot_networkx_2colore_sel(G_sel2, G1, G2, edges1_str, edges2_str, edges1_med, edges2_med,
                          ns=ns, lw=lw, camera=camera, savefig=True, colorset=colorset,
                          figsize=figsize_g, figname=lab_G+lab_fac+'-unic-G-signed-l2m-2')

###### cut = 5

In [ ]:
# Case 1: when # above and below are about the same
ns = 2.
lw = 3.5
cut = 5

## Visualization
# --- 1. Select edges for G1 (minRadiusAvg < cut) ---
edges_G1 = [(u, v, data) for u, v, data in G_sel2.edges(data=True) if data["minRadiusAvg"] < cut]

# --- 2. Select remaining edges for G2 (minRadiusAvg >= cut OR missing) ---
edges_G2 = [(u, v, data) for u, v, data in G_sel2.edges(data=True) if data["minRadiusAvg"] >= cut]

# print("#edges:", len(edges_G1), len(edges_G2))

# --- 3. Build subgraphs while preserving attributes ---
G1 = nx.Graph()
G1.add_nodes_from(G_sel2.nodes(data=True))
G1.add_edges_from(edges_G1)

G2 = nx.Graph()
G2.add_nodes_from(G_sel2.nodes(data=True))
G2.add_edges_from(edges_G2)

# --- 4. Remove isolated nodes ---
# G1.remove_nodes_from(list(nx.isolates(G1)))
# G2.remove_nodes_from(list(nx.isolates(G2)))

pos1 = nx.get_node_attributes(G1, 'pos')
pos2 = nx.get_node_attributes(G2, 'pos')
lab_G = 'vessel-z2-c{}'.format(cut)

In [ ]:
# edge colors
alpha = 0.9
row_c = "rgba(33, 89, 209,{})".format(alpha)
col_c = "rgba(196, 29, 29,{})".format(alpha)
# tick title sizes
ax_fz = 30
# figsize

plot_networkx_12nolabel(G1, G2, pos1, pos2, size=ns, width=lw, row_c=row_c, col_c=col_c,
                        camera=camera, savefig=True, figname=lab_G+'-unic-G', figsize=figsize_g, ax_fz=ax_fz) #row_c='#2159d1', col_c='#c41d1d'

In [ ]:
## GLI analysis
# compute the GLI operator
# GLI between edges directly
edges1 = list(G1.edges())
edges2 = list(G2.edges())

Lam_gli = np.zeros((len(edges1), len(edges2)))
for i in range(Lam_gli.shape[0]):
    e1 = edges1[i]
    e1_pos = [np.array(G1.nodes[e1[0]]['pos']), np.array(G1.nodes[e1[1]]['pos'])]
    for j in range(Lam_gli.shape[1]):
        e2 = edges2[j]
        # check whether they share a vertex
        if e1[0] in e2 or e1[1] in e2:
            Lam_gli[i, j] = 0
        else:
            # record the position
            e2_pos = [np.array(G2.nodes[e2[0]]['pos']), np.array(G2.nodes[e2[1]]['pos'])]
            # direct computation of Gauss linking integral
            gli_ij = Gauss_linking_integral(e1_pos, e2_pos)
            Lam_gli[i, j] = gli_ij

In [ ]:
# Visualization
cmap_max = 0.1
val_max = np.abs(Lam_gli).max()
val_min = np.abs(Lam_gli[Lam_gli != 0]).min()
num_0s = np.sum(Lam_gli == 0)
print("edges in 1 and 2:", Lam_gli.shape)
print("#0 values in Lam_gli:", num_0s)
print("max abs value in Lam_gli:", val_max, "min abs value:", val_min)

In [ ]:
cmap_max = min([cmap_max, val_max])
print("max in cmap:", cmap_max)

plt.figure(figsize=(6, 3.5))
# plt.imshow(Lam_gli, cmap='seismic', vmax=val_max, vmin=-val_max) #, aspect='auto'
plt.imshow(Lam_gli, cmap='seismic', vmax=cmap_max, vmin=-cmap_max) #, aspect='auto'

# Optional: show grid or colorbar
plt.grid(False)
plt.colorbar(shrink=0.6)
plt.savefig("{}-Lam_gli-max{}.png".format(lab_G, cmap_max), dpi=300, bbox_inches='tight')
plt.show()

We see that there are very small values in the GLI operator, should we treat them as zero (i.e., no contribution) or as a contribution?
- Let's maintain the values and explore the patterns there.

In [ ]:
### linking centrality ###
# rows (edges1) and columns (edges2) that are not all zeros
row_sum = np.sum(abs(Lam_gli), axis=1)
col_sum = np.sum(abs(Lam_gli), axis=0)

print("node size:", ns, "line width:", lw)

cmap = 'plasma'
pos1 = nx.get_node_attributes(G1, 'pos')
pos2 = nx.get_node_attributes(G2, 'pos')

plot_networkx_colore(G1, G2, pos1, pos2, row_sum, col_sum, cmap=cmap, xval=0.80,
                     ns=ns, lw=lw, camera=camera, savefig=True, figname=lab_G+'-unic-G-cent',
                     figsize=figsize_c, val_range=crange)

In [ ]:
# bipartite graph clustering
### Construct the bipartite signed graph
fac = 100 # scale up the values, for numerical reasons
Lam_gfac = Lam_gli * fac
# labels_p1 = ['({},{})'.format(e[0], e[1]) for e in edges1]
# labels_p2 = ['({},{})'.format(e[0], e[1]) for e in edges2]

# Gb = build_sibigraph(Lam_gtol, labels_p1, labels_p2)

# use index instead
labels_p1 = [i for i in range(len(edges1))]
labels_p2 = [i+len(edges1) for i in range(len(edges2))]

Gb = build_sibigraph(Lam_gfac, labels_p1, labels_p2)
print("#nodes:", Gb.number_of_nodes(), "#edges:", Gb.number_of_edges())

# prepare connected components if any
Gb_ccs = [Gb.subgraph(c) for c in nx.connected_components(Gb)]
print("#connected components:", len(Gb_ccs))
print("sizes:", [(g.number_of_nodes(), g.number_of_edges()) for g in Gb_ccs])

In [ ]:
### Following tasks inside each cc ###
idx_g = 0

Gb_cc = Gb_ccs[idx_g]

# get the adjacency
nodes_cc = sorted(Gb_cc.nodes())
W_cc = nx.to_numpy_array(Gb_cc, nodelist=nodes_cc)

# get edges lists
edges1_cc = [edges1[i] for i in nodes_cc if i < len(edges1)]
edges2_cc = [edges2[i-len(edges1)] for i in nodes_cc if i >= len(edges1)]
print("two parts:", (len(edges1_cc), len(edges2_cc)))


# check the balance
deg = np.sum(abs(W_cc), axis=1)

# symmetric version
D2_inv = np.diag(1./np.sqrt(deg))
P_sym = D2_inv.dot(W_cc).dot(D2_inv)
eigvals = np.linalg.eigvals(P_sym)
print("Max eigenval of P:", np.max(eigvals))
# print("eigenvalues of P:", eigvals)

In [ ]:
# Laplacian
D = np.diag(deg)
L = D - W_cc
eigvals, eigvecs = np.linalg.eigh(L)
print("Min eigenvalues of L:", min(eigvals)/fac)

# normalized Laplacian
L_sym = np.eye(W_cc.shape[0]) - P_sym
eigvals, eigvecs = np.linalg.eigh(L_sym)
print("Min eigenvalues of L sym:", min(eigvals))

# indices of the smallest value
tol = 0.01
eval_min = min(eigvals)
idx = [i for i, val in enumerate(eigvals) if abs(val-eval_min) < tol]
print("Min eigenvalue(s) of L sym:", eigvals[idx])
print("first few eigenvalues:", eigvals[:5])

plt.figure(figsize=(18, 5))
plt.subplot(121)
# choose the eigenvector(s)
for i in idx:
    plt.plot(eigvecs[:, i], marker='.', linestyle='dashed')
plt.grid(True)

plt.subplot(122)
plt.plot(eigvals, marker='.', linestyle='dashed')
plt.grid(True)
plt.show()

Bi-clustering 1: choose the smallest eigenvalue:

In [ ]:
### Bi-Clustering ###
tol = 1e-4
nodec = []
idx_0s = []

for i,val in enumerate(eigvecs[:, idx[0]]):
    if val > tol:
        nodec.append(2)
    elif val < -tol:
        nodec.append(0)
    else:
        idx_0s.append(i)
        nodec.append(1)

if len(idx_0s) == 0:
    print("clear cut!")
    nodec = [1 if nodec[i] > 0 else 0 for i in range(len(nodec))]
else:
    print("# zeros in the eigenvector:", len(idx_0s))

In [ ]:
# allocation zeros
from itertools import combinations

s = np.array([1. if nodec[i] > 0 else -1. for i in range(len(nodec))]) # default setting where all in the 1-group
best_cost = s.T.dot(L).dot(s)
best_partition = [s]
size = len(idx_0s)
idx_0s = np.array(idx_0s)

res = []
for num in range(1, int((size+1.)/2.)+1):
    for idx_c in combinations(list(range(size)), num):
        s_now = s.copy()
        s_now[idx_0s[list(idx_c)]] = -1 # allocate these edges to the -1-group
        cost = s_now.T.dot(L).dot(s_now)
        if cost < best_cost-tol:
            print("change to -1:", idx_c)
            best_cost = cost
            best_partition = [s_now]
        elif abs(cost-best_cost) < tol:
            best_partition.append(s_now)

print("Best cost:", best_cost/fac)
# print("Best partition:", best_partition)

In [ ]:
### Post processing ###
# apply the optimal orientation
s = best_partition[0]
S = np.diag(s)
W_s = S.dot(W_cc.dot(S))
B_s = W_s[:len(edges1_cc), :][:, -len(edges2_cc):]

# Distribution of the linking values
plt.hist((B_s/fac).flatten(), bins=100)
plt.yscale('log')
plt.show()

In [ ]:
# select strong negative signals
negr, negc = np.where(B_s < -0.1*fac)
print("#very negative edges:", len(negr))

# record
res = [(edges1_cc[negr[i]], edges2_cc[negc[i]]) for i in range(len(negr))]
res_idx = [((n2i_dict[e1[0]], n2i_dict[e1[1]]), (n2i_dict[e2[0]], n2i_dict[e2[1]])) for e1,e2 in res]

# medium
negr, negc = np.where(B_s < -0.05*fac)
print("#medium negative edges:", len(negr))

# record
res2 = [(edges1_cc[negr[i]], edges2_cc[negc[i]]) for i in range(len(negr))]
res2_idx = [((n2i_dict[e1[0]], n2i_dict[e1[1]]), (n2i_dict[e2[0]], n2i_dict[e2[1]])) for e1,e2 in res]

# visualization
edges1_str = [epair[0] for epair in res]
edges2_str = [epair[1] for epair in res]

edges1_med = [epair[0] for epair in res2]
edges2_med = [epair[1] for epair in res2]

In [ ]:
lab_fac = '-f{}'.format(fac)
# ns = 3.5
# lw = 5
print("node size:", ns, "line width:", lw)

plot_networkx_2colore_sel(G_sel2, G1, G2, edges1_str, edges2_str, edges1_med, edges2_med,
                          ns=ns, lw=lw, camera=camera, savefig=True, colorset=colorset,
                          figsize=figsize_g, figname=lab_G+lab_fac+'-unic-G-signed-l2m-1')

Bi-clustering 2: combine smallest eigenvectors

Observation: the first two eigenvalues are close to each other, compared with others

In [ ]:
# Combine the first two eigenvectors to see whether we have better results?
tol = 1e-4

from itertools import combinations

eigvals, eigvecs = np.linalg.eigh(L_sym)
print("Min eigenvalues of L_sym:", min(eigvals)) # !!! L_sym

# indices of the smallest value
tol_ev = 0.031
eval_min = min(eigvals)
idx = [i for i, val in enumerate(eigvals) if abs(val-eval_min) < tol_ev]
print("Min eigenvalue(s) of L_sym:", eigvals[idx])

u1 = eigvecs[:, idx[0]]
u2 = eigvecs[:, idx[1]]

flip_angles, nodes_deg = critical_flip_angles(u1, u2)
print("#Flip angles:", len(flip_angles))
theta_candidates = candidate_mid_angles(flip_angles)

# Get sign patterns for all candidates
S = sign_patterns_from_angles(u1, u2, theta_candidates)  # shape (n, m)

# For each candidate, evaluate your signed cut cost C(theta_j)
best_cost = np.inf
best_theta = []
best_partition = []

for j, theta in enumerate(theta_candidates):
    s = S[:, j]         # +-1 labels
    # 0 entries?
    idx_0 = np.where(s == 0)[0]
    if len(idx_0) > 0:
        print("0 occurs in eigenvalues")
        ## try all combination of assignments
        # all in the negative one
        s[idx_0] = -1
        cost = s.T.dot(L).dot(s)
        if cost < best_cost-tol:
            best_cost = cost
            best_theta = [theta]
            best_partition = [s]
        elif abs(cost-best_cost) < tol:
            best_theta.append(theta)
            best_partition.append(s)

        # all other combinations
        for num in range(1, len(idx_0)):
            for idx_01 in combinations(idx_0, num):
                s_now = s.copy()
                s_now[idx_0] = -1
                s_now[list(idx_01)] = 1
                cost = s_now.T.dot(L).dot(s_now)
                if cost < best_cost-tol:
                    best_cost = cost
                    best_theta = [theta]
                    best_partition = [s_now]
                elif abs(cost-best_cost) < tol:
                    best_theta.append(theta)
                    best_partition.append(s_now)
        # the last one
        s[idx_0] = 1
        cost = s.T.dot(L).dot(s)
        if cost < best_cost-tol:
            best_cost = cost
            best_theta = [theta]
            best_partition = [s]
        elif abs(cost-best_cost) < tol:
            best_theta.append(theta)
            best_partition.append(s)
    else:
        # compute the objective
        cost = s.T.dot(L).dot(s)
        if cost < best_cost-tol:
            best_cost = cost
            best_theta = [theta]
            best_partition = [s]
        elif abs(cost-best_cost) < tol:
            best_theta.append(theta)
            best_partition.append(s)

print("Best theta:", best_theta)
print("Best cost:", best_cost/fac)
print("# Best partitions:", len(best_partition))

In [ ]:
### Post processing ###
# apply the optimal orientation
s = best_partition[0]
S = np.diag(s)
W_s = S.dot(W_cc.dot(S))
B_s = W_s[:len(edges1_cc), :][:, -len(edges2_cc):]

# Distribution of the linking values
plt.hist((B_s/fac).flatten(), bins=100)
plt.yscale('log')
plt.show()

In [ ]:
# select strong negative signals
negr, negc = np.where(B_s < -0.1*fac)
print("#very negative edges:", len(negr))

# record
res = [(edges1_cc[negr[i]], edges2_cc[negc[i]]) for i in range(len(negr))]
res_idx = [((n2i_dict[e1[0]], n2i_dict[e1[1]]), (n2i_dict[e2[0]], n2i_dict[e2[1]])) for e1,e2 in res]

# medium
negr, negc = np.where(B_s < -0.05*fac)
print("#medium negative edges:", len(negr))

# record
res2 = [(edges1_cc[negr[i]], edges2_cc[negc[i]]) for i in range(len(negr))]
res2_idx = [((n2i_dict[e1[0]], n2i_dict[e1[1]]), (n2i_dict[e2[0]], n2i_dict[e2[1]])) for e1,e2 in res]

# visualization
edges1_str = [epair[0] for epair in res]
edges2_str = [epair[1] for epair in res]

edges1_med = [epair[0] for epair in res2]
edges2_med = [epair[1] for epair in res2]

In [ ]:
alphas = [0.3, 0.5, 1.]
color1 = "rgba(0, 0, 139, {})"
color2 = "rgba(139, 0, 0, {})"

colorset = [[color1.format(a), color2.format(a)] for a in alphas]
# change the middle color
alpha = 0.9
colorset[1] = ["rgba(33, 89, 209,{})".format(alpha), "rgba(196, 29, 29,{})".format(alpha)]

In [ ]:
lab_fac = '-f{}'.format(fac)
# ns = 3.5
# lw = 5
print("node size:", ns, "line width:", lw)

plot_networkx_2colore_sel(G_sel2, G1, G2, edges1_str, edges2_str, edges1_med, edges2_med,
                          ns=ns, lw=lw, camera=camera, savefig=True, colorset=colorset,
                          figsize=figsize_g, figname=lab_G+lab_fac+'-unic-G-signed-l2m-2')

###### cut = 1.5

In [ ]:
# Case 1: when # above and below are about the same
ns = 2.
lw = 3.5
cut = 1.5

## Visualization
# --- 1. Select edges for G1 (minRadiusAvg < cut) ---
edges_G1 = [(u, v, data) for u, v, data in G_sel2.edges(data=True) if data["minRadiusAvg"] < cut]

# --- 2. Select remaining edges for G2 (minRadiusAvg >= cut OR missing) ---
edges_G2 = [(u, v, data) for u, v, data in G_sel2.edges(data=True) if data["minRadiusAvg"] >= cut]

# print("#edges:", len(edges_G1), len(edges_G2))

# --- 3. Build subgraphs while preserving attributes ---
G1 = nx.Graph()
G1.add_nodes_from(G_sel2.nodes(data=True))
G1.add_edges_from(edges_G1)

G2 = nx.Graph()
G2.add_nodes_from(G_sel2.nodes(data=True))
G2.add_edges_from(edges_G2)

# --- 4. Remove isolated nodes ---
# G1.remove_nodes_from(list(nx.isolates(G1)))
# G2.remove_nodes_from(list(nx.isolates(G2)))

pos1 = nx.get_node_attributes(G1, 'pos')
pos2 = nx.get_node_attributes(G2, 'pos')
lab_G = 'vessel-z2-c{}'.format(cut)

In [ ]:
# edge colors
alpha = 0.9
row_c = "rgba(33, 89, 209,{})".format(alpha)
col_c = "rgba(196, 29, 29,{})".format(alpha)
# tick title sizes
ax_fz = 30
# figsize

plot_networkx_12nolabel(G1, G2, pos1, pos2, size=ns, width=lw, row_c=row_c, col_c=col_c,
                        camera=camera, savefig=True, figname=lab_G+'-unic-G', figsize=figsize_g, ax_fz=ax_fz) #row_c='#2159d1', col_c='#c41d1d'

In [ ]:
## GLI analysis
# compute the GLI operator
# GLI between edges directly
edges1 = list(G1.edges())
edges2 = list(G2.edges())

Lam_gli = np.zeros((len(edges1), len(edges2)))
for i in range(Lam_gli.shape[0]):
    e1 = edges1[i]
    e1_pos = [np.array(G1.nodes[e1[0]]['pos']), np.array(G1.nodes[e1[1]]['pos'])]
    for j in range(Lam_gli.shape[1]):
        e2 = edges2[j]
        # check whether they share a vertex
        if e1[0] in e2 or e1[1] in e2:
            Lam_gli[i, j] = 0
        else:
            # record the position
            e2_pos = [np.array(G2.nodes[e2[0]]['pos']), np.array(G2.nodes[e2[1]]['pos'])]
            # direct computation of Gauss linking integral
            gli_ij = Gauss_linking_integral(e1_pos, e2_pos)
            Lam_gli[i, j] = gli_ij

In [ ]:
# Visualization
cmap_max = 0.1
val_max = np.abs(Lam_gli).max()
val_min = np.abs(Lam_gli[Lam_gli != 0]).min()
num_0s = np.sum(Lam_gli == 0)
print("edges in 1 and 2:", Lam_gli.shape)
print("#0 values in Lam_gli:", num_0s)
print("max abs value in Lam_gli:", val_max, "min abs value:", val_min)

In [ ]:
cmap_max = min([cmap_max, val_max])
print("max in cmap:", cmap_max)

plt.figure(figsize=(6, 3.5))
# plt.imshow(Lam_gli, cmap='seismic', vmax=val_max, vmin=-val_max) #, aspect='auto'
plt.imshow(Lam_gli, cmap='seismic', vmax=cmap_max, vmin=-cmap_max) #, aspect='auto'

# Optional: show grid or colorbar
plt.grid(False)
plt.colorbar(shrink=0.6)
plt.savefig("{}-Lam_gli-max{}.png".format(lab_G, cmap_max), dpi=300, bbox_inches='tight')
plt.show()

We see that there are very small values in the GLI operator, should we treat them as zero (i.e., no contribution) or as a contribution?
- Let's maintain the values and explore the patterns there.

In [ ]:
### linking centrality ###
# rows (edges1) and columns (edges2) that are not all zeros
row_sum = np.sum(abs(Lam_gli), axis=1)
col_sum = np.sum(abs(Lam_gli), axis=0)

print("node size:", ns, "line width:", lw)

cmap = 'plasma'
pos1 = nx.get_node_attributes(G1, 'pos')
pos2 = nx.get_node_attributes(G2, 'pos')

plot_networkx_colore(G1, G2, pos1, pos2, row_sum, col_sum, cmap=cmap, xval=0.80,
                     ns=ns, lw=lw, camera=camera, savefig=True, figname=lab_G+'-unic-G-cent',
                     figsize=figsize_c, val_range=crange)

In [ ]:
# bipartite graph clustering
### Construct the bipartite signed graph
fac = 100 # scale up the values, for numerical reasons
Lam_gfac = Lam_gli * fac
# labels_p1 = ['({},{})'.format(e[0], e[1]) for e in edges1]
# labels_p2 = ['({},{})'.format(e[0], e[1]) for e in edges2]

# Gb = build_sibigraph(Lam_gtol, labels_p1, labels_p2)

# use index instead
labels_p1 = [i for i in range(len(edges1))]
labels_p2 = [i+len(edges1) for i in range(len(edges2))]

Gb = build_sibigraph(Lam_gfac, labels_p1, labels_p2)
print("#nodes:", Gb.number_of_nodes(), "#edges:", Gb.number_of_edges())

# prepare connected components if any
Gb_ccs = [Gb.subgraph(c) for c in nx.connected_components(Gb)]
print("#connected components:", len(Gb_ccs))
print("sizes:", [(g.number_of_nodes(), g.number_of_edges()) for g in Gb_ccs])

In [ ]:
### Following tasks inside each cc ###
idx_g = 0

Gb_cc = Gb_ccs[idx_g]

# get the adjacency
nodes_cc = sorted(Gb_cc.nodes())
W_cc = nx.to_numpy_array(Gb_cc, nodelist=nodes_cc)

# get edges lists
edges1_cc = [edges1[i] for i in nodes_cc if i < len(edges1)]
edges2_cc = [edges2[i-len(edges1)] for i in nodes_cc if i >= len(edges1)]
print("two parts:", (len(edges1_cc), len(edges2_cc)))


# check the balance
deg = np.sum(abs(W_cc), axis=1)

# symmetric version
D2_inv = np.diag(1./np.sqrt(deg))
P_sym = D2_inv.dot(W_cc).dot(D2_inv)
eigvals = np.linalg.eigvals(P_sym)
print("Max eigenval of P:", np.max(eigvals))
# print("eigenvalues of P:", eigvals)

In [ ]:
# Laplacian
D = np.diag(deg)
L = D - W_cc
eigvals, eigvecs = np.linalg.eigh(L)
print("Min eigenvalues of L:", min(eigvals)/fac)

# normalized Laplacian
L_sym = np.eye(W_cc.shape[0]) - P_sym
eigvals, eigvecs = np.linalg.eigh(L_sym)
print("Min eigenvalues of L sym:", min(eigvals))

# indices of the smallest value
tol = 1e-2
eval_min = min(eigvals)
idx = [i for i, val in enumerate(eigvals) if abs(val-eval_min) < tol]
print("Min eigenvalue(s) of L sym:", eigvals[idx])
print("first few eigenvalues:", eigvals[:5])

plt.figure(figsize=(18, 5))
plt.subplot(121)
# choose the eigenvector(s)
for i in idx:
    plt.plot(eigvecs[:, i], marker='.', linestyle='dashed')
plt.grid(True)

plt.subplot(122)
plt.plot(eigvals, marker='.', linestyle='dashed')
plt.grid(True)
plt.show()

In [ ]:
### Bi-Clustering ###
tol = 1e-4
nodec = []
idx_0s = []

for i,val in enumerate(eigvecs[:, idx[0]]):
    if val > tol:
        nodec.append(2)
    elif val < -tol:
        nodec.append(0)
    else:
        idx_0s.append(i)
        nodec.append(1)

if len(idx_0s) == 0:
    print("clear cut!")
    nodec = [1 if nodec[i] > 0 else 0 for i in range(len(nodec))]
else:
    print("# zeros in the eigenvector:", len(idx_0s))

In [ ]:
# allocation zeros
from itertools import combinations

s = np.array([1. if nodec[i] > 0 else -1. for i in range(len(nodec))]) # default setting where all in the 1-group
best_cost = s.T.dot(L).dot(s)
best_partition = [s]
size = len(idx_0s)
idx_0s = np.array(idx_0s)

res = []
for num in range(1, int((size+1.)/2.)+1):
    for idx_c in combinations(list(range(size)), num):
        s_now = s.copy()
        s_now[idx_0s[list(idx_c)]] = -1 # allocate these edges to the -1-group
        cost = s_now.T.dot(L).dot(s_now)
        if cost < best_cost-tol:
            print("change to -1:", idx_c)
            best_cost = cost
            best_partition = [s_now]
        elif abs(cost-best_cost) < tol:
            best_partition.append(s_now)

print("Best cost:", best_cost/fac)
# print("Best partition:", best_partition)

In [ ]:
### Post processing ###
# apply the optimal orientation
s = best_partition[0]
S = np.diag(s)
W_s = S.dot(W_cc.dot(S))
B_s = W_s[:len(edges1_cc), :][:, -len(edges2_cc):]

# Distribution of the linking values
plt.hist((B_s/fac).flatten(), bins=100)
plt.yscale('log')
plt.show()

In [ ]:
# select strong negative signals
negr, negc = np.where(B_s < -0.1*fac)
print("#very negative edges:", len(negr))

# record
res = [(edges1_cc[negr[i]], edges2_cc[negc[i]]) for i in range(len(negr))]
res_idx = [((n2i_dict[e1[0]], n2i_dict[e1[1]]), (n2i_dict[e2[0]], n2i_dict[e2[1]])) for e1,e2 in res]

# medium
negr, negc = np.where(B_s < -0.05*fac)
print("#medium negative edges:", len(negr))

# record
res2 = [(edges1_cc[negr[i]], edges2_cc[negc[i]]) for i in range(len(negr))]
res2_idx = [((n2i_dict[e1[0]], n2i_dict[e1[1]]), (n2i_dict[e2[0]], n2i_dict[e2[1]])) for e1,e2 in res]

# visualization
edges1_str = [epair[0] for epair in res]
edges2_str = [epair[1] for epair in res]

edges1_med = [epair[0] for epair in res2]
edges2_med = [epair[1] for epair in res2]

In [ ]:
lab_fac = '-f{}'.format(fac)
# ns = 3.5
# lw = 5
print("node size:", ns, "line width:", lw)

plot_networkx_2colore_sel(G_sel2, G1, G2, edges1_str, edges2_str, edges1_med, edges2_med,
                          ns=ns, lw=lw, camera=camera, savefig=True, colorset=colorset,
                          figsize=figsize_g, figname=lab_G+lab_fac+'-unic-G-signed-l2m')

#### Zone III: MY

In [ ]:
# --- Step 1: select a region ---
xlim = [1695., 1760.]
ylim = [4000., 4230.]
zlim = [1640., 1740.]

In [ ]:
nodes_sel, regs_sum = select_vertices_range(df_nodes, xlim, ylim, zlim)
print("regions:", regs_sum[0])
print("counts:", regs_sum[1])

This is correct, we get extrac edges from other cc for MY-sat

In [ ]:
# --- Step 2: select edges touching at least one selected node ---
df_edges_sel = df_edges[df_edges["node1id"].isin(nodes_sel) | df_edges["node2id"].isin(nodes_sel)]

# --- Step 3: update node set with all incident nodes ---
nodes_sel2 = set(df_edges_sel["node1id"]).union(set(df_edges_sel["node2id"]))
df_nodes_sel2 = df_nodes[df_nodes["id"].isin(nodes_sel2)]

print(f"Initial selected nodes: {len(nodes_sel)}")
print(f"Selected edges: {len(df_edges_sel)}")
print(f"Updated node set: {len(nodes_sel2)}")

In [ ]:
# create a node label dictionary
n2i_dict = {id:i for i,id in enumerate(df_nodes_sel2['id'].values)}
print("#nodes in dictionary", len(n2i_dict))

In [ ]:
# Region labels and colors
region_categories = sorted(regs_sum[0])
print(region_categories)

# region_color_map = {
#     region_categories[0]: "red",
#     region_categories[1]: "blue",
#     region_categories[2]: "black",
# }
region_color_map = {
    region_categories[0]: '#BCBD22',
    region_categories[1]: '#9467BD',
    region_categories[2]: '#F58518', #'#8C564B'
}

In [ ]:
# Build a fast lookup for node positions - edges color ~ radius
ns = 2.5
lw = 3.5
xval = 0.8
cmap = 'coolwarm'

pos_dict = df_nodes_sel2.set_index("id")[
    ["pos_x", "pos_y", "pos_z"]
].to_dict("index")

# prepare the color map
r_vals = df_edges_sel["minRadiusAvg"].values
val_min, val_max = r_vals.min(), r_vals.max()
print("max, min abs value in edge vals:", val_max, val_min)

# Normalize + convert colormap
colorscale = matplotlib_to_plotly_colorscale(cmap)
# Colorbar
colorbar_trace = dummy_colorbar_trace(val_min, val_max, colorscale, xval=xval)

# assign colors
norm = mcolors.Normalize(vmin=val_min, vmax=val_max)
cmap_vals = plt.colormaps.get_cmap(cmap)

edge_traces = []
for _, edge in df_edges_sel.iterrows():
    n1, n2 = edge["node1id"], edge["node2id"]

    if (n1 not in pos_dict) or (n2 not in pos_dict):
        continue

    x0, y0, z0 = pos_dict[n1]["pos_x"], pos_dict[n1]["pos_y"], pos_dict[n1]["pos_z"]
    x1, y1, z1 = pos_dict[n2]["pos_x"], pos_dict[n2]["pos_y"], pos_dict[n2]["pos_z"]

    r = edge["minRadiusAvg"]
    # normalized_r = (r - r_min) / (r_max - r_min + 1e-12)
    color = mcolors.to_hex(cmap_vals(norm(r)))
    edge_traces.append(go.Scatter3d(
            x=[x0, x1, None],
            y=[y0, y1, None],
            z=[z0, z1, None],
            mode="lines",
            line=dict(color=color, width=lw),
            hoverinfo="text",
            text=f"Radius: {r:.3f}",
            showlegend=False,
        ))

node_traces = []
for region, color in region_color_map.items():
    df_r = df_nodes_sel2[df_nodes_sel2["region_label"] == region]

    node_traces.append(
        go.Scatter3d(
            x=df_r["pos_x"],
            y=df_r["pos_y"],
            z=df_r["pos_z"],
            mode="markers",
            name=f"Region {region}",
            marker=dict(
                size=ns,
                color=color,
                opacity=0.9,
            ),
            hoverinfo="text",
            text=[
                f"Node {nid}<br>Region: {region}"
                for nid in df_r["id"]
            ],
        )
    )

fig = go.Figure(data=[*node_traces, *edge_traces, colorbar_trace])

axisFormat = dict(
                showbackground=False,
                showticklabels=False,
                autorange=True,
                showgrid=True,
        )


fig.update_layout(
    # title="3D Brain Vascular Subnetwork",
    scene=dict(
        xaxis=dict(title="x"),
        yaxis=dict(title="y"),
        zaxis=dict(title="z"),
        aspectmode="data",
    ),
    margin=dict(l=0, r=0, b=0, t=0),
    # legend=dict(itemsizing="constant"),
)

fig.show()

In [ ]:
#set an appropriate camera
camera = dict(
    eye=dict(x=1.2, y=.8, z=.6),
    center=dict(x=0, y=0, z=0),
    up=dict(x=0, y=0, z=.1)
)
fig.update_layout(scene_camera=camera)
fig.show()

In [ ]:
pio.write_image(fig, "vessel-z3_view.png", width=700, height=600, scale=2) #, scale=2

In [ ]:
# output the range
full_fig = fig.full_figure_for_development()
print(full_fig.layout.scene.xaxis.range)
print(full_fig.layout.scene.yaxis.range)
print(full_fig.layout.scene.zaxis.range)

##### Edge counts

In [ ]:
### see how #edges change along the radius cuts
col_rad = "minRadiusAvg"
# cuts = np.arange(1., 16., 0.5)
cuts = sorted(df_edges_sel[col_rad].unique()[1:-1])

summary = []
for cut in cuts:
    # summarize the edges below and above
    num_below = len(df_edges_sel[df_edges_sel[col_rad] < cut])
    num_above = len(df_edges_sel[df_edges_sel[col_rad] >= cut])

    summary.append([cut, num_below, num_above])

In [ ]:
# visualize the summary
tk_fz = 14
lb_fz = 16
plt.figure(figsize=(7,3.5))
summary = np.array(summary)
plt.plot(summary[:, 0], summary[:, 1], '.--', c='b', alpha=0.6)
plt.plot(summary[:, 0], summary[:, 2], '.--', c='r', alpha=0.6)
# plt.yscale('log')
plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('# edges', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.savefig("vessel-z3-edgrad.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# visualize the summary + cuts
cut_sel = [1.5, 2, 5]

plt.figure(figsize=(7,3.5))
# plot vertical lines
for cut in cut_sel:
    plt.axvline(x=cut, linestyle='--', color='k', alpha=0.5)

summary = np.array(summary)
plt.plot(summary[:, 0], summary[:, 1], '.--', c='b', alpha=0.6)
plt.plot(summary[:, 0], summary[:, 2], '.--', c='r', alpha=0.6)
# plt.yscale('log')
plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('# edges', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.savefig("vessels-z3-edgradc.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
## Q: how does the edge distance correlate with radius?
res_dist = []
res_radi= []
G2 = build_region_graphs(df_nodes_sel2, df_edges_sel)

for _, row in df_edges_sel.iterrows():
    n1, n2 = row["node1id"], row["node2id"]
    res_radi.append(float(row["minRadiusAvg"]))

    # compute the distance between the two nodes
    pos1 = np.array(G2.nodes[n1]['pos'])
    pos2 = np.array(G2.nodes[n2]['pos'])
    dist = np.linalg.norm(pos1-pos2)
    res_dist.append(dist)

In [ ]:
# pearson correlation
from scipy import stats

corr = stats.pearsonr(res_dist, res_radi)
print("correlation:", corr)

In [ ]:
plt.figure(figsize=(5,3.8))
plt.scatter(res_dist, res_radi, marker='.', alpha=.5)
plt.xlabel('Distance')
plt.ylabel('Radius')
plt.savefig("vessels-z3-dist-radius.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(5,3.8))
plt.scatter(res_dist, res_radi, marker='.', alpha=.5)
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Distance')
plt.ylabel('Radius')
plt.savefig("vessels-z3-dist-radius-log.png", dpi=300, bbox_inches='tight')
plt.show()

##### Unbalance scores + Centralities

In [ ]:
### See how balance changes with cut
# sort edges
col_rad = "minRadiusAvg"
df_edges_sel = df_edges_sel.sort_values(by=col_rad)

G_sel = build_region_graphs(df_nodes_sel2, df_edges_sel)

## compute the full GLI between each pair of edges in order
Lgli_full = np.zeros((len(df_edges_sel), len(df_edges_sel)))
for i in range(Lgli_full.shape[0]):
    e1 = [df_edges_sel.iloc[i]['node1id'], df_edges_sel.iloc[i]['node2id']]
    e1_pos = [np.array(G_sel.nodes[e1[0]]['pos']), np.array(G_sel.nodes[e1[1]]['pos'])]
    for j in range(i+1, Lgli_full.shape[1]):
        e2 = [df_edges_sel.iloc[j]['node1id'], df_edges_sel.iloc[j]['node2id']]
        # check whether they share a vertex
        if e1[0] in e2 or e1[1] in e2:
            Lgli_full[i, j] = 0
        else:
            # record the position
            e2_pos = [np.array(G_sel.nodes[e2[0]]['pos']), np.array(G_sel.nodes[e2[1]]['pos'])]
            # direct computation of Gauss linking integral
            gli_ij = Gauss_linking_integral(e1_pos, e2_pos)
            Lgli_full[i, j] = gli_ij

In [ ]:
### See how *centrality distribution* changes with cut
res_cut = []

res_bal = []
res_cme = []
res_csd = []
res_cax = []
idx = 0
for _, row in df_edges_sel.iterrows():

    # obtain the linking
    Lam_gli = Lgli_full[:idx+1, :][:, idx+1:]
    print("Lam_gli shape:", Lam_gli.shape)

    if Lam_gli.shape[1] == 0:
        continue

    # record the cut
    res_cut.append(float(row[col_rad]))
    print("cut:", res_cut[-1])

    idx += 1

    # weigh matrix of the bipartite graph
    W_gli = np.block([[np.zeros((Lam_gli.shape[0], Lam_gli.shape[0])), Lam_gli],
                      [Lam_gli.T, np.zeros((Lam_gli.shape[1], Lam_gli.shape[1]))]])

    ## Balance score
    # prepare connected components if any
    Gb = nx.from_numpy_array(W_gli)
    Gb_ccs = [Gb.subgraph(c) for c in nx.connected_components(Gb)]
    print("#connected components:", len(Gb_ccs))
    print("sizes:", [(g.number_of_nodes(), g.number_of_edges()) for g in Gb_ccs])

    # for each cc
    res = []
    for idx_g in range(len(Gb_ccs)):
        Gb_cc = Gb_ccs[idx_g]
        if Gb_cc.number_of_nodes() < 3:
            continue
        # get the adjacency
        W_cc = nx.to_numpy_array(Gb_cc)
        deg = np.sum(abs(W_cc), axis=1)

        # symmetric version
        D2_inv = np.diag(1./np.sqrt(deg))
        P_sym = D2_inv.dot(W_cc).dot(D2_inv)
        eigvals = np.linalg.eigvals(P_sym)
        # print("Max eigenval of P:", np.max(eigvals))
        res.append(np.max(eigvals))
    print("balance score:", res)
    # set one balance score for each cut value
    if len(res) == 0:
        res.append(np.nan)
    else:
        res_bal.append(np.mean(res).real)

    ## Centralities
    res = np.sum(abs(W_gli), axis=0)
    res_cme.append(np.mean(res))
    res_csd.append(np.std(res))
    res_cax.append(np.max(res))

1. Unbalance score

In [ ]:
res_cut = np.array(res_cut)
res_bal = np.array(res_bal)

In [ ]:
# save data
df_res = pd.DataFrame({'cut': res_cut, 'bal': res_bal})
df_res.to_csv("vessels-z3-balrad.csv", index=False)

In [ ]:
plt.figure(figsize=(7,3.5))
# plt.scatter(cut_arr, bal_arr, marker='.', alpha=0.6)
plt.plot(res_cut, 1-res_bal, marker='.', linestyle='dashed', alpha=0.6, c='blueviolet')
plt.xlabel('Radius cuts')
plt.ylabel('Unalance score')
plt.savefig("vessels-z3-unbalrad-l.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(7,3.5))
for cut in cut_sel:
    plt.axvline(x=cut, linestyle='--', color='k', alpha=0.5)

# plt.scatter(cut_arr, bal_arr, marker='.', alpha=0.6)
plt.plot(res_cut, 1-res_bal, marker='.', linestyle='dashed', alpha=0.6, c='blueviolet')
plt.xlabel('Radius cuts')
plt.ylabel('Unbalance score')
plt.savefig("vessels-z3-unbalrad-lc.png", dpi=300, bbox_inches='tight')
plt.show()

2. Centrality

In [ ]:
# set one balance score for each cut value
res_cut = np.array(res_cut)
res_cme = np.array(res_cme)
res_csd = np.array(res_csd)
res_cax = np.array(res_cax)

In [ ]:
# save data
df_res = pd.DataFrame({'cut': res_cut, 'cent-mean': res_cme, 'cent-std': res_csd, 'cent-max': res_cax})
df_res.to_csv("vessels-z3-cenrad.csv", index=False)

In [ ]:
plt.figure(figsize=(7,3.5))
plt.plot(res_cut, res_cme, marker='.', linestyle='dashed', alpha=0.6, c='orangered')
plt.fill_between(res_cut, res_cme-res_csd, res_cme+res_csd, alpha=0.2, color='orangered')
plt.xlabel('Radius cuts')
plt.ylabel('Centrality')
plt.savefig("vessels-z3-cenrad-l.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(7,3.5))
for cut in cut_sel:
    plt.axvline(x=cut, linestyle='--', color='k', alpha=0.5)

plt.plot(res_cut, res_cme, marker='.', linestyle='dashed', alpha=0.6, c='orangered')
plt.fill_between(res_cut, res_cme-res_csd, res_cme+res_csd, alpha=0.2, color='orangered')
plt.xlabel('Radius cuts')
plt.ylabel('Centrality')
plt.savefig("vessels-z3-cenrad-lc.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(7,3.5))
plt.plot(res_cut, res_cax, marker='.', linestyle='dashed', alpha=0.6, c='orangered')
plt.xlabel('Radius cuts')
plt.ylabel('Centrality')
plt.savefig("vessels-z3-cmxrad-l.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(7,3.5))
for cut in cut_sel:
    plt.axvline(x=cut, linestyle='--', color='k', alpha=0.5)
plt.plot(res_cut, res_cax, marker='.', linestyle='dashed', alpha=0.6, c='orangered')
plt.xlabel('Radius cuts')
plt.ylabel('Centrality')
plt.savefig("vessels-z3-cmxrad-lc.png", dpi=300, bbox_inches='tight')
plt.show()

###### Null model 1: SER and SRG

In [ ]:
# random sample and compute the unbalance score for comparison
num_sam = 10

# choose the model
lab_mod = ['er', 'dist'][0]
p_dist = True if lab_mod == 'dist' else False
print("probability by distance?", p_dist)

num_nodes = len(nodes_sel2)
num_edges = len(df_edges_sel)
# Use empirical bounding box for the null positions
xyz_ranges = ((xlim[0], xlim[1]), (ylim[0], ylim[1]), (zlim[0], zlim[1]))

col_rad = "minRadiusAvg"

resall_cut = []
# balance
resall_bal = []
# centrality
resall_cme = []
resall_csd = []
resall_cax = []

for s in range(num_sam):
    print("sample:", s)

    ### generate null models
    nodes_null_df, edges_null_df, G_null = generate_spatial_null_model(
        n_nodes=num_nodes, m_edges=num_edges, df_edges_sel=df_edges_sel,
        xyz_ranges=xyz_ranges, directed=False, p_dist=p_dist)

    ### sweep the unbalance score
    # sort edges
    edges_null_df = edges_null_df.sort_values(by=col_rad)

    # compute the full GLI between each pair of edges in order
    Lgli_full = np.zeros((len(edges_null_df), len(edges_null_df)))
    for i in range(Lgli_full.shape[0]):
        e1 = [edges_null_df.iloc[i]['node1id'], edges_null_df.iloc[i]['node2id']]
        e1_pos = [np.array(G_null.nodes[e1[0]]['pos']), np.array(G_null.nodes[e1[1]]['pos'])]
        for j in range(i+1, Lgli_full.shape[1]):
            e2 = [edges_null_df.iloc[j]['node1id'], edges_null_df.iloc[j]['node2id']]
            # check whether they share a vertex
            if e1[0] in e2 or e1[1] in e2:
                Lgli_full[i, j] = 0
            else:
                # record the position
                e2_pos = [np.array(G_null.nodes[e2[0]]['pos']), np.array(G_null.nodes[e2[1]]['pos'])]
                # direct computation of Gauss linking integral
                gli_ij = Gauss_linking_integral(e1_pos, e2_pos)
                Lgli_full[i, j] = gli_ij
    # compute the balance score
    res_cut = []
    res_bal = []
    res_cme = []
    res_csd = []
    res_cax = []
    idx = 0
    for _, row in edges_null_df.iterrows():

        # obtain the linking
        Lam_gli = Lgli_full[:idx+1, :][:, idx+1:]
        print("Lam_gli shape:", Lam_gli.shape)

        if Lam_gli.shape[1] == 0:
            continue

        # record the cut
        res_cut.append(float(row[col_rad]))
        print("cut:", res_cut[-1])

        idx += 1
        # characterize the balance
        W_gli = np.block([[np.zeros((Lam_gli.shape[0], Lam_gli.shape[0])), Lam_gli],
                          [Lam_gli.T, np.zeros((Lam_gli.shape[1], Lam_gli.shape[1]))]])

        # prepare connected components if any
        Gb = nx.from_numpy_array(W_gli)
        Gb_ccs = [Gb.subgraph(c) for c in nx.connected_components(Gb)]
        print("#connected components:", len(Gb_ccs))
        print("sizes:", [(g.number_of_nodes(), g.number_of_edges()) for g in Gb_ccs])

        # for each cc
        res = []
        for idx_g in range(len(Gb_ccs)):
            Gb_cc = Gb_ccs[idx_g]
            if Gb_cc.number_of_nodes() < 3:
                continue
            # get the adjacency
            W_cc = nx.to_numpy_array(Gb_cc)
            deg = np.sum(abs(W_cc), axis=1)

            # symmetric version
            D2_inv = np.diag(1./np.sqrt(deg))
            P_sym = D2_inv.dot(W_cc).dot(D2_inv)
            eigvals = np.linalg.eigvals(P_sym)
            # print("Max eigenval of P:", np.max(eigvals))
            res.append(np.max(eigvals))
        print("balance score:", res)
        # set one balance score for each cut value
        if len(res) == 0:
            res.append(np.nan)
        else:
            res_bal.append(np.mean(res).real)

        ## Centrality
        # record the distribution of centralities
        res = np.sum(abs(W_gli), axis=0)
        res_cme.append(np.mean(res))
        res_csd.append(np.std(res))
        res_cax.append(np.max(res))

    # store the results
    resall_cut.append(res_cut)
    resall_bal.append(res_bal)
    resall_cme.append(res_cme)
    resall_csd.append(res_csd)
    resall_cax.append(res_cax)

1. Unbalance score

In [ ]:
cut_arr = np.array(resall_cut).mean(axis=0)
cut_std = np.array(resall_cut).std(axis=0)
# check the std of cuts
print("Cut std:", min(cut_std), max(cut_std))

bal_arr = np.array(resall_bal).mean(axis=0)
bal_std = np.array(resall_bal).std(axis=0)

In [ ]:
# plot average +- standard deviation
plt.figure(figsize=(7,3.5))
plt.plot(cut_arr, 1-bal_arr, linestyle='dashed', marker='.', alpha=0.6, c='k')
plt.fill_between(cut_arr, 1-bal_arr-bal_std, 1-bal_arr+bal_std, alpha=0.2, color='grey')
plt.xlabel('Radius cuts')
plt.ylabel('Balance score')
plt.savefig("/content/drive/MyDrive/ensnarl_res/vessels-z3-unbalrad-sw-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# save the data
df_res = pd.DataFrame({'cut': cut_arr, 'bal': bal_arr, 'bal_std': bal_std})
df_res.to_csv("/content/drive/MyDrive/ensnarl_res/vessels-z3-unbalrad-sw-{}-s{}.csv".format(lab_mod, num_sam), index=False)

2. Centrality

In [ ]:
cut_arr = np.array(resall_cut).mean(axis=0)
cut_std = np.array(resall_cut).std(axis=0)
# check the std of cuts
print("Cut std:", min(cut_std), max(cut_std))

cme_arr = np.array(resall_cme).mean(axis=0)
cme_std = np.array(resall_cme).std(axis=0)
csd_arr = np.array(resall_csd).mean(axis=0)
csd_std = np.array(resall_csd).std(axis=0)
cax_arr = np.array(resall_cax).mean(axis=0)
cax_std = np.array(resall_cax).std(axis=0)

In [ ]:
  # plot average +- standard deviation: centrality mean & centrality deviation
plt.figure(figsize=(7,3.5))
plt.plot(cut_arr, cme_arr, linestyle='dashed', marker='.', alpha=0.6, c='k')
plt.fill_between(cut_arr, cme_arr-csd_arr, cme_arr+csd_arr, alpha=0.2, color='grey')
plt.xlabel('Radius cuts')
plt.ylabel('Centrality')
plt.savefig("vessels-z3-cenrad-sw-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot average +- standard deviation: centrality mean
plt.figure(figsize=(7,3.5))
plt.plot(cut_arr, cme_arr, linestyle='dashed', marker='.', alpha=0.6, c='k')
plt.fill_between(cut_arr, cme_arr-cme_std, cme_arr+cme_std, alpha=0.2, color='grey')
plt.xlabel('Radius cuts')
plt.ylabel('Centrality')
plt.savefig("vessels-z3-cmerad-sw-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot average +- standard deviation: centrality max
plt.figure(figsize=(7,3.5))
plt.plot(cut_arr, cax_arr, linestyle='dashed', marker='.', alpha=0.6, c='k')
plt.fill_between(cut_arr, cax_arr-cax_std, cax_arr+cax_std, alpha=0.2, color='grey')
plt.xlabel('Radius cuts')
plt.ylabel('Centrality')
plt.savefig("vessels-z3-caxrad-sw-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# save the data
df_res = pd.DataFrame({'cut': cut_arr, 'cent-mean-mean': cme_arr, 'cent-mean-std': cme_std,
                       'cent-std-mean': csd_arr, 'cent-std-std': csd_std,
                       'cent-max-mean': cax_arr, 'cent-max-std': cax_std})
df_res.to_csv("vessels-z3-cenrad-sw-{}-s{}.csv".format(lab_mod, num_sam), index=False)

###### Null model 2: shuffle radius

In [ ]:
# random sample and compute the unbalance score for comparison
num_sam = 10

# choose the model
lab_mod = ['radi'][0]
print("null model", lab_mod)

num_nodes = len(nodes_sel2)
num_edges = len(df_edges_sel)
# Use empirical bounding box for the null positions
xyz_ranges = ((xlim[0], xlim[1]), (ylim[0], ylim[1]), (zlim[0], zlim[1]))

col_rad = "minRadiusAvg"

resall_cut = []
# balance
resall_bal = []
# centrality
resall_cut = []
resall_cme = []
resall_csd = []
resall_cax = []

for s in range(num_sam):
    print("sample:", s)

    ### generate null models
    edges_null_df, G_null = generate_radii_null_model(
        m_edges=num_edges, df_edges_sel=df_edges_sel, df_nodes_sel=df_nodes_sel2, directed=False)

    ### sweep the unbalance score
    # sort edges
    edges_null_df = edges_null_df.sort_values(by=col_rad)

    # compute the full GLI between each pair of edges in order
    Lgli_full = np.zeros((len(edges_null_df), len(edges_null_df)))
    for i in range(Lgli_full.shape[0]):
        e1 = [edges_null_df.iloc[i]['node1id'], edges_null_df.iloc[i]['node2id']]
        e1_pos = [np.array(G_null.nodes[e1[0]]['pos']), np.array(G_null.nodes[e1[1]]['pos'])]
        for j in range(i+1, Lgli_full.shape[1]):
            e2 = [edges_null_df.iloc[j]['node1id'], edges_null_df.iloc[j]['node2id']]
            # check whether they share a vertex
            if e1[0] in e2 or e1[1] in e2:
                Lgli_full[i, j] = 0
            else:
                # record the position
                e2_pos = [np.array(G_null.nodes[e2[0]]['pos']), np.array(G_null.nodes[e2[1]]['pos'])]
                # direct computation of Gauss linking integral
                gli_ij = Gauss_linking_integral(e1_pos, e2_pos)
                Lgli_full[i, j] = gli_ij

    # compute the balance score
    res_cut = []
    res_bal = []
    res_cme = []
    res_csd = []
    res_cax = []
    idx = 0
    for _, row in edges_null_df.iterrows():

        # obtain the linking
        Lam_gli = Lgli_full[:idx+1, :][:, idx+1:]
        print("Lam_gli shape:", Lam_gli.shape)

        if Lam_gli.shape[1] == 0:
            continue

        # record the cut
        res_cut.append(float(row[col_rad]))
        print("cut:", res_cut[-1])

        idx += 1
        # weight matrix of the bipartite graph
        W_gli = np.block([[np.zeros((Lam_gli.shape[0], Lam_gli.shape[0])), Lam_gli],
                          [Lam_gli.T, np.zeros((Lam_gli.shape[1], Lam_gli.shape[1]))]])

        ## Balance
        # prepare connected components if any
        Gb = nx.from_numpy_array(W_gli)
        Gb_ccs = [Gb.subgraph(c) for c in nx.connected_components(Gb)]
        print("#connected components:", len(Gb_ccs))
        print("sizes:", [(g.number_of_nodes(), g.number_of_edges()) for g in Gb_ccs])

        # for each cc
        res = []
        for idx_g in range(len(Gb_ccs)):
            Gb_cc = Gb_ccs[idx_g]
            if Gb_cc.number_of_nodes() < 3:
                continue
            # get the adjacency
            W_cc = nx.to_numpy_array(Gb_cc)
            deg = np.sum(abs(W_cc), axis=1)

            # symmetric version
            D2_inv = np.diag(1./np.sqrt(deg))
            P_sym = D2_inv.dot(W_cc).dot(D2_inv)
            eigvals = np.linalg.eigvals(P_sym)
            # print("Max eigenval of P:", np.max(eigvals))
            res.append(np.max(eigvals))
        print("balance score:", res)
        # set one balance score for each cut value
        if len(res) == 0:
            res.append(np.nan)
        else:
            res_bal.append(np.mean(res).real)

        ## Centrality
        # record the distribution of centralities
        res = np.sum(abs(W_gli), axis=0)
        res_cme.append(np.mean(res))
        res_csd.append(np.std(res))
        res_cax.append(np.max(res))

    # store the results
    resall_cut.append(res_cut)
    resall_bal.append(res_bal)
    resall_cme.append(res_cme)
    resall_csd.append(res_csd)
    resall_cax.append(res_cax)

1. Unbalance score

In [ ]:
cut_arr = np.array(resall_cut).mean(axis=0)
cut_std = np.array(resall_cut).std(axis=0)
# check the std of cuts
print("Cut std:", min(cut_std), max(cut_std))

bal_arr = np.array(resall_bal).mean(axis=0)
bal_std = np.array(resall_bal).std(axis=0)

In [ ]:
# plot average +- standard deviation
plt.figure(figsize=(7,3.5))
plt.plot(cut_arr, 1-bal_arr, linestyle='dashed', marker='.', alpha=0.6, c='k')
plt.fill_between(cut_arr, 1-bal_arr-bal_std, 1-bal_arr+bal_std, alpha=0.2, color='grey')
plt.xlabel('Radius cuts')
plt.ylabel('Balance score')
plt.savefig("vessels-z3-unbalrad-null-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# save the data
df_res = pd.DataFrame({'cut': cut_arr, 'bal': bal_arr, 'bal_std': bal_std})
df_res.to_csv("vessels-z3-unbalrad-null-{}-s{}.csv".format(lab_mod, num_sam), index=False)

In [ ]:
### comparison
# read in data
df_bal = pd.read_csv("vessels-z3-balrad.csv")
df_radi = pd.read_csv("vessels-z3-unbalrad-null-radi-s10.csv")

In [ ]:
colors = ['r', 'b', 'y', 'c']
# plot together
plt.figure(figsize=(7,3.5))

# null model - radius
plt.plot(df_radi['cut'], 1-df_radi['bal'], linestyle='dashed', marker='.', alpha=0.3, c=colors[3], label='Rad')
plt.fill_between(df_radi['cut'], 1-df_radi['bal']-df_radi['bal_std'], 1-df_radi['bal']+df_radi['bal_std'], alpha=0.2, color=colors[3])

# actual value
plt.plot(df_bal['cut'], 1-df_bal['bal'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.xlabel('Radius cuts')
plt.ylabel('Unbalance score')
plt.legend()
plt.savefig("vessels-z3-unbalrad-null2.png", dpi=300, bbox_inches='tight')
plt.show()

2. Centrality

In [ ]:
cut_arr = np.array(resall_cut).mean(axis=0)
cut_std = np.array(resall_cut).std(axis=0)
# check the std of cuts
print("Cut std:", min(cut_std), max(cut_std))

cme_arr = np.array(resall_cme).mean(axis=0)
cme_std = np.array(resall_cme).std(axis=0)
csd_arr = np.array(resall_csd).mean(axis=0)
csd_std = np.array(resall_csd).std(axis=0)
cax_arr = np.array(resall_cax).mean(axis=0)
cax_std = np.array(resall_cax).std(axis=0)

In [ ]:
# plot average +- standard deviation: centrality mean & centrality deviation
plt.figure(figsize=(7,3.5))
plt.plot(cut_arr, cme_arr, linestyle='dashed', marker='.', alpha=0.6, c='k')
plt.fill_between(cut_arr, cme_arr-csd_arr, cme_arr+csd_arr, alpha=0.2, color='grey')
plt.xlabel('Radius cuts')
plt.ylabel('Centrality')
plt.savefig("vessels-z3-cenrad-null-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot average +- standard deviation: centrality mean
plt.figure(figsize=(7,3.5))
plt.plot(cut_arr, cme_arr, linestyle='dashed', marker='.', alpha=0.6, c='k')
plt.fill_between(cut_arr, cme_arr-cme_std, cme_arr+cme_std, alpha=0.2, color='grey')
plt.xlabel('Radius cuts')
plt.ylabel('Centrality')
plt.savefig("vessels-z3-cmerad-null-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot average +- standard deviation: centrality max
plt.figure(figsize=(7,3.5))
plt.plot(cut_arr, cax_arr, linestyle='dashed', marker='.', alpha=0.6, c='k')
plt.fill_between(cut_arr, cax_arr-cax_std, cax_arr+cax_std, alpha=0.2, color='grey')
plt.xlabel('Radius cuts')
plt.ylabel('Centrality')
plt.savefig("vessels-z3-caxrad-null-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# save the data
df_res = pd.DataFrame({'cut': cut_arr, 'cent-mean-mean': cme_arr, 'cent-mean-std': cme_std,
                       'cent-std-mean': csd_arr, 'cent-std-std': csd_std,
                       'cent-max-mean': cax_arr, 'cent-max-std': cax_std})
df_res.to_csv("vessels-z3-cenrad-null-{}-s{}.csv".format(lab_mod, num_sam), index=False)

In [ ]:
### comparison
# read in data
df_cen = pd.read_csv("vessels-z3-cenrad.csv")
df_radi = pd.read_csv("vessels-z3-cenrad-null-radi-s10.csv")

In [ ]:
colors = ['r', 'b', 'y', 'c']
# plot together
plt.figure(figsize=(7,3.5))

# null model - radius
plt.plot(df_radi['cut'], df_radi['cent-mean-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[3], label='Rad')
plt.fill_between(df_radi['cut'], df_radi['cent-mean-mean']-df_radi['cent-mean-std'],
                 df_radi['cent-mean-mean']+df_radi['cent-mean-std'], alpha=0.2, color=colors[3])

# actual value
plt.plot(df_cen['cut'], df_cen['cent-mean'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.xlabel('Radius cuts')
plt.ylabel('Centrality Mean')
plt.legend()
plt.savefig("vessels-z3-cmerad-null2.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot together
plt.figure(figsize=(7,3.5))

# null model - radius
plt.plot(df_radi['cut'], df_radi['cent-max-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[3], label='Rad')
plt.fill_between(df_radi['cut'], df_radi['cent-max-mean']-df_radi['cent-max-std'],
                 df_radi['cent-max-mean']+df_radi['cent-max-std'], alpha=0.2, color=colors[3])

# actual value
plt.plot(df_cen['cut'], df_cen['cent-max'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.xlabel('Radius cuts')
plt.ylabel('Centrality Max')
plt.legend()
plt.savefig("vessels-z3-caxrad-null2.png", dpi=300, bbox_inches='tight')
plt.show()

###### Summary

In [ ]:
# # uniform scale
# ylim_unb = [-0.05, 0.63]
# ylim_cme = [0.0005, 45]
# ylim_cax = [0.1, 180]

1. Unbalance scores

In [ ]:
## ALL
df_bal = pd.read_csv("vessels-z3-balrad.csv")
df_radi = pd.read_csv("vessels-z3-unbalrad-null-radi-s10.csv")
df_er = pd.read_csv("vessels-z3-unbalrad-sw-er-s10.csv")
df_dist = pd.read_csv("vessels-z3-unbalrad-sw-dist-s10.csv")

In [ ]:
colors = ['r', 'b', 'y', 'c']
# plot together
tk_fz = 14
lb_fz = 16
lg_fz = 12

plt.figure(figsize=(7,3.5))

# null model - radius
plt.plot(df_radi['cut'], 1-df_radi['bal'], linestyle='dashed', marker='.', alpha=0.3, c=colors[3], label='Rad')
plt.fill_between(df_radi['cut'], 1-df_radi['bal']-df_radi['bal_std'], 1-df_radi['bal']+df_radi['bal_std'], alpha=0.2, color=colors[3])

# null model - ER
plt.plot(df_er['cut'], 1-df_er['bal'], linestyle='dashed', marker='.', alpha=0.3, c=colors[2], label='SER')
plt.fill_between(df_er['cut'], 1-df_er['bal']-df_er['bal_std'], 1-df_er['bal']+df_er['bal_std'], alpha=0.2, color=colors[2])

# null model - dist
plt.plot(df_dist['cut'], 1-df_dist['bal'], linestyle='dashed', marker='.', alpha=0.3, c=colors[1], label='SRG')
plt.fill_between(df_dist['cut'], 1-df_dist['bal']-df_dist['bal_std'], 1-df_dist['bal']+df_dist['bal_std'], alpha=0.2, color=colors[1])

# actual value
plt.plot(df_bal['cut'], 1-df_bal['bal'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('unbalance score', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.legend(fontsize=lg_fz, frameon=False)
plt.savefig("vessels-z3-unbalrad-all.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot together
plt.figure(figsize=(7,3.5))

# null model - ER
plt.plot(df_er['cut'], 1-df_er['bal'], linestyle='dashed', marker='.', alpha=0.3, c=colors[2], label='SER')
plt.fill_between(df_er['cut'], 1-df_er['bal']-df_er['bal_std'], 1-df_er['bal']+df_er['bal_std'], alpha=0.2, color=colors[2])

# null model - dist
plt.plot(df_dist['cut'], 1-df_dist['bal'], linestyle='dashed', marker='.', alpha=0.3, c=colors[1], label='SRG')
plt.fill_between(df_dist['cut'], 1-df_dist['bal']-df_dist['bal_std'], 1-df_dist['bal']+df_dist['bal_std'], alpha=0.2, color=colors[1])

# actual value
plt.plot(df_bal['cut'], 1-df_bal['bal'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.ylim(ylim_unb)
plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('unbalance score', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.legend(fontsize=lg_fz, frameon=False)
plt.savefig("vessels-z3-unbalrad-all3.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(7,3.5))

# null model - radius
plt.plot(df_radi['cut'], 1-df_radi['bal'], linestyle='dashed', marker='.', alpha=0.3, c=colors[3], label='Rad')
plt.fill_between(df_radi['cut'], 1-df_radi['bal']-df_radi['bal_std'], 1-df_radi['bal']+df_radi['bal_std'], alpha=0.2, color=colors[3])

# null model - dist
plt.plot(df_dist['cut'], 1-df_dist['bal'], linestyle='dashed', marker='.', alpha=0.3, c=colors[1], label='SRG')
plt.fill_between(df_dist['cut'], 1-df_dist['bal']-df_dist['bal_std'], 1-df_dist['bal']+df_dist['bal_std'], alpha=0.2, color=colors[1])

# actual value
plt.plot(df_bal['cut'], 1-df_bal['bal'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('unbalance score', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.legend(fontsize=lg_fz, frameon=False)
plt.savefig("vessels-z3-unbalrad-null3.png", dpi=300, bbox_inches='tight')
plt.show()

2. Centrality

In [ ]:
# read in data
df_cen = pd.read_csv("vessels-z3-cenrad.csv")
df_radi = pd.read_csv("vessels-z3-cenrad-null-radi-s10.csv")
df_er = pd.read_csv("vessels-z3-cenrad-sw-er-s10.csv")
df_dist = pd.read_csv("vessels-z3-cenrad-sw-dist-s10.csv")

In [ ]:
colors = ['orangered', 'b', 'y', 'c']

In [ ]:
# plot together
plt.figure(figsize=(7,3.5))

# null model - radius
plt.plot(df_radi['cut'], 1-df_radi['cent-mean-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[3], label='Rad')
plt.fill_between(df_radi['cut'], 1-df_radi['cent-mean-mean']-df_radi['cent-mean-std'],
                 1-df_radi['cent-mean-mean']+df_radi['cent-mean-std'], alpha=0.2, color=colors[3])

# null model - ER
plt.plot(df_er['cut'], df_er['cent-mean-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[2], label='SER')
plt.fill_between(df_er['cut'], df_er['cent-mean-mean']-df_er['cent-mean-std'],
                 df_er['cent-mean-mean']+df_er['cent-mean-std'], alpha=0.2, color=colors[2])

# null model - dist
plt.plot(df_dist['cut'], df_dist['cent-mean-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[1], label='SRG')
plt.fill_between(df_dist['cut'], df_dist['cent-mean-mean']-df_dist['cent-mean-std'],
                 df_dist['cent-mean-mean']+df_dist['cent-mean-std'], alpha=0.2, color=colors[1])

# actual value
plt.plot(df_cen['cut'], df_cen['cent-mean'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

# plt.yscale('log')
plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('centrality mean', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.legend(fontsize=lg_fz, frameon=False)
plt.savefig("vessels-z3-cmerad-all.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot together
plt.figure(figsize=(7,3.5))

# null model - radius
plt.plot(df_radi['cut'], df_radi['cent-mean-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[3], label='Rad')
plt.fill_between(df_radi['cut'], df_radi['cent-mean-mean']-df_radi['cent-mean-std'],
                 df_radi['cent-mean-mean']+df_radi['cent-mean-std'], alpha=0.2, color=colors[3])

# null model - ER
plt.plot(df_er['cut'], df_er['cent-mean-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[2], label='SER')
plt.fill_between(df_er['cut'], df_er['cent-mean-mean']-df_er['cent-mean-std'],
                 df_er['cent-mean-mean']+df_er['cent-mean-std'], alpha=0.2, color=colors[2])

# null model - dist
plt.plot(df_dist['cut'], df_dist['cent-mean-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[1], label='SRG')
plt.fill_between(df_dist['cut'], df_dist['cent-mean-mean']-df_dist['cent-mean-std'],
                 df_dist['cent-mean-mean']+df_dist['cent-mean-std'], alpha=0.2, color=colors[1])

# actual value
plt.plot(df_cen['cut'], df_cen['cent-mean'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.yscale('log')
plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('centrality mean', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.legend(fontsize=lg_fz, frameon=False)
plt.savefig("vessels-z3-cmerad-all-log.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot together
plt.figure(figsize=(7,3.5))

# null model - ER
plt.plot(df_er['cut'], df_er['cent-mean-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[2], label='SER')
plt.fill_between(df_er['cut'], df_er['cent-mean-mean']-df_er['cent-mean-std'],
                 df_er['cent-mean-mean']+df_er['cent-mean-std'], alpha=0.2, color=colors[2])

# null model - dist
plt.plot(df_dist['cut'], df_dist['cent-mean-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[1], label='SRG')
plt.fill_between(df_dist['cut'], df_dist['cent-mean-mean']-df_dist['cent-mean-std'],
                 df_dist['cent-mean-mean']+df_dist['cent-mean-std'], alpha=0.2, color=colors[1])

# actual value
plt.plot(df_cen['cut'], df_cen['cent-mean'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.ylim(ylim_cme)
plt.yscale('log')
plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('centrality mean', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.legend(fontsize=lg_fz, frameon=False)
plt.savefig("vessels-z3-cmerad-all3-log.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot together
plt.figure(figsize=(7,3.5))

# null model - radius
plt.plot(df_radi['cut'], df_radi['cent-mean-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[3], label='Rad')
plt.fill_between(df_radi['cut'], df_radi['cent-mean-mean']-df_radi['cent-mean-std'],
                 df_radi['cent-mean-mean']+df_radi['cent-mean-std'], alpha=0.2, color=colors[3])

# null model - dist
plt.plot(df_dist['cut'], df_dist['cent-mean-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[1], label='SRG')
plt.fill_between(df_dist['cut'], df_dist['cent-mean-mean']-df_dist['cent-mean-std'],
                 df_dist['cent-mean-mean']+df_dist['cent-mean-std'], alpha=0.2, color=colors[1])

# actual value
plt.plot(df_cen['cut'], df_cen['cent-mean'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.yscale('log')
plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('centrality mean', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.legend(fontsize=lg_fz, frameon=False)
plt.savefig("vessels-z3-cmerad-null3-log.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot together
plt.figure(figsize=(7,3.5))

# null model - radius
plt.plot(df_radi['cut'], 1-df_radi['cent-max-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[3], label='Rad')
plt.fill_between(df_radi['cut'], 1-df_radi['cent-max-mean']-df_radi['cent-max-std'],
                 1-df_radi['cent-max-mean']+df_radi['cent-max-std'], alpha=0.2, color=colors[3])

# null model - ER
plt.plot(df_er['cut'], df_er['cent-max-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[2], label='SER')
plt.fill_between(df_er['cut'], df_er['cent-max-mean']-df_er['cent-max-std'],
                 df_er['cent-max-mean']+df_er['cent-max-std'], alpha=0.2, color=colors[2])

# null model - dist
plt.plot(df_dist['cut'], df_dist['cent-max-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[1], label='SRG')
plt.fill_between(df_dist['cut'], df_dist['cent-max-mean']-df_dist['cent-max-std'],
                 df_dist['cent-max-mean']+df_dist['cent-max-std'], alpha=0.2, color=colors[1])

# actual value
plt.plot(df_cen['cut'], df_cen['cent-max'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

# plt.yscale('log')
plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('centrality max', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.legend(fontsize=lg_fz, frameon=False)
plt.savefig("vessels-z3-caxrad-all.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot together
plt.figure(figsize=(7,3.5))

# null model - radius
plt.plot(df_radi['cut'], df_radi['cent-max-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[3], label='Rad')
plt.fill_between(df_radi['cut'], df_radi['cent-max-mean']-df_radi['cent-max-std'],
                 df_radi['cent-max-mean']+df_radi['cent-max-std'], alpha=0.2, color=colors[3])

# null model - ER
plt.plot(df_er['cut'], df_er['cent-max-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[2], label='SER')
plt.fill_between(df_er['cut'], df_er['cent-max-mean']-df_er['cent-max-std'],
                 df_er['cent-max-mean']+df_er['cent-max-std'], alpha=0.2, color=colors[2])

# null model - dist
plt.plot(df_dist['cut'], df_dist['cent-max-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[1], label='SRG')
plt.fill_between(df_dist['cut'], df_dist['cent-max-mean']-df_dist['cent-max-std'],
                 df_dist['cent-max-mean']+df_dist['cent-max-std'], alpha=0.2, color=colors[1])

# actual value
plt.plot(df_cen['cut'], df_cen['cent-max'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.yscale('log')
plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('centrality max', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.legend(fontsize=lg_fz, frameon=False)
plt.savefig("vessels-z3-caxrad-all-log.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot together
plt.figure(figsize=(7,3.5))

# null model - ER
plt.plot(df_er['cut'], df_er['cent-max-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[2], label='SER')
plt.fill_between(df_er['cut'], df_er['cent-max-mean']-df_er['cent-max-std'],
                 df_er['cent-max-mean']+df_er['cent-max-std'], alpha=0.2, color=colors[2])

# null model - dist
plt.plot(df_dist['cut'], df_dist['cent-max-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[1], label='SRG')
plt.fill_between(df_dist['cut'], df_dist['cent-max-mean']-df_dist['cent-max-std'],
                 df_dist['cent-max-mean']+df_dist['cent-max-std'], alpha=0.2, color=colors[1])

# actual value
plt.plot(df_cen['cut'], df_cen['cent-max'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.ylim(ylim_cax)
plt.yscale('log')
plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('centrality max', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.legend(fontsize=lg_fz, frameon=False, loc='lower center')
plt.savefig("vessels-z3-caxrad-all3-log.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot together
plt.figure(figsize=(7,3.5))

# null model - radius
plt.plot(df_radi['cut'], df_radi['cent-max-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[3], label='Rad')
plt.fill_between(df_radi['cut'], df_radi['cent-max-mean']-df_radi['cent-max-std'],
                 df_radi['cent-max-mean']+df_radi['cent-max-std'], alpha=0.2, color=colors[3])

# null model - dist
plt.plot(df_dist['cut'], df_dist['cent-max-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[1], label='SRG')
plt.fill_between(df_dist['cut'], df_dist['cent-max-mean']-df_dist['cent-max-std'],
                 df_dist['cent-max-mean']+df_dist['cent-max-std'], alpha=0.2, color=colors[1])

# actual value
plt.plot(df_cen['cut'], df_cen['cent-max'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.yscale('log')
plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('centrality max', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.legend(fontsize=lg_fz, frameon=False)
plt.savefig("vessels-z3-caxrad-null3-log.png", dpi=300, bbox_inches='tight')
plt.show()

##### Single cut

In [ ]:
# Is the graph connected?
G_sel2 = build_region_graphs(df_nodes_sel2, df_edges_sel)
component_summary = summarize_cc_topk(G_sel2, k=10)

In [ ]:
figsize_g = [850, 500]
figsize_c = [850, 500]
cbar_len = 0.6
xval = 0.90

###### cut = 2

In [ ]:
# Case 1: when # above and below are about the same
ns = 2.
lw = 3.5
cut = 2.

## Visualization
# --- 1. Select edges for G1 (minRadiusAvg < cut) ---
edges_G1 = [(u, v, data) for u, v, data in G_sel2.edges(data=True) if data["minRadiusAvg"] < cut]

# --- 2. Select remaining edges for G2 (minRadiusAvg >= cut OR missing) ---
edges_G2 = [(u, v, data) for u, v, data in G_sel2.edges(data=True) if data["minRadiusAvg"] >= cut]

# print("#edges:", len(edges_G1), len(edges_G2))

# --- 3. Build subgraphs while preserving attributes ---
G1 = nx.Graph()
G1.add_nodes_from(G_sel2.nodes(data=True))
G1.add_edges_from(edges_G1)

G2 = nx.Graph()
G2.add_nodes_from(G_sel2.nodes(data=True))
G2.add_edges_from(edges_G2)

# --- 4. Remove isolated nodes ---
# G1.remove_nodes_from(list(nx.isolates(G1)))
# G2.remove_nodes_from(list(nx.isolates(G2)))

pos1 = nx.get_node_attributes(G1, 'pos')
pos2 = nx.get_node_attributes(G2, 'pos')
lab_G = 'vessel-z3-c{}'.format(cut)

In [ ]:
# edge colors
alpha = 0.9
row_c = "rgba(33, 89, 209,{})".format(alpha)
col_c = "rgba(196, 29, 29,{})".format(alpha)
# tick title sizes
ax_fz = 30
# figsize
figsize_g = [850, 500]

plot_networkx_12nolabel(G1, G2, pos1, pos2, size=ns, width=lw, row_c=row_c, col_c=col_c,
                        camera=camera, savefig=True, figname=lab_G+'-unic-G', figsize=figsize_g, ax_fz=ax_fz) #row_c='#2159d1', col_c='#c41d1d'

In [ ]:
## GLI analysis
# compute the GLI operator
# GLI between edges directly
edges1 = list(G1.edges())
edges2 = list(G2.edges())

Lam_gli = np.zeros((len(edges1), len(edges2)))
for i in range(Lam_gli.shape[0]):
    e1 = edges1[i]
    e1_pos = [np.array(G1.nodes[e1[0]]['pos']), np.array(G1.nodes[e1[1]]['pos'])]
    for j in range(Lam_gli.shape[1]):
        e2 = edges2[j]
        # check whether they share a vertex
        if e1[0] in e2 or e1[1] in e2:
            Lam_gli[i, j] = 0
        else:
            # record the position
            e2_pos = [np.array(G2.nodes[e2[0]]['pos']), np.array(G2.nodes[e2[1]]['pos'])]
            # direct computation of Gauss linking integral
            gli_ij = Gauss_linking_integral(e1_pos, e2_pos)
            Lam_gli[i, j] = gli_ij

In [ ]:
# Visualization
cmap_max = 0.1
val_max = np.abs(Lam_gli).max()
val_min = np.abs(Lam_gli[Lam_gli != 0]).min()
num_0s = np.sum(Lam_gli == 0)
print("edges in 1 and 2:", Lam_gli.shape)
print("#0 values in Lam_gli:", num_0s)
print("max abs value in Lam_gli:", val_max, "min abs value:", val_min)

In [ ]:
cmap_max = min([cmap_max, val_max])
print("max in cmap:", cmap_max)

plt.figure(figsize=(6, 3.5))
# plt.imshow(Lam_gli, cmap='seismic', vmax=val_max, vmin=-val_max) #, aspect='auto'
plt.imshow(Lam_gli, cmap='seismic', vmax=cmap_max, vmin=-cmap_max) #, aspect='auto'

# Optional: show grid or colorbar
plt.grid(False)
plt.colorbar(shrink=0.6)
plt.savefig("{}-Lam_gli-max{}.png".format(lab_G, cmap_max), dpi=300, bbox_inches='tight')
plt.show()

We see that there are very small values in the GLI operator, should we treat them as zero (i.e., no contribution) or as a contribution?
- Let's maintain the values and explore the patterns there.

In [ ]:
### linking centrality ###
# set the same range (from exploratory analysis)
cmin = 0.0067179
cmax = 3.5062 # 5.9063 -- there is only one edge of this value in the case of c=5
crange = [cmin, cmax]

In [ ]:
### linking centrality ###
figsize_c = [850, 500]
cbar_len = 0.6
xval = 0.90

# rows (edges1) and columns (edges2) that are not all zeros
row_sum = np.sum(abs(Lam_gli), axis=1)
col_sum = np.sum(abs(Lam_gli), axis=0)

print("node size:", ns, "line width:", lw)

cmap = 'plasma'
pos1 = nx.get_node_attributes(G1, 'pos')
pos2 = nx.get_node_attributes(G2, 'pos')

plot_networkx_colore(G1, G2, pos1, pos2, row_sum, col_sum, cmap=cmap, xval=xval,
                     ns=ns, lw=lw, camera=camera, savefig=True, figname=lab_G+'-unic-G-cent',
                     figsize=figsize_c, val_range=crange, cbar_len=cbar_len)

In [ ]:
# bipartite graph clustering
### Construct the bipartite signed graph
fac = 100 # scale up the values, for numerical reasons
Lam_gfac = Lam_gli * fac
# labels_p1 = ['({},{})'.format(e[0], e[1]) for e in edges1]
# labels_p2 = ['({},{})'.format(e[0], e[1]) for e in edges2]

# Gb = build_sibigraph(Lam_gtol, labels_p1, labels_p2)

# use index instead
labels_p1 = [i for i in range(len(edges1))]
labels_p2 = [i+len(edges1) for i in range(len(edges2))]

Gb = build_sibigraph(Lam_gfac, labels_p1, labels_p2)
print("#nodes:", Gb.number_of_nodes(), "#edges:", Gb.number_of_edges())

# prepare connected components if any
Gb_ccs = [Gb.subgraph(c) for c in nx.connected_components(Gb)]
print("#connected components:", len(Gb_ccs))
print("sizes:", [(g.number_of_nodes(), g.number_of_edges()) for g in Gb_ccs])

In [ ]:
### Following tasks inside each cc ###
idx_g = 0

Gb_cc = Gb_ccs[idx_g]

# get the adjacency
nodes_cc = sorted(Gb_cc.nodes())
W_cc = nx.to_numpy_array(Gb_cc, nodelist=nodes_cc)

# get edges lists
edges1_cc = [edges1[i] for i in nodes_cc if i < len(edges1)]
edges2_cc = [edges2[i-len(edges1)] for i in nodes_cc if i >= len(edges1)]
print("two parts:", (len(edges1_cc), len(edges2_cc)))


# check the balance
deg = np.sum(abs(W_cc), axis=1)

# symmetric version
D2_inv = np.diag(1./np.sqrt(deg))
P_sym = D2_inv.dot(W_cc).dot(D2_inv)
eigvals = np.linalg.eigvals(P_sym)
print("Max eigenval of P:", np.max(eigvals))
# print("eigenvalues of P:", eigvals)

In [ ]:
# Laplacian
D = np.diag(deg)
L = D - W_cc
eigvals, eigvecs = np.linalg.eigh(L)
print("Min eigenvalues of L:", min(eigvals)/fac)

# normalized Laplacian
L_sym = np.eye(W_cc.shape[0]) - P_sym
eigvals, eigvecs = np.linalg.eigh(L_sym)
print("Min eigenvalues of L sym:", min(eigvals))

# indices of the smallest value
tol = 1e-2
eval_min = min(eigvals)
idx = [i for i, val in enumerate(eigvals) if abs(val-eval_min) < tol]
print("Min eigenvalue(s) of L sym:", eigvals[idx])
print("first few eigenvalues:", eigvals[:5])

plt.figure(figsize=(18, 5))
plt.subplot(121)
# choose the eigenvector(s)
for i in idx:
    plt.plot(eigvecs[:, i], marker='.', linestyle='dashed')
plt.grid(True)

plt.subplot(122)
plt.plot(eigvals, marker='.', linestyle='dashed')
plt.grid(True)
plt.show()

Bi-clustering 1: choose the smallest eigenvalue:

In [ ]:
### Bi-Clustering ###
tol = 1e-4
nodec = []
idx_0s = []

for i,val in enumerate(eigvecs[:, idx[0]]):
    if val > tol:
        nodec.append(2)
    elif val < -tol:
        nodec.append(0)
    else:
        idx_0s.append(i)
        nodec.append(1)

if len(idx_0s) == 0:
    print("clear cut!")
    nodec = [1 if nodec[i] > 0 else 0 for i in range(len(nodec))]
else:
    print("# zeros in the eigenvector:", len(idx_0s))

In [ ]:
# allocation zeros
from itertools import combinations

s = np.array([1. if nodec[i] > 0 else -1. for i in range(len(nodec))]) # default setting where all in the 1-group
best_cost = s.T.dot(L).dot(s)
best_partition = [s]
size = len(idx_0s)
idx_0s = np.array(idx_0s)

res = []
for num in range(1, int((size+1.)/2.)+1):
    for idx_c in combinations(list(range(size)), num):
        s_now = s.copy()
        s_now[idx_0s[list(idx_c)]] = -1 # allocate these edges to the -1-group
        cost = s_now.T.dot(L).dot(s_now)
        if cost < best_cost-tol:
            print("change to -1:", idx_c)
            best_cost = cost
            best_partition = [s_now]
        elif abs(cost-best_cost) < tol:
            best_partition.append(s_now)

print("Best cost:", best_cost/fac)
# print("Best partition:", best_partition)

In [ ]:
### Post processing ###
# apply the optimal orientation
s = best_partition[0]
S = np.diag(s)
W_s = S.dot(W_cc.dot(S))
B_s = W_s[:len(edges1_cc), :][:, -len(edges2_cc):]

# Distribution of the linking values
plt.hist((B_s/fac).flatten(), bins=100)
plt.yscale('log')
plt.show()

In [ ]:
# select strong negative signals
negr, negc = np.where(B_s < -0.1*fac)
print("#very negative edges:", len(negr))

# record
res = [(edges1_cc[negr[i]], edges2_cc[negc[i]]) for i in range(len(negr))]
res_idx = [((n2i_dict[e1[0]], n2i_dict[e1[1]]), (n2i_dict[e2[0]], n2i_dict[e2[1]])) for e1,e2 in res]

# medium
negr, negc = np.where(B_s < -0.05*fac)
print("#medium negative edges:", len(negr))

# record
res2 = [(edges1_cc[negr[i]], edges2_cc[negc[i]]) for i in range(len(negr))]
res2_idx = [((n2i_dict[e1[0]], n2i_dict[e1[1]]), (n2i_dict[e2[0]], n2i_dict[e2[1]])) for e1,e2 in res]

# visualization
edges1_str = [epair[0] for epair in res]
edges2_str = [epair[1] for epair in res]

edges1_med = [epair[0] for epair in res2]
edges2_med = [epair[1] for epair in res2]

In [ ]:
## select the colors
alphas = [0.3, 0.5, 1.]
color1 = "rgba(0, 0, 139, {})"
color2 = "rgba(139, 0, 0, {})"

colorset = [[color1.format(a), color2.format(a)] for a in alphas]
# change the middle color
alpha = 0.9
colorset[1] = ["rgba(33, 89, 209,{})".format(alpha), "rgba(196, 29, 29,{})".format(alpha)]

In [ ]:
lab_fac = '-f{}'.format(fac)
# ns = 3.5
# lw = 5
print("node size:", ns, "line width:", lw)

plot_networkx_2colore_sel(G_sel2, G1, G2, edges1_str, edges2_str, edges1_med, edges2_med,
                          ns=ns, lw=lw, camera=camera, savefig=True, colorset=colorset,
                          figsize=figsize_g, figname=lab_G+lab_fac+'-unic-G-signed-l2m-1')

In [ ]:
# if we only want to show the most negative ones:
plot_networkx_2colore_sel(G_sel2, G1, G2, edges1_str, edges2_str, [], [],
                          ns=ns, lw=lw, camera=camera, savefig=True, colorset=colorset,
                          figsize=figsize_g, figname=lab_G+lab_fac+'-unic-G-signed-l2m-str-1')

Bi-clustering 2: combine smallest eigenvectors

We observe that the last two eigenvalues are very close to each other.

In [ ]:
# Combine the first two eigenvectors to see whether we have better results?
tol = 1e-4

from itertools import combinations

eigvals, eigvecs = np.linalg.eigh(L_sym)
print("Min eigenvalues of L_sym:", min(eigvals)) # !!! L_sym

# indices of the smallest value
tol_ev = 0.02
eval_min = min(eigvals)
idx = [i for i, val in enumerate(eigvals) if abs(val-eval_min) < tol_ev]
print("Min eigenvalue(s) of L_sym:", eigvals[idx])

u1 = eigvecs[:, idx[0]]
u2 = eigvecs[:, idx[1]]

flip_angles, nodes_deg = critical_flip_angles(u1, u2)
print("#Flip angles:", len(flip_angles))
theta_candidates = candidate_mid_angles(flip_angles)

# Get sign patterns for all candidates
S = sign_patterns_from_angles(u1, u2, theta_candidates)  # shape (n, m)

# For each candidate, evaluate your signed cut cost C(theta_j)
best_cost = np.inf
best_theta = []
best_partition = []

for j, theta in enumerate(theta_candidates):
    s = S[:, j]         # +-1 labels
    # 0 entries?
    idx_0 = np.where(s == 0)[0]
    if len(idx_0) > 0:
        print("0 occurs in eigenvalues")
        ## try all combination of assignments
        # all in the negative one
        s[idx_0] = -1
        cost = s.T.dot(L).dot(s)
        if cost < best_cost-tol:
            best_cost = cost
            best_theta = [theta]
            best_partition = [s]
        elif abs(cost-best_cost) < tol:
            best_theta.append(theta)
            best_partition.append(s)

        # all other combinations
        for num in range(1, len(idx_0)):
            for idx_01 in combinations(idx_0, num):
                s_now = s.copy()
                s_now[idx_0] = -1
                s_now[list(idx_01)] = 1
                cost = s_now.T.dot(L).dot(s_now)
                if cost < best_cost-tol:
                    best_cost = cost
                    best_theta = [theta]
                    best_partition = [s_now]
                elif abs(cost-best_cost) < tol:
                    best_theta.append(theta)
                    best_partition.append(s_now)
        # the last one
        s[idx_0] = 1
        cost = s.T.dot(L).dot(s)
        if cost < best_cost-tol:
            best_cost = cost
            best_theta = [theta]
            best_partition = [s]
        elif abs(cost-best_cost) < tol:
            best_theta.append(theta)
            best_partition.append(s)
    else:
        # compute the objective
        cost = s.T.dot(L).dot(s)
        if cost < best_cost-tol:
            best_cost = cost
            best_theta = [theta]
            best_partition = [s]
        elif abs(cost-best_cost) < tol:
            best_theta.append(theta)
            best_partition.append(s)

print("Best theta:", best_theta)
print("Best cost:", best_cost/fac)
print("# Best partitions:", len(best_partition))

In [ ]:
### Post processing ###
# apply the optimal orientation
s = best_partition[0]
S = np.diag(s)
W_s = S.dot(W_cc.dot(S))
B_s = W_s[:len(edges1_cc), :][:, -len(edges2_cc):]

# Distribution of the linking values
plt.hist((B_s/fac).flatten(), bins=100)
plt.yscale('log')
plt.show()

In [ ]:
# select strong negative signals
negr, negc = np.where(B_s < -0.1*fac)
print("#very negative edges:", len(negr))

# record
res = [(edges1_cc[negr[i]], edges2_cc[negc[i]]) for i in range(len(negr))]
res_idx = [((n2i_dict[e1[0]], n2i_dict[e1[1]]), (n2i_dict[e2[0]], n2i_dict[e2[1]])) for e1,e2 in res]

# medium
negr, negc = np.where(B_s < -0.05*fac)
print("#medium negative edges:", len(negr))

# record
res2 = [(edges1_cc[negr[i]], edges2_cc[negc[i]]) for i in range(len(negr))]
res2_idx = [((n2i_dict[e1[0]], n2i_dict[e1[1]]), (n2i_dict[e2[0]], n2i_dict[e2[1]])) for e1,e2 in res]

# visualization
edges1_str = [epair[0] for epair in res]
edges2_str = [epair[1] for epair in res]

edges1_med = [epair[0] for epair in res2]
edges2_med = [epair[1] for epair in res2]

In [ ]:
alphas = [0.3, 0.5, 1.]
color1 = "rgba(0, 0, 139, {})"
color2 = "rgba(139, 0, 0, {})"

colorset = [[color1.format(a), color2.format(a)] for a in alphas]
# change the middle color
alpha = 0.9
colorset[1] = ["rgba(33, 89, 209,{})".format(alpha), "rgba(196, 29, 29,{})".format(alpha)]

In [ ]:
lab_fac = '-f{}'.format(fac)
# ns = 3.5
# lw = 5
print("node size:", ns, "line width:", lw)

plot_networkx_2colore_sel(G_sel2, G1, G2, edges1_str, edges2_str, edges1_med, edges2_med,
                          ns=ns, lw=lw, camera=camera, savefig=True, colorset=colorset,
                          figsize=figsize_g, figname=lab_G+lab_fac+'-unic-G-signed-l2m-2')

In [ ]:
print("node size:", ns, "line width:", lw)

plot_networkx_2colore_sel(G_sel2, G1, G2, edges1_str, edges2_str, [], [],
                          ns=ns, lw=lw, camera=camera, savefig=True, colorset=colorset,
                          figsize=figsize_g, figname=lab_G+lab_fac+'-unic-G-signed-l2m-str-2')

###### cut = 5

In [ ]:
# Case 1: when # above and below are about the same
ns = 2.
lw = 3.5
cut = 5

## Visualization
# --- 1. Select edges for G1 (minRadiusAvg < cut) ---
edges_G1 = [(u, v, data) for u, v, data in G_sel2.edges(data=True) if data["minRadiusAvg"] < cut]

# --- 2. Select remaining edges for G2 (minRadiusAvg >= cut OR missing) ---
edges_G2 = [(u, v, data) for u, v, data in G_sel2.edges(data=True) if data["minRadiusAvg"] >= cut]

# print("#edges:", len(edges_G1), len(edges_G2))

# --- 3. Build subgraphs while preserving attributes ---
G1 = nx.Graph()
G1.add_nodes_from(G_sel2.nodes(data=True))
G1.add_edges_from(edges_G1)

G2 = nx.Graph()
G2.add_nodes_from(G_sel2.nodes(data=True))
G2.add_edges_from(edges_G2)

# --- 4. Remove isolated nodes ---
# G1.remove_nodes_from(list(nx.isolates(G1)))
# G2.remove_nodes_from(list(nx.isolates(G2)))

pos1 = nx.get_node_attributes(G1, 'pos')
pos2 = nx.get_node_attributes(G2, 'pos')
lab_G = 'vessel-z3-c{}'.format(cut)

In [ ]:
# edge colors
alpha = 0.9
row_c = "rgba(33, 89, 209,{})".format(alpha)
col_c = "rgba(196, 29, 29,{})".format(alpha)
# tick title sizes
ax_fz = 30
# figsize

plot_networkx_12nolabel(G1, G2, pos1, pos2, size=ns, width=lw, row_c=row_c, col_c=col_c,
                        camera=camera, savefig=True, figname=lab_G+'-unic-G', figsize=figsize_g, ax_fz=ax_fz) #row_c='#2159d1', col_c='#c41d1d'

In [ ]:
## GLI analysis
# compute the GLI operator
# GLI between edges directly
edges1 = list(G1.edges())
edges2 = list(G2.edges())

Lam_gli = np.zeros((len(edges1), len(edges2)))
for i in range(Lam_gli.shape[0]):
    e1 = edges1[i]
    e1_pos = [np.array(G1.nodes[e1[0]]['pos']), np.array(G1.nodes[e1[1]]['pos'])]
    for j in range(Lam_gli.shape[1]):
        e2 = edges2[j]
        # check whether they share a vertex
        if e1[0] in e2 or e1[1] in e2:
            Lam_gli[i, j] = 0
        else:
            # record the position
            e2_pos = [np.array(G2.nodes[e2[0]]['pos']), np.array(G2.nodes[e2[1]]['pos'])]
            # direct computation of Gauss linking integral
            gli_ij = Gauss_linking_integral(e1_pos, e2_pos)
            Lam_gli[i, j] = gli_ij

In [ ]:
# Visualization
cmap_max = 0.1
val_max = np.abs(Lam_gli).max()
val_min = np.abs(Lam_gli[Lam_gli != 0]).min()
num_0s = np.sum(Lam_gli == 0)
print("edges in 1 and 2:", Lam_gli.shape)
print("#0 values in Lam_gli:", num_0s)
print("max abs value in Lam_gli:", val_max, "min abs value:", val_min)

In [ ]:
cmap_max = min([cmap_max, val_max])
print("max in cmap:", cmap_max)

plt.figure(figsize=(6, 3.5))
# plt.imshow(Lam_gli, cmap='seismic', vmax=val_max, vmin=-val_max) #, aspect='auto'
plt.imshow(Lam_gli, cmap='seismic', vmax=cmap_max, vmin=-cmap_max) #, aspect='auto'

# Optional: show grid or colorbar
plt.grid(False)
plt.colorbar(shrink=0.6)
plt.savefig("{}-Lam_gli-max{}.png".format(lab_G, cmap_max), dpi=300, bbox_inches='tight')
plt.show()

We see that there are very small values in the GLI operator, should we treat them as zero (i.e., no contribution) or as a contribution?
- Let's maintain the values and explore the patterns there.

In [ ]:
### linking centrality ###

# rows (edges1) and columns (edges2) that are not all zeros
row_sum = np.sum(abs(Lam_gli), axis=1)
col_sum = np.sum(abs(Lam_gli), axis=0)

print("node size:", ns, "line width:", lw)

cmap = 'plasma'
pos1 = nx.get_node_attributes(G1, 'pos')
pos2 = nx.get_node_attributes(G2, 'pos')

plot_networkx_colore(G1, G2, pos1, pos2, row_sum, col_sum, cmap=cmap, xval=xval,
                     ns=ns, lw=lw, camera=camera, savefig=True, figname=lab_G+'-unic-G-cent',
                     figsize=figsize_c, val_range=crange, cbar_len=cbar_len)

In [ ]:
# bipartite graph clustering
### Construct the bipartite signed graph
fac = 100 # scale up the values, for numerical reasons
Lam_gfac = Lam_gli * fac
# labels_p1 = ['({},{})'.format(e[0], e[1]) for e in edges1]
# labels_p2 = ['({},{})'.format(e[0], e[1]) for e in edges2]

# Gb = build_sibigraph(Lam_gtol, labels_p1, labels_p2)

# use index instead
labels_p1 = [i for i in range(len(edges1))]
labels_p2 = [i+len(edges1) for i in range(len(edges2))]

Gb = build_sibigraph(Lam_gfac, labels_p1, labels_p2)
print("#nodes:", Gb.number_of_nodes(), "#edges:", Gb.number_of_edges())

# prepare connected components if any
Gb_ccs = [Gb.subgraph(c) for c in nx.connected_components(Gb)]
print("#connected components:", len(Gb_ccs))
print("sizes:", [(g.number_of_nodes(), g.number_of_edges()) for g in Gb_ccs])

In [ ]:
### Following tasks inside each cc ###
idx_g = 0

Gb_cc = Gb_ccs[idx_g]

# get the adjacency
nodes_cc = sorted(Gb_cc.nodes())
W_cc = nx.to_numpy_array(Gb_cc, nodelist=nodes_cc)

# get edges lists
edges1_cc = [edges1[i] for i in nodes_cc if i < len(edges1)]
edges2_cc = [edges2[i-len(edges1)] for i in nodes_cc if i >= len(edges1)]
print("two parts:", (len(edges1_cc), len(edges2_cc)))


# check the balance
deg = np.sum(abs(W_cc), axis=1)

# symmetric version
D2_inv = np.diag(1./np.sqrt(deg))
P_sym = D2_inv.dot(W_cc).dot(D2_inv)
eigvals = np.linalg.eigvals(P_sym)
print("Max eigenval of P:", np.max(eigvals))
# print("eigenvalues of P:", eigvals)

In [ ]:
# Laplacian
D = np.diag(deg)
L = D - W_cc
eigvals, eigvecs = np.linalg.eigh(L)
print("Min eigenvalues of L:", min(eigvals)/fac)

# normalized Laplacian
L_sym = np.eye(W_cc.shape[0]) - P_sym
eigvals, eigvecs = np.linalg.eigh(L_sym)
print("Min eigenvalues of L sym:", min(eigvals))

# indices of the smallest value
tol = 0.01
eval_min = min(eigvals)
idx = [i for i, val in enumerate(eigvals) if abs(val-eval_min) < tol]
print("Min eigenvalue(s) of L sym:", eigvals[idx])
print("first few eigenvalues:", eigvals[:5])

plt.figure(figsize=(18, 5))
plt.subplot(121)
# choose the eigenvector(s)
for i in idx:
    plt.plot(eigvecs[:, i], marker='.', linestyle='dashed')
plt.grid(True)

plt.subplot(122)
plt.plot(eigvals, marker='.', linestyle='dashed')
plt.grid(True)
plt.show()

In [ ]:
### Bi-Clustering ###
tol = 1e-4
nodec = []
idx_0s = []

for i,val in enumerate(eigvecs[:, idx[0]]):
    if val > tol:
        nodec.append(2)
    elif val < -tol:
        nodec.append(0)
    else:
        idx_0s.append(i)
        nodec.append(1)

if len(idx_0s) == 0:
    print("clear cut!")
    nodec = [1 if nodec[i] > 0 else 0 for i in range(len(nodec))]
else:
    print("# zeros in the eigenvector:", len(idx_0s))

In [ ]:
# allocation zeros
from itertools import combinations

s = np.array([1. if nodec[i] > 0 else -1. for i in range(len(nodec))]) # default setting where all in the 1-group
best_cost = s.T.dot(L).dot(s)
best_partition = [s]
size = len(idx_0s)
idx_0s = np.array(idx_0s)

res = []
for num in range(1, int((size+1.)/2.)+1):
    for idx_c in combinations(list(range(size)), num):
        s_now = s.copy()
        s_now[idx_0s[list(idx_c)]] = -1 # allocate these edges to the -1-group
        cost = s_now.T.dot(L).dot(s_now)
        if cost < best_cost-tol:
            print("change to -1:", idx_c)
            best_cost = cost
            best_partition = [s_now]
        elif abs(cost-best_cost) < tol:
            best_partition.append(s_now)

print("Best cost:", best_cost/fac)
# print("Best partition:", best_partition)

In [ ]:
### Post processing ###
# apply the optimal orientation
s = best_partition[0]
S = np.diag(s)
W_s = S.dot(W_cc.dot(S))
B_s = W_s[:len(edges1_cc), :][:, -len(edges2_cc):]

# Distribution of the linking values
plt.hist((B_s/fac).flatten(), bins=100)
plt.yscale('log')
plt.show()

In [ ]:
# select strong negative signals
negr, negc = np.where(B_s < -0.1*fac)
print("#very negative edges:", len(negr))

# record
res = [(edges1_cc[negr[i]], edges2_cc[negc[i]]) for i in range(len(negr))]
res_idx = [((n2i_dict[e1[0]], n2i_dict[e1[1]]), (n2i_dict[e2[0]], n2i_dict[e2[1]])) for e1,e2 in res]

# medium
negr, negc = np.where(B_s < -0.05*fac)
print("#medium negative edges:", len(negr))

# record
res2 = [(edges1_cc[negr[i]], edges2_cc[negc[i]]) for i in range(len(negr))]
res2_idx = [((n2i_dict[e1[0]], n2i_dict[e1[1]]), (n2i_dict[e2[0]], n2i_dict[e2[1]])) for e1,e2 in res]

# visualization
edges1_str = [epair[0] for epair in res]
edges2_str = [epair[1] for epair in res]

edges1_med = [epair[0] for epair in res2]
edges2_med = [epair[1] for epair in res2]

In [ ]:
lab_fac = '-f{}'.format(fac)
# ns = 3.5
# lw = 5
print("node size:", ns, "line width:", lw)

plot_networkx_2colore_sel(G_sel2, G1, G2, edges1_str, edges2_str, edges1_med, edges2_med,
                          ns=ns, lw=lw, camera=camera, savefig=True, colorset=colorset,
                          figsize=figsize_g, figname=lab_G+lab_fac+'-unic-G-signed-l2m')

In [ ]:
print("node size:", ns, "line width:", lw)

plot_networkx_2colore_sel(G_sel2, G1, G2, edges1_str, edges2_str, [], [],
                          ns=ns, lw=lw, camera=camera, savefig=True, colorset=colorset,
                          figsize=figsize_g, figname=lab_G+lab_fac+'-unic-G-signed-l2m-str')

###### cut = 1.5

In [ ]:
# Case 1: when # above and below are about the same
ns = 2.
lw = 3.5
cut = 1.5

## Visualization
# --- 1. Select edges for G1 (minRadiusAvg < cut) ---
edges_G1 = [(u, v, data) for u, v, data in G_sel2.edges(data=True) if data["minRadiusAvg"] < cut]

# --- 2. Select remaining edges for G2 (minRadiusAvg >= cut OR missing) ---
edges_G2 = [(u, v, data) for u, v, data in G_sel2.edges(data=True) if data["minRadiusAvg"] >= cut]

# print("#edges:", len(edges_G1), len(edges_G2))

# --- 3. Build subgraphs while preserving attributes ---
G1 = nx.Graph()
G1.add_nodes_from(G_sel2.nodes(data=True))
G1.add_edges_from(edges_G1)

G2 = nx.Graph()
G2.add_nodes_from(G_sel2.nodes(data=True))
G2.add_edges_from(edges_G2)

# --- 4. Remove isolated nodes ---
# G1.remove_nodes_from(list(nx.isolates(G1)))
# G2.remove_nodes_from(list(nx.isolates(G2)))

pos1 = nx.get_node_attributes(G1, 'pos')
pos2 = nx.get_node_attributes(G2, 'pos')
lab_G = 'vessel-z3-c{}'.format(cut)

In [ ]:
# edge colors
alpha = 0.9
row_c = "rgba(33, 89, 209,{})".format(alpha)
col_c = "rgba(196, 29, 29,{})".format(alpha)
# tick title sizes
ax_fz = 30
# figsize

plot_networkx_12nolabel(G1, G2, pos1, pos2, size=ns, width=lw, row_c=row_c, col_c=col_c,
                        camera=camera, savefig=True, figname=lab_G+'-unic-G', figsize=figsize_g, ax_fz=ax_fz) #row_c='#2159d1', col_c='#c41d1d'

In [ ]:
## GLI analysis
# compute the GLI operator
# GLI between edges directly
edges1 = list(G1.edges())
edges2 = list(G2.edges())

Lam_gli = np.zeros((len(edges1), len(edges2)))
for i in range(Lam_gli.shape[0]):
    e1 = edges1[i]
    e1_pos = [np.array(G1.nodes[e1[0]]['pos']), np.array(G1.nodes[e1[1]]['pos'])]
    for j in range(Lam_gli.shape[1]):
        e2 = edges2[j]
        # check whether they share a vertex
        if e1[0] in e2 or e1[1] in e2:
            Lam_gli[i, j] = 0
        else:
            # record the position
            e2_pos = [np.array(G2.nodes[e2[0]]['pos']), np.array(G2.nodes[e2[1]]['pos'])]
            # direct computation of Gauss linking integral
            gli_ij = Gauss_linking_integral(e1_pos, e2_pos)
            Lam_gli[i, j] = gli_ij

In [ ]:
# Visualization
cmap_max = 0.1
val_max = np.abs(Lam_gli).max()
val_min = np.abs(Lam_gli[Lam_gli != 0]).min()
num_0s = np.sum(Lam_gli == 0)
print("edges in 1 and 2:", Lam_gli.shape)
print("#0 values in Lam_gli:", num_0s)
print("max abs value in Lam_gli:", val_max, "min abs value:", val_min)

In [ ]:
cmap_max = min([cmap_max, val_max])
print("max in cmap:", cmap_max)

plt.figure(figsize=(6, 3.5))
# plt.imshow(Lam_gli, cmap='seismic', vmax=val_max, vmin=-val_max) #, aspect='auto'
plt.imshow(Lam_gli, cmap='seismic', vmax=cmap_max, vmin=-cmap_max) #, aspect='auto'

# Optional: show grid or colorbar
plt.grid(False)
plt.colorbar(shrink=0.6)
plt.savefig("{}-Lam_gli-max{}.png".format(lab_G, cmap_max), dpi=300, bbox_inches='tight')
plt.show()

We see that there are very small values in the GLI operator, should we treat them as zero (i.e., no contribution) or as a contribution?
- Let's maintain the values and explore the patterns there.

In [ ]:
### linking centrality ###
# rows (edges1) and columns (edges2) that are not all zeros
row_sum = np.sum(abs(Lam_gli), axis=1)
col_sum = np.sum(abs(Lam_gli), axis=0)

print("node size:", ns, "line width:", lw)

cmap = 'plasma'
pos1 = nx.get_node_attributes(G1, 'pos')
pos2 = nx.get_node_attributes(G2, 'pos')

plot_networkx_colore(G1, G2, pos1, pos2, row_sum, col_sum, cmap=cmap, xval=xval,
                     ns=ns, lw=lw, camera=camera, savefig=True, figname=lab_G+'-unic-G-cent',
                     figsize=figsize_c, val_range=crange, cbar_len=cbar_len)

In [ ]:
# bipartite graph clustering
### Construct the bipartite signed graph
fac = 100 # scale up the values, for numerical reasons
Lam_gfac = Lam_gli * fac
# labels_p1 = ['({},{})'.format(e[0], e[1]) for e in edges1]
# labels_p2 = ['({},{})'.format(e[0], e[1]) for e in edges2]

# Gb = build_sibigraph(Lam_gtol, labels_p1, labels_p2)

# use index instead
labels_p1 = [i for i in range(len(edges1))]
labels_p2 = [i+len(edges1) for i in range(len(edges2))]

Gb = build_sibigraph(Lam_gfac, labels_p1, labels_p2)
print("#nodes:", Gb.number_of_nodes(), "#edges:", Gb.number_of_edges())

# prepare connected components if any
Gb_ccs = [Gb.subgraph(c) for c in nx.connected_components(Gb)]
print("#connected components:", len(Gb_ccs))
print("sizes:", [(g.number_of_nodes(), g.number_of_edges()) for g in Gb_ccs])

In [ ]:
### Following tasks inside each cc ###
idx_g = 0

Gb_cc = Gb_ccs[idx_g]

# get the adjacency
nodes_cc = sorted(Gb_cc.nodes())
W_cc = nx.to_numpy_array(Gb_cc, nodelist=nodes_cc)

# get edges lists
edges1_cc = [edges1[i] for i in nodes_cc if i < len(edges1)]
edges2_cc = [edges2[i-len(edges1)] for i in nodes_cc if i >= len(edges1)]
print("two parts:", (len(edges1_cc), len(edges2_cc)))


# check the balance
deg = np.sum(abs(W_cc), axis=1)

# symmetric version
D2_inv = np.diag(1./np.sqrt(deg))
P_sym = D2_inv.dot(W_cc).dot(D2_inv)
eigvals = np.linalg.eigvals(P_sym)
print("Max eigenval of P:", np.max(eigvals))
# print("eigenvalues of P:", eigvals)

In [ ]:
# Laplacian
D = np.diag(deg)
L = D - W_cc
eigvals, eigvecs = np.linalg.eigh(L)
print("Min eigenvalues of L:", min(eigvals)/fac)

# normalized Laplacian
L_sym = np.eye(W_cc.shape[0]) - P_sym
eigvals, eigvecs = np.linalg.eigh(L_sym)
print("Min eigenvalues of L sym:", min(eigvals))

# indices of the smallest value
tol = 1e-2
eval_min = min(eigvals)
idx = [i for i, val in enumerate(eigvals) if abs(val-eval_min) < tol]
print("Min eigenvalue(s) of L sym:", eigvals[idx])
print("first few eigenvalues:", eigvals[:5])

plt.figure(figsize=(18, 5))
plt.subplot(121)
# choose the eigenvector(s)
for i in idx:
    plt.plot(eigvecs[:, i], marker='.', linestyle='dashed')
plt.grid(True)

plt.subplot(122)
plt.plot(eigvals, marker='.', linestyle='dashed')
plt.grid(True)
plt.show()

In [ ]:
### Bi-Clustering ###
tol = 1e-4
nodec = []
idx_0s = []

for i,val in enumerate(eigvecs[:, idx[0]]):
    if val > tol:
        nodec.append(2)
    elif val < -tol:
        nodec.append(0)
    else:
        idx_0s.append(i)
        nodec.append(1)

if len(idx_0s) == 0:
    print("clear cut!")
    nodec = [1 if nodec[i] > 0 else 0 for i in range(len(nodec))]
else:
    print("# zeros in the eigenvector:", len(idx_0s))

In [ ]:
# allocation zeros
from itertools import combinations

s = np.array([1. if nodec[i] > 0 else -1. for i in range(len(nodec))]) # default setting where all in the 1-group
best_cost = s.T.dot(L).dot(s)
best_partition = [s]
size = len(idx_0s)
idx_0s = np.array(idx_0s)

res = []
for num in range(1, int((size+1.)/2.)+1):
    for idx_c in combinations(list(range(size)), num):
        s_now = s.copy()
        s_now[idx_0s[list(idx_c)]] = -1 # allocate these edges to the -1-group
        cost = s_now.T.dot(L).dot(s_now)
        if cost < best_cost-tol:
            print("change to -1:", idx_c)
            best_cost = cost
            best_partition = [s_now]
        elif abs(cost-best_cost) < tol:
            best_partition.append(s_now)

print("Best cost:", best_cost/fac)
# print("Best partition:", best_partition)

In [ ]:
### Post processing ###
# apply the optimal orientation
s = best_partition[0]
S = np.diag(s)
W_s = S.dot(W_cc.dot(S))
B_s = W_s[:len(edges1_cc), :][:, -len(edges2_cc):]

# Distribution of the linking values
plt.hist((B_s/fac).flatten(), bins=100)
plt.yscale('log')
plt.show()

In [ ]:
# select strong negative signals
negr, negc = np.where(B_s < -0.1*fac)
print("#very negative edges:", len(negr))

# record
res = [(edges1_cc[negr[i]], edges2_cc[negc[i]]) for i in range(len(negr))]
res_idx = [((n2i_dict[e1[0]], n2i_dict[e1[1]]), (n2i_dict[e2[0]], n2i_dict[e2[1]])) for e1,e2 in res]

# medium
negr, negc = np.where(B_s < -0.05*fac)
print("#medium negative edges:", len(negr))

# record
res2 = [(edges1_cc[negr[i]], edges2_cc[negc[i]]) for i in range(len(negr))]
res2_idx = [((n2i_dict[e1[0]], n2i_dict[e1[1]]), (n2i_dict[e2[0]], n2i_dict[e2[1]])) for e1,e2 in res]

# visualization
edges1_str = [epair[0] for epair in res]
edges2_str = [epair[1] for epair in res]

edges1_med = [epair[0] for epair in res2]
edges2_med = [epair[1] for epair in res2]

In [ ]:
lab_fac = '-f{}'.format(fac)
# ns = 3.5
# lw = 5
print("node size:", ns, "line width:", lw)

plot_networkx_2colore_sel(G_sel2, G1, G2, edges1_str, edges2_str, edges1_med, edges2_med,
                          ns=ns, lw=lw, camera=camera, savefig=True, colorset=colorset,
                          figsize=figsize_g, figname=lab_G+lab_fac+'-unic-G-signed-l2m')

In [ ]:
print("node size:", ns, "line width:", lw)

plot_networkx_2colore_sel(G_sel2, G1, G2, edges1_str, edges2_str, [], [],
                          ns=ns, lw=lw, camera=camera, savefig=True, colorset=colorset,
                          figsize=figsize_g, figname=lab_G+lab_fac+'-unic-G-signed-l2m-str')